In [49]:
# ============================================================
# CELL 1: IMPORTS & SETUP
# ============================================================
# Central import cell for the glass-box ML cascade notebook.
# All packages are imported here to avoid scattered imports
# across experiment cells (LR → RF/GLASS-BRW → EBM).
#
# Dependencies:
#   pip install numpy pandas scipy scikit-learn optuna interpret
#              matplotlib seaborn statsmodels
# ============================================================

# ── Standard library ──────────────────────────────────────────
import time                                 # Performance timing
import warnings                             # Warning suppression
from pathlib import Path                    # Filesystem paths
from datetime import datetime               # Timestamps
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass

# ── Core numerical / data ─────────────────────────────────────
import numpy as np                          # Array operations
import pandas as pd                         # DataFrames & tabular data

# ── SciPy ─────────────────────────────────────────────────────
from scipy import stats                     # Statistical tests
from scipy.special import expit             # Sigmoid function
from scipy.stats import (
    ks_2samp,                               # Kolmogorov–Smirnov two-sample test
    loguniform,                             # Log-uniform distribution (RandomSearch)
    pointbiserialr,                         # Point-biserial correlation
    spearmanr,                              # Spearman rank correlation
    uniform,                                # Uniform distribution (RandomSearch)
)

# ── Statsmodels ───────────────────────────────────────────────
from statsmodels.stats.outliers_influence import variance_inflation_factor  # VIF for multicollinearity

# ── Scikit-learn: preprocessing & pipelines ───────────────────
from sklearn.preprocessing import (
    StandardScaler,                         # Z-score normalization
    MinMaxScaler,                           # Min-max scaling [0,1]
    KBinsDiscretizer,                       # Discretization for MI estimation
)
from sklearn.pipeline import Pipeline       # Estimator pipelines

# ── Scikit-learn: model selection ─────────────────────────────
from sklearn.model_selection import (
    StratifiedKFold,                        # Stratified CV splitter
    cross_val_score,                        # Quick CV scoring
    RandomizedSearchCV,                     # Random hyperparameter search (Cell A)
    GridSearchCV,                           # Grid hyperparameter search
    train_test_split,                       # Hold-out split
)

# ── Scikit-learn: metrics ─────────────────────────────────────
from sklearn.metrics import (
    roc_auc_score,                          # Area under ROC curve (primary metric)
    roc_curve,                              # FPR/TPR curve
    precision_score,                        # Precision
    recall_score,                           # Recall / sensitivity
    f1_score,                               # F1 (harmonic mean P & R)
    fbeta_score,                            # Fβ (weighted harmonic mean)
    accuracy_score,                         # Accuracy
    brier_score_loss,                       # Brier score (calibration)
    precision_recall_curve,                 # PR curve
    mutual_info_score,                      # Mutual information
)

# ── Scikit-learn: estimators & utilities ──────────────────────
from sklearn.linear_model import (
    LogisticRegression,                     # Stage 1: LR (14 features)
    LinearRegression,                       # R² shape complexity analysis
)
from sklearn.ensemble import RandomForestClassifier  # Stage 2: RF/GLASS-BRW (35 features)
from sklearn.utils.class_weight import compute_sample_weight  # Balanced class weighting
from sklearn.inspection import permutation_importance         # Permutation-based importance

# ── Optuna: Bayesian hyperparameter optimization ──────────────
import optuna                               # Cell B: 150-trial Bayesian optimization
from optuna.samplers import TPESampler      # Tree-structured Parzen Estimator
from optuna.pruners import MedianPruner     # Prune unpromising trials mid-CV

# ── InterpretML: Explainable Boosting Machine ─────────────────
from interpret.glassbox import ExplainableBoostingClassifier  # Stage 3: EBM

# ── Visualization ─────────────────────────────────────────────
import matplotlib.pyplot as plt             # Plotting
import seaborn as sns                       # Statistical visualization

# ── Project-specific ──────────────────────────────────────────
from data.preprocessing.bank_preprocessing import BankPreprocessor  # UCI bank data pipeline

# ── Global configuration ──────────────────────────────────────
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.style.use("seaborn-v0_8-darkgrid")
np.random.seed(42)

# ── Startup confirmation ──────────────────────────────────────
print("✅ All imports loaded")
print(f"   Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ All imports loaded
   Timestamp: 2026-02-27 12:41:21


In [50]:
# ============================================================
# CELL 2: LOAD & PREPROCESS
# ============================================================
df = pd.read_csv("./data/raw/bank-additional-full.csv", sep=";")
print(f"Loaded: {df.shape}")

preprocessor = BankPreprocessor(drop_leaky=True)
df_processed = preprocessor.fit_transform(df)


Loaded: (41188, 21)

🔧 PREPROCESSING: Bank Marketing Dataset
✅ Target 'y': {0: 36548, 1: 4640}
✅ Binary cols (unknown=-1): ['default', 'housing', 'loan']
✅ contact: {0: 26144, 1: 15044}
✅ month: [np.int8(3), np.int8(4), np.int8(5), np.int8(6), np.int8(7), np.int8(8), np.int8(9), np.int8(10), np.int8(11), np.int8(12)]
✅ day_of_week: [np.int8(0), np.int8(1), np.int8(2), np.int8(3), np.int8(4)]
✅ poutcome: {0: 35563, 1: 4252, 2: 1373}
✅ education: ordinal 0-6, unknown=-1
✅ job: 12 categories label-encoded
✅ marital: {1: 24928, 2: 11568, 0: 4612, -1: 80}

📊 Economic features (already numeric):
  emp.var.rate: [-3.400, 1.400]
  cons.price.idx: [92.201, 94.767]
  cons.conf.idx: [-50.800, -26.900]
  euribor3m: [0.634, 5.045]
  nr.employed: [4963.600, 5228.100]

📊 Campaign features (already numeric):
  age: [17, 98]
  duration: [0, 4918]
  campaign: [1, 56]
  pdays: [0, 999]
  previous: [0, 7]
⚠️  Dropped leaky features: ['duration', 'pdays', 'poutcome']

✅ df_proc ready: (41188, 18)
Memory: 2

In [51]:
# ============================================================
# CELL 3: SETUP PATHS & UTILITIES
# ============================================================

# Create output directories
OUTPUT_DIR = Path("research_logs")
FIG_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

print(f"📁 Output directory: {OUTPUT_DIR.absolute()}")
print(f"📁 Figures directory: {FIG_DIR.absolute()}")


@dataclass
class SeparationMetrics:
    """Container for feature separation metrics."""
    feature: str
    feature_type: str  # 'numeric' or 'categorical'
    
    # Numeric metrics
    cohens_d: Optional[float] = None
    ks_stat: Optional[float] = None
    ks_pvalue: Optional[float] = None
    point_biserial: Optional[float] = None
    
    # Common metrics
    mutual_info: Optional[float] = None
    auc_probe: Optional[float] = None
    
    # Categorical metrics
    cramers_v: Optional[float] = None
    target_rate_range: Optional[Tuple[float, float]] = None
    n_categories: Optional[int] = None
    
    # Derived composite score
    composite_score: Optional[float] = None
    
    def to_dict(self) -> Dict:
        """Convert to dictionary for DataFrame construction."""
        return {
            'feature': self.feature,
            'type': self.feature_type,
            'cohens_d': self.cohens_d,
            'ks_stat': self.ks_stat,
            'ks_pvalue': self.ks_pvalue,
            'point_biserial': self.point_biserial,
            'mutual_info': self.mutual_info,
            'auc_probe': self.auc_probe,
            'cramers_v': self.cramers_v,
            'target_rate_range': self.target_rate_range,
            'n_categories': self.n_categories,
            'composite_score': self.composite_score
        }


def cramers_v(x: pd.Series, y: pd.Series) -> float:
    """
    Calculate Cramér's V statistic for categorical association.
    
    V = sqrt(chi2 / (n * min(k-1, r-1)))
    
    Parameters
    ----------
    x : pd.Series
        Categorical feature
    y : pd.Series
        Binary target
        
    Returns
    -------
    float
        Cramér's V in [0, 1]
    """
    contingency = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(contingency)[0]
    n = len(x)
    min_dim = min(contingency.shape[0] - 1, contingency.shape[1] - 1)
    
    if min_dim == 0:
        return 0.0
    
    return np.sqrt(chi2 / (n * min_dim))


def cohens_d(group1: np.ndarray, group2: np.ndarray) -> float:
    """
    Calculate Cohen's d effect size.
    
    d = (mean1 - mean2) / pooled_std
    
    Parameters
    ----------
    group1, group2 : np.ndarray
        Two groups to compare
        
    Returns
    -------
    float
        Cohen's d (signed)
    """
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    
    if pooled_std == 0:
        return 0.0
    
    return (np.mean(group1) - np.mean(group2)) / pooled_std


def wilson_ci(successes: int, trials: int, confidence: float = 0.95) -> Tuple[float, float]:
    """
    Wilson score interval for binomial proportion.
    
    More accurate than normal approximation for small samples.
    
    Parameters
    ----------
    successes : int
        Number of successes
    trials : int
        Total trials
    confidence : float
        Confidence level (default 0.95)
        
    Returns
    -------
    Tuple[float, float]
        (lower_bound, upper_bound)
    """
    if trials == 0:
        return (0.0, 0.0)
    
    z = stats.norm.ppf(1 - (1 - confidence) / 2)
    p = successes / trials
    
    denominator = 1 + z**2 / trials
    centre = (p + z**2 / (2 * trials)) / denominator
    spread = z * np.sqrt(p * (1 - p) / trials + z**2 / (4 * trials**2)) / denominator
    
    return (max(0, centre - spread), min(1, centre + spread))


print("✅ Utility functions defined")

📁 Output directory: c:\Users\phill\projects\ai-business-coach\prototype\glass_pipeline\research_logs
📁 Figures directory: c:\Users\phill\projects\ai-business-coach\prototype\glass_pipeline\research_logs\figures
✅ Utility functions defined


In [52]:
# ============================================================
# CELL 4: SANITY CHECKS
# ============================================================

def sanity_check_data(df: pd.DataFrame) -> None:
    """
    Validate preprocessed dataset assumptions.
    
    Checks:
    - Target column exists and is binary
    - No leaky features present
    - All features are numeric
    - No missing values
    """
    print("\n" + "="*80)
    print("🔍 SANITY CHECKS")
    print("="*80)
    
    # Detect target column
    target_candidates = ['y', 'target', 'label', 'subscribed']
    target_col = None
    
    for col in target_candidates:
        if col in df.columns:
            target_col = col
            break
    
    if target_col is None:
        raise ValueError(f"❌ No target column found. Looked for: {target_candidates}")
    
    print(f"✅ Target column: '{target_col}'")
    
    # Check target values
    target_values = sorted(df[target_col].unique())
    assert len(target_values) == 2, f"❌ Target must be binary, got: {target_values}"
    assert set(target_values) == {0, 1}, f"❌ Target must be 0/1, got: {target_values}"
    
    class_dist = df[target_col].value_counts().sort_index()
    print(f"   Class distribution: {class_dist.to_dict()}")
    print(f"   Imbalance ratio: {class_dist[0] / class_dist[1]:.2f}:1")
    
    # Check for leaky features
    leaky_features = ['duration', 'pdays', 'poutcome']
    present_leaky = [f for f in leaky_features if f in df.columns]
    
    if present_leaky:
        print(f"⚠️  WARNING: Leaky features present: {present_leaky}")
        print("   These should have been dropped by preprocessor!")
    else:
        print(f"✅ No leaky features: {leaky_features}")
    
    # Check all numeric
    non_numeric = df.select_dtypes(exclude=['number']).columns.tolist()
    assert len(non_numeric) == 0, f"❌ Non-numeric columns found: {non_numeric}"
    print(f"✅ All columns numeric: {len(df.columns)} features")
    
    # Check missing values
    missing = df.isnull().sum().sum()
    assert missing == 0, f"❌ Missing values detected: {missing}"
    print(f"✅ No missing values")
    
    # Dataset shape
    print(f"\n📊 Dataset shape: {df.shape}")
    print(f"   Features (excluding target): {len(df.columns) - 1}")
    print(f"   Samples: {len(df):,}")
    
    return target_col


# Run sanity checks
TARGET_COL = sanity_check_data(df_processed)


🔍 SANITY CHECKS
✅ Target column: 'y'
   Class distribution: {0: 36548, 1: 4640}
   Imbalance ratio: 7.88:1
✅ No leaky features: ['duration', 'pdays', 'poutcome']
✅ All columns numeric: 18 features
✅ No missing values

📊 Dataset shape: (41188, 18)
   Features (excluding target): 17
   Samples: 41,188


In [ ]:
# ============================================================
# CELL 5: FEATURE TYPING
# ============================================================

def classify_features(df: pd.DataFrame, target_col: str) -> Tuple[List[str], List[str]]:
    """
    Classify features as numeric or categorical.
    
    Heuristic:
    - Integer features with <= 10 unique values → categorical
    - All others → numeric
    
    Parameters
    ----------
    df : pd.DataFrame
        Processed dataframe
    target_col : str
        Name of target column (excluded from features)
        
    Returns
    -------
    numeric_features : List[str]
        Continuous/high-cardinality features
    categorical_features : List[str]
        Low-cardinality integer features
    """
    features = [col for col in df.columns if col != target_col]
    
    numeric_features = []
    categorical_features = []
    
    print("\n" + "="*80)
    print("🏷️  FEATURE TYPING")
    print("="*80)
    
    for col in features:
        n_unique = df[col].nunique()
        dtype = df[col].dtype
        
        # Heuristic: int with <= 10 unique values is categorical
        if dtype in ['int8', 'int16', 'int32', 'int64'] and n_unique <= 10:
            categorical_features.append(col)
            print(f"📊 {col:20s} → categorical  (n_unique={n_unique})")
        else:
            numeric_features.append(col)
            print(f"📈 {col:20s} → numeric      (n_unique={n_unique})")
    
    print(f"\n✅ Numeric features: {len(numeric_features)}")
    print(f"✅ Categorical features: {len(categorical_features)}")
    
    return numeric_features, categorical_features


NUMERIC_FEATURES, CATEGORICAL_FEATURES = classify_features(df_processed, TARGET_COL)


In [53]:
# ============================================================
# CELL 6: SEPARATION METRICS COMPUTATION
# ============================================================

def compute_numeric_separation(df: pd.DataFrame, 
                                feature: str, 
                                target_col: str) -> SeparationMetrics:
    """
    Compute separation metrics for numeric feature.
    
    Metrics:
    - Cohen's d (effect size)
    - KS statistic (distribution difference)
    - Point-biserial correlation
    - Mutual information
    - Single-feature probe AUC
    
    Parameters
    ----------
    df : pd.DataFrame
        Dataset
    feature : str
        Feature name
    target_col : str
        Target column name
        
    Returns
    -------
    SeparationMetrics
        Computed metrics
    """
    X = df[feature].values.reshape(-1, 1)
    y = df[target_col].values
    
    # Split by class
    class_0 = df[df[target_col] == 0][feature].values
    class_1 = df[df[target_col] == 1][feature].values
    
    # Cohen's d
    d = cohens_d(class_1, class_0)
    
    # KS test
    ks_stat, ks_pval = ks_2samp(class_0, class_1)
    
    # Point-biserial correlation
    pb_corr, _ = pointbiserialr(y, df[feature])
    
    # Mutual information
    # Discretize for MI calculation
    n_bins = min(10, df[feature].nunique())
    if n_bins > 1:
        discretizer = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='quantile')
        X_binned = discretizer.fit_transform(X).ravel()
        mi = mutual_info_score(y, X_binned)
    else:
        mi = 0.0
    
    # Single-feature probe AUC
    lr = LogisticRegression(max_iter=500, random_state=42)
    lr.fit(X, y)
    y_pred_proba = lr.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, y_pred_proba)
    
    # Composite score (weighted average of normalized metrics)
    # Normalize to [0, 1] range
    normalized_metrics = {
        'abs_cohens_d': min(abs(d) / 2.0, 1.0),  # Large effect if |d| > 2
        'ks_stat': ks_stat,
        'abs_pb_corr': abs(pb_corr),
        'mi': mi / np.log(2) if mi > 0 else 0,  # Normalize by max possible
        'auc_delta': 2 * abs(auc - 0.5)  # Distance from random (0.5)
    }
    
    composite = np.mean(list(normalized_metrics.values()))
    
    return SeparationMetrics(
        feature=feature,
        feature_type='numeric',
        cohens_d=d,
        ks_stat=ks_stat,
        ks_pvalue=ks_pval,
        point_biserial=pb_corr,
        mutual_info=mi,
        auc_probe=auc,
        composite_score=composite
    )


def compute_categorical_separation(df: pd.DataFrame,
                                     feature: str,
                                     target_col: str) -> SeparationMetrics:
    """
    Compute separation metrics for categorical feature.
    
    Metrics:
    - Cramér's V (association strength)
    - Mutual information
    - Target rate range (min/max across categories)
    - Single-feature probe AUC
    
    Parameters
    ----------
    df : pd.DataFrame
        Dataset
    feature : str
        Feature name
    target_col : str
        Target column name
        
    Returns
    -------
    SeparationMetrics
        Computed metrics
    """
    X = df[feature].values.reshape(-1, 1)
    y = df[target_col].values
    
    # Cramér's V
    cv = cramers_v(df[feature], df[target_col])
    
    # Mutual information
    mi = mutual_info_score(y, df[feature])
    
    # Target rate by category
    target_rates = df.groupby(feature)[target_col].mean()
    tr_range = (target_rates.min(), target_rates.max())
    
    # Number of categories
    n_cats = df[feature].nunique()
    
    # Single-feature probe AUC (one-hot encoding)
    lr = LogisticRegression(max_iter=500, random_state=42)
    lr.fit(X, y)
    y_pred_proba = lr.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, y_pred_proba)
    
    # Composite score
    normalized_metrics = {
        'cramers_v': cv,
        'mi': mi / np.log(n_cats) if n_cats > 1 else 0,
        'tr_spread': tr_range[1] - tr_range[0],
        'auc_delta': 2 * abs(auc - 0.5)
    }
    
    composite = np.mean(list(normalized_metrics.values()))
    
    return SeparationMetrics(
        feature=feature,
        feature_type='categorical',
        cramers_v=cv,
        mutual_info=mi,
        auc_probe=auc,
        target_rate_range=tr_range,
        n_categories=n_cats,
        composite_score=composite
    )


def compute_all_separations(df: pd.DataFrame,
                             numeric_features: List[str],
                             categorical_features: List[str],
                             target_col: str) -> pd.DataFrame:
    """
    Compute separation metrics for all features.
    
    Parameters
    ----------
    df : pd.DataFrame
        Dataset
    numeric_features : List[str]
        Numeric feature names
    categorical_features : List[str]
        Categorical feature names
    target_col : str
        Target column name
        
    Returns
    -------
    pd.DataFrame
        Feature rankings with all metrics
    """
    print("\n" + "="*80)
    print("📊 COMPUTING SEPARATION METRICS")
    print("="*80)
    
    all_metrics = []
    
    # Numeric features
    print(f"\n📈 Numeric features ({len(numeric_features)}):")
    for i, feat in enumerate(numeric_features, 1):
        print(f"  [{i:2d}/{len(numeric_features)}] {feat}...", end='\r')
        metrics = compute_numeric_separation(df, feat, target_col)
        all_metrics.append(metrics.to_dict())
    
    print(f"\n✅ Numeric features complete")
    
    # Categorical features
    print(f"\n📊 Categorical features ({len(categorical_features)}):")
    for i, feat in enumerate(categorical_features, 1):
        print(f"  [{i:2d}/{len(categorical_features)}] {feat}...", end='\r')
        metrics = compute_categorical_separation(df, feat, target_col)
        all_metrics.append(metrics.to_dict())
    
    print(f"\n✅ Categorical features complete")
    
    # Convert to DataFrame
    df_metrics = pd.DataFrame(all_metrics)
    
    # Sort by composite score
    df_metrics = df_metrics.sort_values('composite_score', ascending=False).reset_index(drop=True)
    
    print(f"\n✅ Separation metrics computed for {len(df_metrics)} features")
    
    return df_metrics


# Compute all separation metrics
df_separation = compute_all_separations(
    df_processed, 
    NUMERIC_FEATURES, 
    CATEGORICAL_FEATURES, 
    TARGET_COL
)


📊 COMPUTING SEPARATION METRICS

📈 Numeric features (8):
  [ 8/8] nr.employed......
✅ Numeric features complete

📊 Categorical features (9):
  [ 9/9] previous......
✅ Categorical features complete

✅ Separation metrics computed for 17 features


In [54]:
# ============================================================
# CELL 7: RANKING TABLES
# ============================================================

def display_feature_rankings(df_metrics: pd.DataFrame, top_n: int = 20) -> None:
    """
    Display formatted feature ranking tables.
    
    Parameters
    ----------
    df_metrics : pd.DataFrame
        Feature separation metrics
    top_n : int
        Number of top features to display
    """
    print("\n" + "="*80)
    print("🏆 FEATURE RANKINGS (Top {})".format(top_n))
    print("="*80)
    
    # Overall ranking
    print("\n📊 OVERALL RANKING (by composite score)")
    print("-"*80)
    
    top_features = df_metrics.head(top_n).copy()
    
    # Format for display
    display_cols = ['feature', 'type', 'composite_score', 'auc_probe', 'mutual_info']
    
    for i, row in top_features.iterrows():
        print(f"\n{i+1:2d}. {row['feature']:20s} ({row['type']})")
        print(f"    Composite: {row['composite_score']:.4f}  |  AUC: {row['auc_probe']:.4f}  |  MI: {row['mutual_info']:.4f}")
        
        if row['type'] == 'numeric':
            print(f"    Cohen's d: {row['cohens_d']:+.3f}  |  KS: {row['ks_stat']:.3f}  |  r_pb: {row['point_biserial']:+.3f}")
        else:
            print(f"    Cramér's V: {row['cramers_v']:.3f}  |  Categories: {row['n_categories']}  |  TR range: [{row['target_rate_range'][0]:.3f}, {row['target_rate_range'][1]:.3f}]")
    
    # Numeric-only ranking
    print("\n\n📈 TOP NUMERIC FEATURES")
    print("-"*80)
    
    df_numeric = df_metrics[df_metrics['type'] == 'numeric'].head(10)
    
    for i, row in df_numeric.iterrows():
        print(f"{list(df_numeric.index).index(i)+1:2d}. {row['feature']:20s}  |  d={row['cohens_d']:+.3f}  |  KS={row['ks_stat']:.3f}  |  AUC={row['auc_probe']:.3f}")
    
    # Categorical-only ranking
    print("\n\n📊 TOP CATEGORICAL FEATURES")
    print("-"*80)
    
    df_categorical = df_metrics[df_metrics['type'] == 'categorical'].head(10)
    
    if len(df_categorical) > 0:
        for i, row in df_categorical.iterrows():
            print(f"{list(df_categorical.index).index(i)+1:2d}. {row['feature']:20s}  |  V={row['cramers_v']:.3f}  |  n_cat={row['n_categories']}  |  AUC={row['auc_probe']:.3f}")
    else:
        print("(No categorical features)")


display_feature_rankings(df_separation, top_n=20)

# Save to CSV
csv_path = OUTPUT_DIR / "feature_rankings.csv"
df_separation.to_csv(csv_path, index=False)
print(f"\n💾 Saved rankings to: {csv_path}")


🏆 FEATURE RANKINGS (Top 20)

📊 OVERALL RANKING (by composite score)
--------------------------------------------------------------------------------

 1. nr.employed          (numeric)
    Composite: 0.3948  |  AUC: 0.7490  |  MI: 0.0617
    Cohen's d: -1.200  |  KS: 0.432  |  r_pb: -0.355

 2. euribor3m            (numeric)
    Composite: 0.3637  |  AUC: 0.7435  |  MI: 0.0550
    Cohen's d: -1.023  |  KS: 0.433  |  r_pb: -0.308

 3. emp.var.rate         (numeric)
    Composite: 0.3471  |  AUC: 0.7168  |  MI: 0.0534
    Cohen's d: -0.989  |  KS: 0.432  |  r_pb: -0.298

 4. previous             (categorical)
    Composite: 0.2966  |  AUC: 0.6093  |  MI: 0.0193
    Cramér's V: 0.236  |  Categories: 8.0  |  TR range: [0.000, 0.722]

 5. month                (categorical)
    Composite: 0.1937  |  AUC: 0.5239  |  MI: 0.0264
    Cramér's V: 0.274  |  Categories: 10.0  |  TR range: [0.064, 0.505]

 6. cons.price.idx       (numeric)
    Composite: 0.1697  |  AUC: 0.6106  |  MI: 0.0315
    Co

In [55]:
# ============================================================
# CELL 8: VISUALIZATIONS FOR TOP FEATURES
# ============================================================

def plot_numeric_feature(df: pd.DataFrame, 
                         feature: str, 
                         target_col: str,
                         save_path: Optional[Path] = None) -> None:
    """
    Create 3-panel visualization for numeric feature separation.
    
    Panels:
    1. Violin plot by class
    2. KDE overlay
    3. ECDF comparison
    
    Parameters
    ----------
    df : pd.DataFrame
        Dataset
    feature : str
        Feature name
    target_col : str
        Target column name
    save_path : Path, optional
        Where to save figure
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    class_0 = df[df[target_col] == 0][feature]
    class_1 = df[df[target_col] == 1][feature]
    
    # Panel 1: Violin plot
    parts = axes[0].violinplot(
        [class_0, class_1],
        positions=[0, 1],
        showmeans=True,
        showmedians=True
    )
    axes[0].set_xticks([0, 1])
    axes[0].set_xticklabels(['NOT_SUBSCRIBE', 'SUBSCRIBE'])
    axes[0].set_ylabel(feature)
    axes[0].set_title('Distribution by Class')
    axes[0].grid(True, alpha=0.3)
    
    # Panel 2: KDE overlay
    axes[1].hist(class_0, bins=30, alpha=0.5, label='NOT_SUBSCRIBE', density=True, color='C0')
    axes[1].hist(class_1, bins=30, alpha=0.5, label='SUBSCRIBE', density=True, color='C1')
    axes[1].set_xlabel(feature)
    axes[1].set_ylabel('Density')
    axes[1].set_title('Histogram Overlay (Normalized)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Panel 3: ECDF
    class_0_sorted = np.sort(class_0)
    class_1_sorted = np.sort(class_1)
    
    ecdf_0 = np.arange(1, len(class_0_sorted) + 1) / len(class_0_sorted)
    ecdf_1 = np.arange(1, len(class_1_sorted) + 1) / len(class_1_sorted)
    
    axes[2].plot(class_0_sorted, ecdf_0, label='NOT_SUBSCRIBE', alpha=0.7)
    axes[2].plot(class_1_sorted, ecdf_1, label='SUBSCRIBE', alpha=0.7)
    axes[2].set_xlabel(feature)
    axes[2].set_ylabel('ECDF')
    axes[2].set_title('Empirical CDF')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close()
    else:
        plt.show()


def plot_categorical_feature(df: pd.DataFrame,
                              feature: str,
                              target_col: str,
                              save_path: Optional[Path] = None) -> None:
    """
    Create 2-panel visualization for categorical feature separation.
    
    Panels:
    1. Target rate by category with Wilson CI
    2. Count distribution by class
    
    Parameters
    ----------
    df : pd.DataFrame
        Dataset
    feature : str
        Feature name
    target_col : str
        Target column name
    save_path : Path, optional
        Where to save figure
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Compute target rate and CI by category
    grouped = df.groupby(feature)[target_col].agg(['sum', 'count', 'mean'])
    grouped = grouped.sort_values('mean', ascending=False)
    
    categories = grouped.index.tolist()
    target_rates = grouped['mean'].values
    
    # Wilson confidence intervals
    cis = [wilson_ci(int(row['sum']), int(row['count'])) for _, row in grouped.iterrows()]
    ci_lower = np.array([ci[0] for ci in cis])
    ci_upper = np.array([ci[1] for ci in cis])
    
    # FIX: Ensure error bars are always non-negative
    # Sometimes edge cases (0% or 100% categories) can cause slight misalignment
    yerr_lower = np.maximum(0, target_rates - ci_lower)
    yerr_upper = np.maximum(0, ci_upper - target_rates)
    
    # Panel 1: Target rate with CI
    x_pos = np.arange(len(categories))
    axes[0].bar(x_pos, target_rates, alpha=0.6, color='steelblue')
    axes[0].errorbar(
        x_pos, target_rates,
        yerr=[yerr_lower, yerr_upper],  # Fixed: use clipped values
        fmt='none', ecolor='black', capsize=5, alpha=0.7
    )
    axes[0].set_xticks(x_pos)
    axes[0].set_xticklabels(categories, rotation=45, ha='right')
    axes[0].set_ylabel('P(SUBSCRIBE)')
    axes[0].set_title(f'Target Rate by {feature}')
    axes[0].axhline(df[target_col].mean(), color='red', linestyle='--', alpha=0.5, label='Overall rate')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Panel 2: Count distribution
    crosstab = pd.crosstab(df[feature], df[target_col])
    crosstab.plot(kind='bar', stacked=False, ax=axes[1], alpha=0.7)
    axes[1].set_xlabel(feature)
    axes[1].set_ylabel('Count')
    axes[1].set_title(f'Class Distribution by {feature}')
    axes[1].legend(['NOT_SUBSCRIBE', 'SUBSCRIBE'])
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close()
    else:
        plt.show()


def generate_all_plots(df: pd.DataFrame,
                       df_metrics: pd.DataFrame,
                       target_col: str,
                       top_n: int = 17) -> None:
    """
    Generate plots for top N features.
    
    Parameters
    ----------
    df : pd.DataFrame
        Dataset
    df_metrics : pd.DataFrame
        Feature separation metrics
    target_col : str
        Target column name
    top_n : int
        Number of top features to plot
    """
    print("\n" + "="*80)
    print(f"📊 GENERATING PLOTS FOR TOP {top_n} FEATURES")
    print("="*80)
    
    top_features = df_metrics.head(top_n)
    
    for i, row in top_features.iterrows():
        feature = row['feature']
        feature_type = row['type']
        
        print(f"\n[{i+1}/{top_n}] Plotting: {feature} ({feature_type})")
        
        save_path = FIG_DIR / f"{i+1:02d}_{feature}.png"
        
        if feature_type == 'numeric':
            plot_numeric_feature(df, feature, target_col, save_path)
        else:
            plot_categorical_feature(df, feature, target_col, save_path)
        
        print(f"    💾 Saved: {save_path.name}")
    
    print(f"\n✅ All plots saved to: {FIG_DIR}")


# Generate plots
generate_all_plots(df_processed, df_separation, TARGET_COL, top_n=17)


📊 GENERATING PLOTS FOR TOP 17 FEATURES

[1/17] Plotting: nr.employed (numeric)
    💾 Saved: 01_nr.employed.png

[2/17] Plotting: euribor3m (numeric)
    💾 Saved: 02_euribor3m.png

[3/17] Plotting: emp.var.rate (numeric)
    💾 Saved: 03_emp.var.rate.png

[4/17] Plotting: previous (categorical)
    💾 Saved: 04_previous.png

[5/17] Plotting: month (categorical)
    💾 Saved: 05_month.png

[6/17] Plotting: cons.price.idx (numeric)
    💾 Saved: 06_cons.price.idx.png

[7/17] Plotting: contact (categorical)
    💾 Saved: 07_contact.png

[8/17] Plotting: cons.conf.idx (numeric)
    💾 Saved: 08_cons.conf.idx.png

[9/17] Plotting: default (categorical)
    💾 Saved: 09_default.png

[10/17] Plotting: education (categorical)
    💾 Saved: 10_education.png

[11/17] Plotting: campaign (numeric)
    💾 Saved: 11_campaign.png

[12/17] Plotting: marital (categorical)
    💾 Saved: 12_marital.png

[13/17] Plotting: age (numeric)
    💾 Saved: 13_age.png

[14/17] Plotting: job (numeric)
    💾 Saved: 14_job.png

In [56]:
# ============================================================
# CELL 9: INTERACTION DISCOVERY
# ============================================================

def search_interactions_mi(df: pd.DataFrame,
                            features: List[str],
                            target_col: str,
                            top_k: int = 25,
                            max_pairs: int = 500) -> pd.DataFrame:
    """
    Search for feature interactions using mutual information.
    
    Strategy:
    - Sample top features only (to keep runtime reasonable)
    - Discretize both features into 5x5 grid
    - Compute MI(f1_binned, f2_binned → y)
    - Compare to sum of individual MIs (interaction lift)
    
    Parameters
    ----------
    df : pd.DataFrame
        Dataset
    features : List[str]
        Feature names to consider
    target_col : str
        Target column name
    top_k : int
        Number of top interactions to return
    max_pairs : int
        Maximum pairs to evaluate (runtime limit)
        
    Returns
    -------
    pd.DataFrame
        Top interaction candidates with lift scores
    """
    print("\n" + "="*80)
    print("🔍 INTERACTION DISCOVERY (Mutual Information)")
    print("="*80)
    
    # Limit to top features by individual MI
    print(f"\n📊 Computing individual MIs...")
    individual_mis = {}
    
    for feat in features:
        if df[feat].nunique() > 1:
            y = df[target_col].values
            
            # Discretize if numeric
            if feat in NUMERIC_FEATURES:
                n_bins = min(5, df[feat].nunique())
                discretizer = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='quantile')
                X_binned = discretizer.fit_transform(df[feat].values.reshape(-1, 1)).ravel()
            else:
                X_binned = df[feat].values
            
            mi = mutual_info_score(y, X_binned)
            individual_mis[feat] = mi
    
    # Sort and take top features
    top_features = sorted(individual_mis.items(), key=lambda x: x[1], reverse=True)[:30]
    top_feature_names = [f[0] for f in top_features]
    
    print(f"✅ Selected top {len(top_feature_names)} features for interaction search")
    
    # Enumerate pairs
    from itertools import combinations
    pairs = list(combinations(top_feature_names, 2))
    
    # Limit pairs
    if len(pairs) > max_pairs:
        print(f"⚠️  Limiting to {max_pairs} pairs (from {len(pairs)} possible)")
        pairs = pairs[:max_pairs]
    
    print(f"\n🔍 Evaluating {len(pairs)} pairs...")
    
    interaction_results = []
    y = df[target_col].values
    
    for idx, (f1, f2) in enumerate(pairs):
        if (idx + 1) % 50 == 0:
            print(f"  Progress: {idx+1}/{len(pairs)}", end='\r')
        
        # Discretize both features
        X1 = df[f1].values.reshape(-1, 1)
        X2 = df[f2].values.reshape(-1, 1)
        
        # Discretize
        n_bins_1 = min(5, df[f1].nunique())
        n_bins_2 = min(5, df[f2].nunique())
        
        disc1 = KBinsDiscretizer(n_bins=n_bins_1, encode='ordinal', strategy='quantile')
        disc2 = KBinsDiscretizer(n_bins=n_bins_2, encode='ordinal', strategy='quantile')
        
        X1_binned = disc1.fit_transform(X1).ravel() if f1 in NUMERIC_FEATURES else df[f1].values
        X2_binned = disc2.fit_transform(X2).ravel() if f2 in NUMERIC_FEATURES else df[f2].values
        
        # Create joint feature
        joint = X1_binned * 100 + X2_binned  # Simple hash
        
        # Compute MI of joint
        mi_joint = mutual_info_score(y, joint)
        
        # Compute individual MIs
        mi_f1 = individual_mis[f1]
        mi_f2 = individual_mis[f2]
        
        # Interaction lift
        mi_sum = mi_f1 + mi_f2
        lift = mi_joint - mi_sum if mi_sum > 0 else 0
        
        interaction_results.append({
            'feature_1': f1,
            'feature_2': f2,
            'mi_joint': mi_joint,
            'mi_f1': mi_f1,
            'mi_f2': mi_f2,
            'mi_sum': mi_sum,
            'lift': lift,
            'lift_pct': (lift / mi_sum * 100) if mi_sum > 0 else 0
        })
    
    print(f"\n✅ Evaluated {len(pairs)} pairs")
    
    # Convert to DataFrame and sort
    df_interactions = pd.DataFrame(interaction_results)
    df_interactions = df_interactions.sort_values('lift', ascending=False).reset_index(drop=True)
    
    return df_interactions.head(top_k)


# Search for interactions
df_interactions = search_interactions_mi(
    df_processed,
    NUMERIC_FEATURES + CATEGORICAL_FEATURES,
    TARGET_COL,
    top_k=25,
    max_pairs=500
)

print("\n" + "="*80)
print("🏆 TOP 25 INTERACTION CANDIDATES")
print("="*80)

for i, row in df_interactions.head(25).iterrows():
    print(f"\n{i+1:2d}. {row['feature_1']} × {row['feature_2']}")
    print(f"    Joint MI: {row['mi_joint']:.4f}  |  Sum MI: {row['mi_sum']:.4f}  |  Lift: {row['lift']:.4f} ({row['lift_pct']:.1f}%)")

# Save interactions
interactions_path = OUTPUT_DIR / "interaction_rankings.csv"
df_interactions.to_csv(interactions_path, index=False)
print(f"\n💾 Saved interactions to: {interactions_path}")



🔍 INTERACTION DISCOVERY (Mutual Information)

📊 Computing individual MIs...
✅ Selected top 17 features for interaction search

🔍 Evaluating 136 pairs...
  Progress: 100/136
✅ Evaluated 136 pairs

🏆 TOP 25 INTERACTION CANDIDATES

 1. month × cons.price.idx
    Joint MI: 0.0642  |  Sum MI: 0.0475  |  Lift: 0.0167 (35.1%)

 2. cons.price.idx × contact
    Joint MI: 0.0492  |  Sum MI: 0.0327  |  Lift: 0.0164 (50.2%)

 3. cons.price.idx × cons.conf.idx
    Joint MI: 0.0512  |  Sum MI: 0.0398  |  Lift: 0.0114 (28.6%)

 4. month × cons.conf.idx
    Joint MI: 0.0556  |  Sum MI: 0.0451  |  Lift: 0.0104 (23.1%)

 5. cons.price.idx × previous
    Joint MI: 0.0443  |  Sum MI: 0.0404  |  Lift: 0.0040 (9.9%)

 6. month × day_of_week
    Joint MI: 0.0296  |  Sum MI: 0.0267  |  Lift: 0.0028 (10.6%)

 7. month × contact
    Joint MI: 0.0404  |  Sum MI: 0.0381  |  Lift: 0.0024 (6.2%)

 8. cons.conf.idx × day_of_week
    Joint MI: 0.0211  |  Sum MI: 0.0190  |  Lift: 0.0020 (10.7%)

 9. age × education
 

In [57]:
# ============================================================
# CELL 10A: FEATURE ENGINEERING
# ============================================================

def engineer_bank_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create engineered features for bank marketing dataset.
    
    Tiers:
    1. Economic crisis indicators (highest priority)
    2. Interaction features (from MI analysis)
    3. Campaign timing features
    4. Customer segmentation
    5. Polynomial and ratio features
    
    Parameters
    ----------
    df : pd.DataFrame
        Processed dataframe with original features
        
    Returns
    -------
    pd.DataFrame
        Dataframe with original + engineered features
    """
    print("\n" + "="*80)
    print("🔧 FEATURE ENGINEERING")
    print("="*80)
    
    df_eng = df.copy()
    original_cols = len(df.columns)
    
    # ===================================================================
    # TIER 1: ECONOMIC CRISIS FEATURES (HIGHEST PRIORITY)
    # ===================================================================
    print("\n📊 Tier 1: Economic Crisis Indicators...")
    
    # Master crisis score
    df_eng['economic_crisis_score'] = (
        (df_eng['euribor3m'] < 1.5).astype(int) * 3 +
        (df_eng['emp.var.rate'] < -1).astype(int) * 2 +
        (df_eng['nr.employed'] < 5100).astype(int)
    )
    
    df_eng['in_crisis'] = (df_eng['economic_crisis_score'] >= 3).astype(int)
    df_eng['euribor_ultra_low'] = (df_eng['euribor3m'] < 1.0).astype(int)
    df_eng['euribor_very_low'] = (df_eng['euribor3m'] < 1.5).astype(int)
    df_eng['emp_var_negative'] = (df_eng['emp.var.rate'] < 0).astype(int)
    df_eng['emp_var_severe'] = (df_eng['emp.var.rate'] < -2).astype(int)
    df_eng['nr_employed_low'] = (df_eng['nr.employed'] < 5100).astype(int)
    
    print(f"   Created 7 crisis features")
    
    # ===================================================================
    # TIER 2: INTERACTION FEATURES
    # ===================================================================
    print("\n📊 Tier 2: Interaction Features...")
    
    # Month × Economic interactions
    df_eng['march_low_cpi'] = (
        (df_eng['month'] == 3) & (df_eng['cons.price.idx'] < 93.5)
    ).astype(int)
    
    df_eng['sept_low_cpi'] = (
        (df_eng['month'] == 9) & (df_eng['cons.price.idx'] < 93.5)
    ).astype(int)
    
    # Contact × Economic
    df_eng['cellular_low_cpi'] = (
        (df_eng['contact'] == 0) & (df_eng['cons.price.idx'] < 93.5)
    ).astype(int)
    
    df_eng['cellular_crisis'] = (
        (df_eng['contact'] == 0) & (df_eng['economic_crisis_score'] >= 3)
    ).astype(int)
    
    # Month × Confidence
    df_eng['good_month_low_confidence'] = (
        df_eng['month'].isin([3, 9, 10, 12]) & 
        (df_eng['cons.conf.idx'] < -40)
    ).astype(int)
    
    # Previous × Economic
    df_eng['vip_in_crisis'] = (
        (df_eng['previous'] >= 5) & (df_eng['economic_crisis_score'] >= 3)
    ).astype(int)
    
    # Education × Economic
    df_eng['low_edu_crisis'] = (
        (df_eng['education'] <= 1) & (df_eng['euribor3m'] < 1.5)
    ).astype(int)
    
    print(f"   Created 7 interaction features")
    
    # ===================================================================
    # TIER 3: CAMPAIGN TIMING
    # ===================================================================
    print("\n📊 Tier 3: Campaign Timing Features...")
    
    df_eng['high_conversion_month'] = df_eng['month'].isin([3, 9, 10, 12]).astype(int)
    df_eng['avoid_month'] = df_eng['month'].isin([5, 6, 7]).astype(int)
    df_eng['Q1'] = (df_eng['month'] <= 3).astype(int)
    df_eng['Q4'] = (df_eng['month'] >= 10).astype(int)
    df_eng['december'] = (df_eng['month'] == 12).astype(int)
    
    print(f"   Created 5 timing features")
    
    # ===================================================================
    # TIER 4: CUSTOMER SEGMENTATION
    # ===================================================================
    print("\n📊 Tier 4: Customer Segmentation...")
    
    # Contact quality
    df_eng['cellular_contact'] = (df_eng['contact'] == 0).astype(int)
    df_eng['premium_contact'] = (
        (df_eng['contact'] == 0) & (df_eng['previous'] >= 1)
    ).astype(int)
    
    # Education segments
    df_eng['education_low'] = (df_eng['education'] <= 1).astype(int)
    df_eng['education_high'] = (df_eng['education'] >= 5).astype(int)
    df_eng['education_unknown'] = (df_eng['education'] == -1).astype(int)
    
    df_eng['financially_vulnerable'] = (
        (df_eng['education'] <= 2) & (df_eng['default'] != 1)
    ).astype(int)
    
    # Previous contact segments
    df_eng['warm_lead'] = (df_eng['previous'] >= 1).astype(int)
    df_eng['hot_lead'] = (df_eng['previous'] >= 5).astype(int)
    
    # Unknown flags
    df_eng['default_unknown'] = (df_eng['default'] == -1).astype(int)
    
    print(f"   Created 9 segmentation features")
    
    # ===================================================================
    # TIER 5: POLYNOMIAL & RATIOS
    # ===================================================================
    print("\n📊 Tier 5: Polynomial & Ratio Features...")
    
    df_eng['euribor_squared'] = df_eng['euribor3m'] ** 2
    df_eng['emp_var_squared'] = df_eng['emp.var.rate'] ** 2
    
    # Safe ratios (avoid division by zero and extreme values)
    # Clip denominators to avoid extreme ratios
    emp_var_safe = df_eng['emp.var.rate'].clip(lower=-5, upper=5) + 3.1  # Shift to ensure positive
    cons_conf_safe = df_eng['cons.conf.idx'].clip(lower=-60, upper=-20) + 50.1  # Shift to ensure positive
    
    df_eng['euribor_emp_ratio'] = df_eng['euribor3m'] / emp_var_safe
    df_eng['cpi_confidence_ratio'] = df_eng['cons.price.idx'] / cons_conf_safe
    
    # Clip ratios to reasonable bounds
    df_eng['euribor_emp_ratio'] = df_eng['euribor_emp_ratio'].clip(lower=-10, upper=10)
    df_eng['cpi_confidence_ratio'] = df_eng['cpi_confidence_ratio'].clip(lower=-10, upper=10)
    
    print(f"   Created 4 polynomial/ratio features")
    print(f"   Ratios clipped to [-10, 10] range for stability")
    
    # ===================================================================
    # SUMMARY & VERIFICATION
    # ===================================================================
    new_features = len(df_eng.columns) - original_cols
    
    print("\n" + "="*80)
    print(f"✅ Feature Engineering Complete")
    print(f"   Original features: {original_cols}")
    print(f"   New features: {new_features}")
    print(f"   Total features: {len(df_eng.columns)}")
    
    # Verify original features are preserved
    original_features = df.columns.tolist()
    preserved_features = [col for col in original_features if col in df_eng.columns]
    
    print(f"\n🔍 Verification:")
    print(f"   Original features preserved: {len(preserved_features)}/{len(original_features)}")
    
    if len(preserved_features) == len(original_features):
        print(f"   ✅ All original features intact!")
        print(f"\n   Original features: {', '.join(preserved_features[:10])}{'...' if len(preserved_features) > 10 else ''}")
    else:
        print(f"   ⚠️  WARNING: Some features missing!")
        missing = set(original_features) - set(preserved_features)
        print(f"   Missing: {missing}")
    
    # Check for inf/nan values
    inf_cols = df_eng.columns[np.isinf(df_eng).any()].tolist()
    nan_cols = df_eng.columns[df_eng.isnull().any()].tolist()
    
    if inf_cols:
        print(f"\n   ⚠️  Infinity values detected in: {inf_cols}")
        print(f"      These will be replaced with NaN and then median-imputed")
    if nan_cols:
        print(f"\n   ⚠️  NaN values detected in: {nan_cols}")
        print(f"      These will be median-imputed")
    
    # Clean up inf/nan values
    if inf_cols or nan_cols:
        df_eng = df_eng.replace([np.inf, -np.inf], np.nan)
        
        # Impute with median (excluding target column)
        feature_cols = [col for col in df_eng.columns if col != 'y']
        for col in feature_cols:
            if df_eng[col].isnull().any():
                median_val = df_eng[col].median()
                df_eng[col] = df_eng[col].fillna(median_val)
                print(f"      Imputed {col} with median: {median_val:.4f}")
    else:
        print(f"\n   ✅ No inf/NaN values detected")
    
    print("="*80)
    
    return df_eng


# Apply feature engineering
df_engineered = engineer_bank_features(df_processed)

# Quick check
print(f"\n📊 Engineered dataset shape: {df_engineered.shape}")
print(f"   Target distribution: {df_engineered[TARGET_COL].value_counts().to_dict()}")



🔧 FEATURE ENGINEERING

📊 Tier 1: Economic Crisis Indicators...
   Created 7 crisis features

📊 Tier 2: Interaction Features...
   Created 7 interaction features

📊 Tier 3: Campaign Timing Features...
   Created 5 timing features

📊 Tier 4: Customer Segmentation...
   Created 9 segmentation features

📊 Tier 5: Polynomial & Ratio Features...
   Created 4 polynomial/ratio features
   Ratios clipped to [-10, 10] range for stability

✅ Feature Engineering Complete
   Original features: 18
   New features: 32
   Total features: 50

🔍 Verification:
   Original features preserved: 18/18
   ✅ All original features intact!

   Original features: age, job, marital, education, default, housing, loan, contact, month, day_of_week...

   ✅ No inf/NaN values detected

📊 Engineered dataset shape: (41188, 50)
   Target distribution: {0: 36548, 1: 4640}


In [58]:
# ============================================================
# CELL 10B: CALCULUS-BASED INTEGRAL FEATURES
# ============================================================
"""
ADVANCED FEATURE ENGINEERING: Integral-Based Features
======================================================

CONTEXT FOR OPUS (Future Analysis):
-----------------------------------
We discovered that standard ML features struggle in "overlap zones" where 
P(subscribe) ≈ 0.3-0.7. In these regions, class distributions heavily overlap,
causing poor precision. Our hypothesis: calculus-based features (derivatives + 
integrals) can disambiguate these zones while maintaining interpretability 
through careful business framing.

This cell implements INTEGRAL features specifically. Tomorrow we'll analyze:
1. Do these features improve Recall@10%FPR by 3-5%?
2. Do they rank in top 15-20 importance for RF/EBM?
3. Do they help specifically in overlap zones?
4. Are they interpretable enough for production use?

If yes → Apply to cascade pipeline (LR → GLASS-BRW → EBM)

Mathematical Foundation:
------------------------
- Derivatives capture INSTANTANEOUS patterns (velocity, acceleration, inflection)
- Integrals capture ACCUMULATED patterns (cumulative effects, density, volume)
- Together: Maximum disambiguation power in overlap zones

Strategy: Create features that accumulate multi-dimensional signals to separate
overlapping class distributions where standard features fail.

Key Innovation: Using calculus NOT for complexity, but for INTERPRETABILITY
- Each integral has clear business meaning
- "Economic stress integral" = total accumulated pressure
- "Neighborhood density" = local subscription clustering
- "Campaign pressure" = repeated contact under stress

This challenges ML dogma that "complex features = uninterpretable"
"""

print("\n" + "="*80)
print("🧮 CALCULUS-BASED INTEGRAL FEATURES")
print("="*80)
print("\nPhilosophy: In overlap zones where P(subscribe) ≈ 0.3-0.7,")
print("standard features fail. Integral features capture accumulated")
print("patterns that differentiate subscribers from non-subscribers.")
print("="*80)


def create_integral_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create calculus-based integral features for overlap zone disambiguation.
    
    Features created:
    1. Economic Stress Integral - Multi-dimensional accumulation
    2. Neighborhood Subscription Density - Spatial probability mass
    3. Cumulative Campaign Pressure - Pseudo-temporal accumulation
    4. Pressure-Density Synergy - Integral × integral interaction
    
    Parameters
    ----------
    df : pd.DataFrame
        Engineered dataframe with base features
        
    Returns
    -------
    df : pd.DataFrame
        Dataframe with additional integral features
    """
    print("\n📊 Feature 1: Economic Stress Integral")
    print("-"*80)
    
    # =========================================================================
    # FEATURE 1: ECONOMIC STRESS INTEGRAL
    # =========================================================================
    """
    Mathematical Formulation:
    -------------------------
    S_integral = (1/W) · Σᵢ wᵢ · fᵢ(xᵢ)
    
    Where:
    - f₁(euribor)   = max(0, (1.5 - euribor) / 1.5)      [Crisis proximity]
    - f₂(emp_var)   = max(0, (-emp_var) / 3)             [Employment decline]
    - f₃(nr_empl)   = max(0, (5200 - nr_employed) / 100) [Employment deficit]
    - w₁=3, w₂=2, w₃=1 (weights from feature importance analysis)
    - W = 6 (normalization constant)
    
    This approximates the continuous integral:
    
        S_integral = ∫ [w₁·f₁(x) + w₂·f₂(x) + w₃·f₃(x)] d(feature_space)
    
    By discretizing over feature dimensions and using weighted sum (Riemann sum).
    
    Business Interpretation:
    ------------------------
    "Total accumulated economic pressure across multiple dimensions"
    
    Scale:
    - 0.0 = No economic stress (normal times, all indicators healthy)
    - 0.3 = Mild stress (one factor slightly stressed)
    - 0.5 = Moderate stress (multiple factors in decline)
    - 0.7 = High stress (most factors stressed)
    - 1.0 = Severe crisis (all factors at maximum stress)
    
    Why this works:
    ---------------
    1. Individual features overlap heavily in 0.3-0.7 probability zone
    2. But their WEIGHTED INTEGRAL is more separable
    3. Captures multi-dimensional stress that single features miss
    4. Different from simple sum - each dimension weighted by predictive power
    5. Normalized to [0,1] for interpretability
    
    Example Cases:
    -------------
    Case A: euribor=2.0, emp_var=0.5, nr_emp=5150
    → Low stress on all → S_integral ≈ 0.15 (normal economy)
    
    Case B: euribor=1.2, emp_var=-0.5, nr_emp=5120
    → Moderate stress → S_integral ≈ 0.45 (soft recession)
    
    Case C: euribor=0.8, emp_var=-2.5, nr_emp=5050
    → High stress on all → S_integral ≈ 0.85 (severe crisis)
    """
    
    # Component 1: Interest rate stress (highest weight=3)
    # Normalized stress increases as euribor drops below 1.5% threshold
    euribor_stress = np.maximum(0, (1.5 - df['euribor3m']) / 1.5)
    
    # Component 2: Employment variation stress (weight=2)
    # Normalized stress increases as employment variation becomes negative
    emp_var_stress = np.maximum(0, (-df['emp.var.rate']) / 3)
    
    # Component 3: Employment level stress (weight=1)
    # Normalized stress increases as employment drops below 5200 threshold
    nr_employed_stress = np.maximum(0, (5200 - df['nr.employed']) / 100)
    
    # Weighted integral using trapezoidal rule approximation
    # This is the discrete approximation of the continuous integral
    df['economic_stress_integral'] = (
        euribor_stress * 3 +        # Weight=3 (strongest signal from analysis)
        emp_var_stress * 2 +         # Weight=2 (secondary signal)
        nr_employed_stress * 1       # Weight=1 (tertiary signal)
    ) / 6  # Normalize to [0, 1] range (W=6 is sum of weights)
    
    print(f"   ✅ Created: economic_stress_integral")
    print(f"   Formula: (3·euribor_stress + 2·emp_stress + 1·nr_emp_stress) / 6")
    print(f"   Range: [{df['economic_stress_integral'].min():.3f}, {df['economic_stress_integral'].max():.3f}]")
    print(f"   Mean:  {df['economic_stress_integral'].mean():.3f}")
    print(f"   Std:   {df['economic_stress_integral'].std():.3f}")
    print(f"   Business: Total accumulated economic pressure (0=normal, 1=severe crisis)")
    
    
    print("\n📊 Feature 2: Neighborhood Subscription Density")
    print("-"*80)
    
    # =========================================================================
    # FEATURE 2: NEIGHBORHOOD SUBSCRIPTION DENSITY (Spatial Integral)
    # =========================================================================
    """
    Mathematical Formulation:
    -------------------------
    ρ(x) = ∫∫∫ P(y=1 | x') · K(||x - x'||) dx'
    
    Where:
    - P(y=1 | x') = Target probability at point x' in feature space
    - K(d) = exp(-d²/(2σ²)) = Gaussian kernel with bandwidth σ
    - ||x - x'|| = Euclidean distance: √[Σᵢ(xᵢ - x'ᵢ)²]
    - σ = 1.0 (bandwidth parameter)
    
    This is Kernel Density Estimation (KDE) of subscription probability.
    
    Discrete approximation (what we actually compute):
    
        ρ(x) ≈ Σⱼ yⱼ · wⱼ(x)
        
        where: wⱼ(x) = exp(-dⱼ²/2) / Σₖ exp(-dₖ²/2)
               dⱼ = ||x - xⱼ||  (Euclidean distance to point j)
               yⱼ = target value at point j (0 or 1)
    
    Business Interpretation:
    ------------------------
    "How common are subscribers in this customer's neighborhood?"
    "What's the local subscription rate among similar customers?"
    
    Scale:
    - 0.0-0.1 = Very sparse subscribers (unlikely to convert)
    - 0.1-0.3 = Below-average subscriber density
    - 0.3-0.7 = Mixed neighborhood (OVERLAP ZONE - needs disambiguation)
    - 0.7-0.9 = High subscriber density
    - 0.9-1.0 = Very dense cluster of subscribers (very likely to convert)
    
    Why this works:
    ---------------
    1. Captures LOCAL context that global features miss
    2. "Birds of a feather" principle - cluster membership matters
    3. Smooths over noise by integrating across neighborhood
    4. KEY: In overlap zones where euribor is ambiguous, local density helps!
    
    Example: Two customers with euribor=2.0 (ambiguous)
    - Customer A: In dense subscriber cluster → ρ=0.8 → Likely subscribes
    - Customer B: Isolated in non-subscriber area → ρ=0.2 → Likely doesn't
    
    Computational Note:
    ------------------
    For large datasets (>10k), we use reference point approximation:
    - Sample 5000 reference points randomly
    - Compute distances only to these points (not all N²)
    - Reduces O(N²) to O(N·5000) while maintaining accuracy
    """
    
    print("   Computing spatial density integral (Gaussian kernel)...")
    print("   Using top 3 economic features: euribor3m, nr.employed, emp.var.rate")
    print("   This may take 30-60 seconds for 41k samples...")
    
    # Use top 3 economic features for distance calculation
    # (These showed highest separation in feature analysis)
    key_features = ['euribor3m', 'nr.employed', 'emp.var.rate']
    
    # Normalize features for distance calculation (ensure equal weighting)
    from sklearn.preprocessing import StandardScaler
    scaler_density = StandardScaler()
    X_normalized = scaler_density.fit_transform(df[key_features])
    
    # Compute neighborhood density using Gaussian kernel
    from scipy.spatial.distance import cdist
    
    # For efficiency on large datasets, use reference point approximation
    if len(df) > 10000:
        # Sample reference points for kernel density estimation
        np.random.seed(42)
        n_ref = min(5000, len(df))
        sample_idx = np.random.choice(len(df), size=n_ref, replace=False)
        X_reference = X_normalized[sample_idx]
        y_reference = df[TARGET_COL].iloc[sample_idx].values
        
        # Compute distances: d[i,j] = ||x_i - x_ref_j||
        distances = cdist(X_normalized, X_reference, metric='euclidean')
        
        # Gaussian kernel weights: K(d) = exp(-d²/2σ²) with σ=1.0
        bandwidth = 1.0
        kernel_weights = np.exp(-(distances**2) / (2 * bandwidth**2))
        
        # Normalize weights so each row sums to 1 (probability distribution)
        kernel_weights = kernel_weights / (kernel_weights.sum(axis=1, keepdims=True) + 1e-10)
        
        # Weighted average of target values (integral approximation)
        # This computes: ρ(x) ≈ Σⱼ yⱼ · wⱼ(x)
        densities = np.dot(kernel_weights, y_reference)
    else:
        # Full computation for smaller datasets (use all points)
        distances = cdist(X_normalized, X_normalized, metric='euclidean')
        bandwidth = 1.0
        kernel_weights = np.exp(-(distances**2) / (2 * bandwidth**2))
        kernel_weights = kernel_weights / (kernel_weights.sum(axis=1, keepdims=True) + 1e-10)
        densities = np.dot(kernel_weights, df[TARGET_COL].values)
    
    df['neighborhood_subscription_density'] = densities
    
    print(f"   ✅ Created: neighborhood_subscription_density")
    print(f"   Formula: ρ(x) = Σⱼ P(yⱼ) · exp(-||x-xⱼ||²/2) / Z")
    print(f"   Range: [{df['neighborhood_subscription_density'].min():.3f}, {df['neighborhood_subscription_density'].max():.3f}]")
    print(f"   Mean:  {df['neighborhood_subscription_density'].mean():.3f}")
    print(f"   Std:   {df['neighborhood_subscription_density'].std():.3f}")
    print(f"   Business: Local subscription rate in customer's economic neighborhood")
    
    
    print("\n📊 Feature 3: Cumulative Campaign Pressure")
    print("-"*80)
    
    # =========================================================================
    # FEATURE 3: CUMULATIVE CAMPAIGN PRESSURE (Pseudo-Temporal Integral)
    # =========================================================================
    """
    Mathematical Formulation:
    -------------------------
    C_integral = ∫₀ᵗ S(τ) dτ
    
    Where:
    - t = number of previous contacts (time proxy)
    - S(τ) = economic stress at contact τ
    
    Discrete approximation (assuming constant stress):
    
        C_integral ≈ t · S̄ = previous_contacts · economic_stress_integral
    
    This assumes each previous contact occurred under similar economic conditions
    (reasonable if contacts clustered in time during campaign).
    
    More formally, using Riemann sum:
    
        ∫₀ᵗ S(τ) dτ ≈ Σᵢ₌₁ᵗ S(τᵢ) · Δτ
                     ≈ S_current · t  (if stress is roughly constant)
    
    Business Interpretation:
    ------------------------
    "Total accumulated marketing pressure under economic stress"
    "Repeated contact intensity weighted by economic conditions"
    
    Scale:
    - 0.0 = No previous contact OR no economic stress
    - 0.0-0.5 = Few contacts OR low stress (normal marketing)
    - 0.5-2.0 = Moderate contacts with moderate stress
    - 2.0-5.0 = Many contacts under high stress (intensive campaign in crisis)
    - 5.0+ = VIP customers heavily contacted during severe crisis
    
    Why this works:
    ---------------
    1. VIP customers (previous≥5) under stress behave VERY differently from
       VIP customers in normal times - this captures that difference
    2. Captures accumulated effect of repeated contact during crisis
    3. This is an INTERACTION INTEGRAL: ∫ f(previous) · g(economy) dt
    4. Not just previous × stress, but accumulated exposure over pseudo-time
    
    Example Cases:
    -------------
    Case A: previous=0, stress=0.8 → C=0.0 (crisis but no contact)
    Case B: previous=3, stress=0.2 → C=0.6 (some contact, normal times)
    Case C: previous=7, stress=0.8 → C=5.6 (VIP heavily contacted in crisis)
    
    Case C is the "perfect storm" - should have highest conversion!
    """
    
    df['cumulative_campaign_pressure'] = (
        df['previous'] * df['economic_stress_integral']
    )
    
    print(f"   ✅ Created: cumulative_campaign_pressure")
    print(f"   Formula: ∫₀ᵗ S(τ)dτ ≈ previous · economic_stress_integral")
    print(f"   Range: [{df['cumulative_campaign_pressure'].min():.3f}, {df['cumulative_campaign_pressure'].max():.3f}]")
    print(f"   Mean:  {df['cumulative_campaign_pressure'].mean():.3f}")
    print(f"   Std:   {df['cumulative_campaign_pressure'].std():.3f}")
    print(f"   Business: Total marketing pressure accumulated during economic stress")
    
    
    print("\n📊 Feature 4: Pressure-Density Synergy")
    print("-"*80)
    
    # =========================================================================
    # FEATURE 4: PRESSURE-DENSITY SYNERGY (Integral × Integral Interaction)
    # =========================================================================
    """
    Mathematical Formulation:
    -------------------------
    Synergy(x) = [∫₀ᵗ S(τ)dτ] × [∫∫∫ P(y|x')·K(||x-x'||)dx']
    
    Simplified:
        Synergy = cumulative_campaign_pressure × neighborhood_subscription_density
    
    This is the product of two integrals:
    - Temporal integral (accumulated pressure over contacts)
    - Spatial integral (probability mass in neighborhood)
    
    Business Interpretation:
    ------------------------
    "High-pressure customers in high-conversion neighborhoods"
    "VIP customers under stress, surrounded by subscribers"
    
    Why this is powerful:
    --------------------
    1. Each integral alone is informative
    2. But their PRODUCT identifies the "sweet spot":
       - High campaign pressure (VIP under stress)
       - AND in high-subscriber cluster
       → MAXIMUM conversion probability
    
    3. This captures multi-dimensional synergy that linear models miss
    
    Example Scenarios:
    -----------------
    A: High pressure (5.0), low density (0.2) → Synergy=1.0
       "VIP in crisis, but isolated" → Moderate conversion
    
    B: Low pressure (0.5), high density (0.8) → Synergy=0.4
       "In good cluster, but not contacted much" → Moderate conversion
    
    C: High pressure (5.0), high density (0.8) → Synergy=4.0
       "VIP in crisis, in subscriber cluster" → HIGHEST conversion!
    
    This is exactly the kind of non-linear interaction that disambiguates
    overlap zones where linear combinations fail.
    """
    
    df['pressure_density_synergy'] = (
        df['cumulative_campaign_pressure'] * 
        df['neighborhood_subscription_density']
    )
    
    print(f"   ✅ Created: pressure_density_synergy")
    print(f"   Formula: [∫S(τ)dτ] × [∫P(x')K(x,x')dx'] = C_pressure × ρ_density")
    print(f"   Range: [{df['pressure_density_synergy'].min():.3f}, {df['pressure_density_synergy'].max():.3f}]")
    print(f"   Mean:  {df['pressure_density_synergy'].mean():.3f}")
    print(f"   Std:   {df['pressure_density_synergy'].std():.3f}")
    print(f"   Business: Synergy between campaign intensity and local conversion clusters")
    
    return df


# Apply integral feature engineering
print("\n🔧 Applying integral feature engineering...")
df_engineered = create_integral_features(df_engineered)

print("\n" + "="*80)
print("✅ CALCULUS-BASED FEATURES COMPLETE")
print("="*80)
print(f"\n📊 Total features now: {len(df_engineered.columns)}")
print(f"   Original features:              17")
print(f"   Engineered (Tier 1-5):          32")
print(f"   Calculus-based integrals:        4")
print(f"   TOTAL:                          {len(df_engineered.columns)} features")

print("\n🧮 Integral Feature Summary:")
print(f"   1. economic_stress_integral           - Weighted multi-dimensional accumulation")
print(f"   2. neighborhood_subscription_density  - Spatial probability mass (Gaussian kernel)")
print(f"   3. cumulative_campaign_pressure       - Pseudo-temporal stress accumulation")
print(f"   4. pressure_density_synergy           - Integral × integral interaction")

print("\n🎯 Expected Impact on Model Performance:")
print("   Primary Goal: Improve separation in overlap zones (P ≈ 0.3-0.7)")
print("   Expected: +2-5% Recall@10%FPR improvement")
print("   Features should rank in top 15-20 importance for RF/EBM")
print("   Test: Compare RF/EBM performance with vs without these features")

print("\n📋 FOR OPUS (Tomorrow's Analysis):")
print("   1. Check if integral features rank in top 20 importance")
print("   2. Compute Recall@10%FPR improvement vs baseline")
print("   3. Analyze if they help specifically in 0.3-0.7 probability zone")
print("   4. Validate interpretability for stakeholders")
print("   5. If successful → Apply to cascade pipeline")

print("\n" + "="*80)



🧮 CALCULUS-BASED INTEGRAL FEATURES

Philosophy: In overlap zones where P(subscribe) ≈ 0.3-0.7,
standard features fail. Integral features capture accumulated
patterns that differentiate subscribers from non-subscribers.

🔧 Applying integral feature engineering...

📊 Feature 1: Economic Stress Integral
--------------------------------------------------------------------------------
   ✅ Created: economic_stress_integral
   Formula: (3·euribor_stress + 2·emp_stress + 1·nr_emp_stress) / 6
   Range: [0.000, 0.944]
   Mean:  0.185
   Std:   0.276
   Business: Total accumulated economic pressure (0=normal, 1=severe crisis)

📊 Feature 2: Neighborhood Subscription Density
--------------------------------------------------------------------------------
   Computing spatial density integral (Gaussian kernel)...
   Using top 3 economic features: euribor3m, nr.employed, emp.var.rate
   This may take 30-60 seconds for 41k samples...
   ✅ Created: neighborhood_subscription_density
   Formula: ρ(x) =

In [59]:
# ============================================================
# CELL 10C: DERIVATIVE / SLOPE / DECAY FEATURES
# ============================================================
"""
DERIVATIVE-BASED FEATURE ENGINEERING
=====================================

From the distribution plots, we observe clear slope/decay behavior:

1. nr.employed: Sharp cliff around 5100-5150. Below 5100, subscription 
   probability jumps dramatically. The ECDF shows a steep S-curve with 
   the SUBSCRIBE curve shifted left — the derivative (slope) of that 
   shift is maximal right at the cliff edge.

2. euribor3m: Bimodal distribution with most subscribers clustered <1.5.
   The ECDF shows SUBSCRIBE curve racing ahead of NOT_SUBSCRIBE below ~2.0,
   then flattening. Classic exponential decay in conversion probability 
   as rates rise.

3. emp.var.rate: Similar story — subscribers concentrate around -1.8 to -2.0.
   The density peak for SUBSCRIBE is shifted left. The derivative of 
   P(subscribe|emp.var.rate) has a steep negative slope crossing 0.

APPROACH:
---------
A) SIGMOID SLOPE FEATURES — Model transition zones as logistic functions.
   First derivative of sigmoid peaks at inflection point = "cliff edge proximity"

B) EXPONENTIAL DECAY FEATURES — Model the decay pattern directly.
   P(subscribe) ≈ exp(-λ·x) for certain features.

C) PIECEWISE GRADIENT FEATURES — Bin feature, compute empirical P(subscribe)
   per bin, take discrete first-differences (∂P/∂x).

D) CURVATURE (SECOND DERIVATIVE) — Where slope changes fastest (inflection 
   points), that's where models need the most help.

Complementary to integral features:
- Integrals = accumulated area (how much stress total)
- Derivatives = instantaneous rate of change (how fast things are shifting)
"""

def create_derivative_features(df: pd.DataFrame, target_col: str = 'y') -> pd.DataFrame:
    """
    Create derivative/slope/decay features for economic indicators.
    
    Parameters
    ----------
    df : pd.DataFrame
        Engineered dataframe with base + integral features
    target_col : str
        Name of binary target column
        
    Returns
    -------
    pd.DataFrame
        Dataframe with additional derivative features
    """
    print("\n" + "="*80)
    print("📐 DERIVATIVE / SLOPE / DECAY FEATURES")
    print("="*80)
    print("\nPhilosophy: Integrals capture accumulation. Derivatives capture")
    print("the RATE OF CHANGE — where distributions shift fastest between classes.")
    print("These target the cliff edges and inflection points visible in our plots.")
    print("="*80)
    
    original_cols = set(df.columns)
    
    # =================================================================
    # PART A: SIGMOID SLOPE FEATURES
    # =================================================================
    """
    Sigmoid slope captures "distance from the cliff edge" in a smooth,
    differentiable way. The first derivative of σ(x) = σ(x)·(1-σ(x)),
    which peaks at the inflection point (the cliff).
    
    We fit sigmoid centers and scales from the distribution plots:
    - nr.employed: Cliff at ~5100, sharp transition (scale ~30)
    - euribor3m:   Cliff at ~1.5, moderate transition (scale ~0.5)  
    - emp.var.rate: Cliff at ~-1.0, moderate transition (scale ~0.5)
    
    For each feature x:
      sigmoid_value = σ(-(x - center) / scale)   [probability-like, 0→1]
      sigmoid_slope = σ'(x) = value * (1 - value) [peaks at inflection]
    """
    print("\n📊 Part A: Sigmoid Slope Features")
    print("-"*80)
    
    # Parameters tuned from distribution analysis
    # (center, scale) — center is the inflection point, scale is steepness
    sigmoid_params = {
        'nr.employed':  (5100, 30),   # Sharp cliff at 5100
        'euribor3m':    (1.5, 0.5),   # Transition around 1.5%
        'emp.var.rate': (-1.0, 0.5),  # Transition around -1.0
    }
    
    for feat, (center, scale) in sigmoid_params.items():
        if feat not in df.columns:
            print(f"   ⚠️ {feat} not found, skipping")
            continue
        
        # Sigmoid VALUE: smooth 0-1 transform centered at cliff
        # Negative sign so that LOWER values → HIGHER sigmoid (matches subscribe direction)
        z = -(df[feat] - center) / scale
        sig_val = expit(z)  # 1/(1+exp(-z))
        
        # Sigmoid SLOPE (first derivative): peaks at the cliff edge
        # σ'(z) = σ(z) · (1 - σ(z))
        # Maximum at z=0 (the inflection point), decays symmetrically
        sig_slope = sig_val * (1 - sig_val)
        
        # Normalize slope to [0, 1] (max of sigmoid derivative is 0.25 at z=0)
        sig_slope_norm = sig_slope / 0.25
        
        safe_name = feat.replace('.', '_')
        df[f'{safe_name}_sigmoid'] = sig_val
        df[f'{safe_name}_sigmoid_slope'] = sig_slope_norm
        
        print(f"   ✅ {safe_name}_sigmoid: σ(-(x-{center})/{scale})")
        print(f"      Range: [{sig_val.min():.3f}, {sig_val.max():.3f}], "
              f"Mean: {sig_val.mean():.3f}")
        print(f"   ✅ {safe_name}_sigmoid_slope: peaks at x={center} (the cliff)")
        print(f"      Range: [{sig_slope_norm.min():.3f}, {sig_slope_norm.max():.3f}], "
              f"Mean: {sig_slope_norm.mean():.3f}")
    
    # =================================================================
    # PART B: EXPONENTIAL DECAY FEATURES
    # =================================================================
    """
    From the ECDF plots, P(subscribe) decays roughly exponentially as 
    euribor3m and nr.employed increase. We model:
    
        P̂(subscribe|x) ≈ A · exp(-λ · (x - x_min))
    
    The DECAY RATE itself (λ·x term) is a monotonic compression that 
    captures the exponential nature better than linear features.
    """
    print("\n📊 Part B: Exponential Decay Features")
    print("-"*80)
    
    # euribor3m: decay from ~0.5 subscribe rate at euribor=0.6 to ~0.05 at euribor=5
    euribor_decay = np.exp(-0.5 * np.maximum(0, df['euribor3m'] - 0.6))
    df['euribor_decay'] = euribor_decay
    print(f"   ✅ euribor_decay: exp(-0.5 · max(0, euribor - 0.6))")
    print(f"      Range: [{euribor_decay.min():.3f}, {euribor_decay.max():.3f}], "
          f"Mean: {euribor_decay.mean():.3f}")
    print(f"      Business: Expected conversion boost from low interest rates")
    
    # nr.employed: decay from low employment toward high
    nr_decay = np.exp(-0.02 * np.maximum(0, df['nr.employed'] - 4964))
    df['nr_employed_decay'] = nr_decay
    print(f"   ✅ nr_employed_decay: exp(-0.02 · max(0, nr.employed - 4964))")
    print(f"      Range: [{nr_decay.min():.3f}, {nr_decay.max():.3f}], "
          f"Mean: {nr_decay.mean():.3f}")
    print(f"      Business: Employment level conversion decay")
    
    # emp.var.rate: decay as employment variation increases from negative
    emp_decay = np.exp(-0.8 * np.maximum(0, df['emp.var.rate'] + 2.0))
    df['emp_var_decay'] = emp_decay
    print(f"   ✅ emp_var_decay: exp(-0.8 · max(0, emp.var.rate + 2.0))")
    print(f"      Range: [{emp_decay.min():.3f}, {emp_decay.max():.3f}], "
          f"Mean: {emp_decay.mean():.3f}")
    print(f"      Business: Employment decline severity (decay from peak)")
    
    # =================================================================
    # PART C: PIECEWISE GRADIENT FEATURES (Empirical ∂P/∂x)
    # =================================================================
    """
    Compute the empirical gradient of P(subscribe) w.r.t. each feature.
    
    Method:
    1. Bin the feature into quantile-based bins
    2. Compute P(subscribe) per bin → empirical response curve
    3. Take first differences: ΔP/Δx per bin
    4. Map each observation to its bin's gradient
    
    High gradient = you're on the slope (model uncertainty zone)
    Low gradient = you're on a plateau (model confident)
    """
    print("\n📊 Part C: Piecewise Gradient Features (∂P/∂x)")
    print("-"*80)
    
    gradient_features = ['euribor3m', 'nr.employed', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx']
    n_bins = 20
    
    for feat in gradient_features:
        if feat not in df.columns:
            continue
        
        safe_name = feat.replace('.', '_')
        
        try:
            bins = pd.qcut(df[feat], q=n_bins, duplicates='drop')
        except ValueError:
            bins = pd.cut(df[feat], bins=n_bins, duplicates='drop')
        
        bin_rates = df.groupby(bins, observed=True)[target_col].mean()
        rate_values = bin_rates.values
        gradient_values = np.gradient(rate_values)
        
        bin_to_gradient = dict(zip(bin_rates.index, gradient_values))
        bin_to_rate = dict(zip(bin_rates.index, rate_values))
        
        df[f'{safe_name}_gradient'] = bins.map(bin_to_gradient).astype(float)
        df[f'{safe_name}_local_rate'] = bins.map(bin_to_rate).astype(float)
        df[f'{safe_name}_abs_gradient'] = df[f'{safe_name}_gradient'].abs()
        
        for col in [f'{safe_name}_gradient', f'{safe_name}_local_rate', f'{safe_name}_abs_gradient']:
            if df[col].isna().any():
                df[col] = df[col].fillna(0.0)
        
        print(f"   ✅ {safe_name}_gradient: ∂P(subscribe)/∂{feat} ({n_bins} bins)")
        print(f"      Gradient range: [{df[f'{safe_name}_gradient'].min():.4f}, "
              f"{df[f'{safe_name}_gradient'].max():.4f}]")
        print(f"   ✅ {safe_name}_abs_gradient: |∂P/∂x| (slope intensity)")
        print(f"   ✅ {safe_name}_local_rate: empirical P(subscribe) at this value")
    
    # =================================================================
    # PART D: CURVATURE (SECOND DERIVATIVE) FEATURES
    # =================================================================
    """
    Second derivative captures INFLECTION POINTS — where the slope itself
    is changing fastest. |∂²P/∂x²| peaks at inflection = max model uncertainty.
    """
    print("\n📊 Part D: Curvature (Second Derivative) Features")
    print("-"*80)
    
    curvature_features = ['euribor3m', 'nr.employed', 'emp.var.rate']
    
    for feat in curvature_features:
        safe_name = feat.replace('.', '_')
        
        try:
            bins = pd.qcut(df[feat], q=n_bins, duplicates='drop')
        except ValueError:
            bins = pd.cut(df[feat], bins=n_bins, duplicates='drop')
        
        bin_rates = df.groupby(bins, observed=True)[target_col].mean()
        rate_values = bin_rates.values
        
        if len(rate_values) >= 3:
            curvature_values = np.gradient(np.gradient(rate_values))
        else:
            curvature_values = np.zeros_like(rate_values)
        
        bin_to_curvature = dict(zip(bin_rates.index, curvature_values))
        
        df[f'{safe_name}_curvature'] = bins.map(bin_to_curvature).astype(float)
        df[f'{safe_name}_abs_curvature'] = df[f'{safe_name}_curvature'].abs()
        
        for col in [f'{safe_name}_curvature', f'{safe_name}_abs_curvature']:
            if df[col].isna().any():
                df[col] = df[col].fillna(0.0)
        
        print(f"   ✅ {safe_name}_curvature: ∂²P/∂x² (inflection detector)")
        print(f"      Range: [{df[f'{safe_name}_curvature'].min():.5f}, "
              f"{df[f'{safe_name}_curvature'].max():.5f}]")
        print(f"   ✅ {safe_name}_abs_curvature: |∂²P/∂x²| (inflection intensity)")
    
    # =================================================================
    # PART E: COMPOSITE DERIVATIVE FEATURES
    # =================================================================
    print("\n📊 Part E: Composite Derivative Features")
    print("-"*80)
    
    # 1. Combined slope intensity across economic features
    slope_cols = [c for c in df.columns if c.endswith('_abs_gradient') and 
                  any(f in c for f in ['euribor', 'nr_employed', 'emp_var'])]
    if slope_cols:
        df['economic_slope_intensity'] = df[slope_cols].mean(axis=1)
        print(f"   ✅ economic_slope_intensity: mean(|∂P/∂x|) across economic features")
    
    # 2. Combined curvature
    curv_cols = [c for c in df.columns if c.endswith('_abs_curvature')]
    if curv_cols:
        df['economic_curvature_intensity'] = df[curv_cols].mean(axis=1)
        print(f"   ✅ economic_curvature_intensity: mean(|∂²P/∂x²|)")
    
    # 3. Joint decay product across all three economic features
    df['joint_economic_decay'] = (
        df['euribor_decay'] * df['emp_var_decay'] * df['nr_employed_decay']
    )
    print(f"   ✅ joint_economic_decay: Π(decay_i) across economic features")
    print(f"      Range: [{df['joint_economic_decay'].min():.4f}, "
          f"{df['joint_economic_decay'].max():.4f}]")
    
    # 4. Slope × stress interaction
    if 'economic_stress_integral' in df.columns:
        df['slope_x_stress'] = (
            df['economic_slope_intensity'] * df['economic_stress_integral']
        )
        print(f"   ✅ slope_x_stress: slope_intensity × stress_integral")
    
    # 5. Decay × density interaction
    if 'neighborhood_subscription_density' in df.columns:
        df['decay_x_density'] = (
            df['joint_economic_decay'] * df['neighborhood_subscription_density']
        )
        print(f"   ✅ decay_x_density: joint_decay × neighborhood_density")
    
    # =================================================================
    # SUMMARY & CLEANUP
    # =================================================================
    new_cols = set(df.columns) - original_cols
    
    # Verify no inf/nan
    inf_cols_list = [c for c in new_cols if np.isinf(df[c]).any()]
    nan_cols_list = [c for c in new_cols if df[c].isna().any()]
    if inf_cols_list:
        print(f"\n   ⚠️ Inf values in: {inf_cols_list}")
        df[list(inf_cols_list)] = df[list(inf_cols_list)].replace([np.inf, -np.inf], np.nan)
    if nan_cols_list or inf_cols_list:
        for c in new_cols:
            if df[c].isna().any():
                med = df[c].median()
                df[c] = df[c].fillna(med)
                print(f"   Imputed {c} with median: {med:.4f}")
    else:
        print(f"\n   ✅ No inf/NaN values detected")
    
    sigmoid_feats = [c for c in new_cols if 'sigmoid' in c]
    decay_feats = [c for c in new_cols if 'decay' in c]
    gradient_feats = [c for c in new_cols if 'gradient' in c or 'local_rate' in c]
    curvature_feats = [c for c in new_cols if 'curvature' in c]
    composite_feats = [c for c in new_cols if c not in sigmoid_feats + decay_feats + gradient_feats + curvature_feats]
    
    print(f"\n{'='*80}")
    print(f"✅ DERIVATIVE FEATURES COMPLETE: {len(new_cols)} new features")
    print(f"   A) Sigmoid slope:       {len(sigmoid_feats)}")
    print(f"   B) Exponential decay:   {len(decay_feats)}")
    print(f"   C) Piecewise gradient:  {len(gradient_feats)}")
    print(f"   D) Curvature (∂²P/∂x²): {len(curvature_feats)}")
    print(f"   E) Composites:          {len(composite_feats)}")
    print(f"{'='*80}")
    
    return df




In [60]:
# ============================================================
# CELL 10D: DAY_OF_WEEK DISTRIBUTION ANALYSIS & FEATURES
# ============================================================
"""
DAY OF WEEK ANALYSIS
====================
day_of_week ranked #15 (Composite: 0.0163, AUC: 0.509) — nearly useless alone.
But interaction analysis showed:
  month × day_of_week:          Lift = 10.6%
  cons.conf.idx × day_of_week:  Lift = 10.7%

Strategy: Create interaction features, not main-effect features.
"""


def analyze_day_of_week(df: pd.DataFrame, target_col: str = 'y') -> pd.DataFrame:
    """
    Analyze day_of_week distribution and create interaction features.
    """
    print("\n" + "="*80)
    print("📅 DAY OF WEEK ANALYSIS & FEATURES")
    print("="*80)
    
    if 'day_of_week' not in df.columns:
        print("   ⚠️ day_of_week not in dataset, skipping")
        return df
    
    # --- Distribution Analysis ---
    print("\n📊 Distribution Analysis:")
    print("-"*80)
    
    dow_stats = df.groupby('day_of_week').agg(
        count=(target_col, 'count'),
        subscribe_rate=(target_col, 'mean'),
        subscribe_count=(target_col, 'sum')
    ).sort_values('subscribe_rate', ascending=False)
    
    overall_rate = df[target_col].mean()
    
    print(f"\n   Overall subscribe rate: {overall_rate:.4f}")
    print(f"\n   {'Day':>5} | {'Count':>7} | {'Sub Rate':>9} | {'Lift vs Avg':>11} | {'Subscribers':>11}")
    print(f"   {'-'*5}-+-{'-'*7}-+-{'-'*9}-+-{'-'*11}-+-{'-'*11}")
    
    for day, row in dow_stats.iterrows():
        lift = row['subscribe_rate'] / overall_rate
        print(f"   {day:>5} | {row['count']:>7.0f} | {row['subscribe_rate']:>9.4f} | "
              f"{lift:>10.3f}x | {row['subscribe_count']:>11.0f}")
    
    rate_range = dow_stats['subscribe_rate'].max() - dow_stats['subscribe_rate'].min()
    print(f"\n   Rate range: {rate_range:.4f} (max - min)")
    print(f"   Verdict: {'Weak main effect' if rate_range < 0.03 else 'Moderate main effect'}")
    
    # --- Feature Engineering ---
    print(f"\n📊 Day of Week Feature Engineering:")
    print("-"*80)
    
    # 1. Target-encoded day (smoothed to avoid leakage)
    global_mean = df[target_col].mean()
    smoothing_factor = 100
    
    dow_counts = df.groupby('day_of_week')[target_col].count()
    dow_means = df.groupby('day_of_week')[target_col].mean()
    dow_smoothed = (dow_counts * dow_means + smoothing_factor * global_mean) / (dow_counts + smoothing_factor)
    
    df['dow_target_encoded'] = df['day_of_week'].map(dow_smoothed)
    print(f"   ✅ dow_target_encoded: smoothed target encoding (factor={smoothing_factor})")
    
    # 2. Day × Month interaction encoding
    if 'month' in df.columns:
        dm_key = df['day_of_week'].astype(str) + '_' + df['month'].astype(str)
        dm_stats = df.groupby(dm_key)[target_col].agg(['mean', 'count'])
        dm_smoothed = (dm_stats['count'] * dm_stats['mean'] + smoothing_factor * global_mean) / (dm_stats['count'] + smoothing_factor)
        df['dow_month_encoded'] = dm_key.map(dm_smoothed)
        df['dow_month_encoded'] = df['dow_month_encoded'].fillna(global_mean)
        print(f"   ✅ dow_month_encoded: day×month interaction encoding")
    
    # 3. Day × economic stress interaction
    if 'economic_stress_integral' in df.columns:
        df['dow_x_stress'] = df['dow_target_encoded'] * df['economic_stress_integral']
        print(f"   ✅ dow_x_stress: day_encoded × economic_stress")
    
    # 4. Binary: above-average contact day
    good_days = dow_stats[dow_stats['subscribe_rate'] > overall_rate].index.tolist()
    df['good_contact_day'] = df['day_of_week'].isin(good_days).astype(int)
    print(f"   ✅ good_contact_day: 1 if day has above-average conversion")
    print(f"      Good days: {good_days}")
    
    # 5. Day × contact method
    if 'contact' in df.columns:
        df['dow_x_contact'] = df['dow_target_encoded'] * (1 - df['contact'])
        print(f"   ✅ dow_x_contact: day_encoded × cellular_contact")
    
    print(f"\n{'='*80}")
    return df




In [61]:
# ============================================================
# CELL 10E: REVISITING 'previous' — TEMPORAL INTERACTIONS
# ============================================================
"""
PREVIOUS FEATURE REHAB
======================
`previous` ranked #4 (Composite: 0.2966) but is a mixed bag:
- previous=0 dominates (>30k obs, ~10% subscribe rate)  
- previous≥5 has ~60-70% subscribe rate BUT tiny sample sizes
- Raw feature is extremely right-skewed

Strategy: Binary/bucketed versions + interaction features that leverage
the strong signal in "previously contacted" without tail noise.
"""


def create_previous_interactions(df: pd.DataFrame, target_col: str = 'y') -> pd.DataFrame:
    """
    Create interaction features using `previous` that capture temporal patterns.
    """
    print("\n" + "="*80)
    print("🔄 PREVIOUS FEATURE: TEMPORAL INTERACTIONS")
    print("="*80)
    
    if 'previous' not in df.columns:
        print("   ⚠️ previous not in dataset, skipping")
        return df
    
    # 1. Bucketed previous (reduces noise in tail)
    df['previous_bucket'] = np.where(
        df['previous'] == 0, 0,
        np.where(df['previous'] == 1, 1, 2)
    )
    print(f"   ✅ previous_bucket: [0=none, 1=once, 2=repeat]")
    
    # 2. Binary prior relationship flag
    df['has_prior_contact'] = (df['previous'] > 0).astype(int)
    print(f"   ✅ has_prior_contact: binary prior relationship flag")
    
    # 3. Previous × high conversion month
    if 'high_conversion_month' in df.columns:
        df['prior_x_good_month'] = df['has_prior_contact'] * df['high_conversion_month']
        print(f"   ✅ prior_x_good_month: prior_contact × high_conversion_month")
    
    # 4. Previous × economic stress
    if 'economic_stress_integral' in df.columns:
        df['prior_x_stress'] = df['has_prior_contact'] * df['economic_stress_integral']
        print(f"   ✅ prior_x_stress: prior_contact × economic_stress")
    
    # 5. Previous × contact method
    if 'contact' in df.columns:
        df['prior_x_cellular'] = df['has_prior_contact'] * (1 - df['contact'])
        print(f"   ✅ prior_x_cellular: prior_contact × cellular_contact")
    
    # 6. Campaign intensity split by history
    if 'campaign' in df.columns:
        df['cold_intensity'] = df['campaign'] * (1 - df['has_prior_contact'])
        df['warm_intensity'] = df['campaign'] * df['has_prior_contact']
        print(f"   ✅ cold_intensity: campaign × (no prior contact)")
        print(f"   ✅ warm_intensity: campaign × (has prior contact)")
    
    # 7. Previous × decay features
    if 'euribor_decay' in df.columns:
        df['prior_x_euribor_decay'] = df['has_prior_contact'] * df['euribor_decay']
        print(f"   ✅ prior_x_euribor_decay: prior_contact × euribor_decay")
    
    print(f"\n{'='*80}")
    return df


# ============================================================
# APPLY ALL NEW FEATURE ENGINEERING
# ============================================================
print("\n🔧 Applying derivative + day_of_week + previous features...")
df_engineered = create_derivative_features(df_engineered)
df_engineered = analyze_day_of_week(df_engineered)
df_engineered = create_previous_interactions(df_engineered)

print(f"\n📊 Total features now: {len(df_engineered.columns)}")





🔧 Applying derivative + day_of_week + previous features...

📐 DERIVATIVE / SLOPE / DECAY FEATURES

Philosophy: Integrals capture accumulation. Derivatives capture
the RATE OF CHANGE — where distributions shift fastest between classes.
These target the cliff edges and inflection points visible in our plots.

📊 Part A: Sigmoid Slope Features
--------------------------------------------------------------------------------
   ✅ nr_employed_sigmoid: σ(-(x-5100)/30)
      Range: [0.014, 0.990], Mean: 0.227
   ✅ nr_employed_sigmoid_slope: peaks at x=5100 (the cliff)
      Range: [0.042, 1.000], Mean: 0.322
   ✅ euribor3m_sigmoid: σ(-(x-1.5)/0.5)
      Range: [0.001, 0.850], Mean: 0.214
   ✅ euribor3m_sigmoid_slope: peaks at x=1.5 (the cliff)
      Range: [0.003, 1.000], Mean: 0.288
   ✅ emp_var_rate_sigmoid: σ(-(x--1.0)/0.5)
      Range: [0.008, 0.992], Mean: 0.297
   ✅ emp_var_rate_sigmoid_slope: peaks at x=-1.0 (the cliff)
      Range: [0.032, 0.990], Mean: 0.224

📊 Part B: Exponential Dec

In [62]:
# ============================================================
# CELL 10F: OVERLAP ZONE INTERACTION FEATURES
# ============================================================
"""
OVERLAP ZONE DISAMBIGUATION
============================

PROBLEM:
--------
From the violin plots, our best engineered features (economic_stress_integral,
nr_employed_sigmoid, euribor_decay, euribor3m_sigmoid, joint_economic_decay)
all show strong separation in the HIGH stress / LOW rate regime, but collapse
into overlapping distributions near their low ends:

    economic_stress_integral:  overlap at 0.0-0.2
    nr_employed_sigmoid:       overlap at 0.0-0.1
    euribor_decay:             overlap at 0.0-0.2
    euribor3m_sigmoid:         overlap at 0.0-0.1
    joint_economic_decay:      overlap at 0.0-0.05

These overlap zones correspond to the "healthy economy" regime — high euribor,
high employment, positive emp.var.rate. In this regime, ALL economic features
lose discriminative power simultaneously because the macro signal simply isn't
there.

INSIGHT:
--------
The overlap zones are STAGGERED across feature types but ALIGNED within
economic features. This means:
- Economic features fail together (correlated collapse)
- But BEHAVIORAL features (contact, month, previous, default) still have
  signal in exactly these zones

MI INTERACTION ANALYSIS CONFIRMS THIS:
- month × cons.price.idx:      35.1% lift (timing × price level)
- cons.price.idx × contact:    50.2% lift (price level × contact method)
- cons.price.idx × cons.conf.idx: 28.6% lift (price × confidence)
- month × cons.conf.idx:       23.1% lift (timing × sentiment)
- age × education:             22.7% lift (demographics cross)

STRATEGY:
---------
Create features that ONLY fire in the overlap zones, using behavioral
and timing signals to disambiguate where economic features can't.

Mathematical framing: These are CONDITIONAL interactions —
    f(x) = g(behavioral) · 𝟙[economic_feature ∈ overlap_zone]

This is equivalent to a piecewise function that activates a different
feature subspace depending on the economic regime.
"""

def create_overlap_zone_interactions(df: pd.DataFrame, target_col: str = 'y') -> pd.DataFrame:
    """
    Create interaction features targeting overlap zones where economic
    features lose discriminative power.
    
    Parameters
    ----------
    df : pd.DataFrame
        Engineered dataframe with integral + derivative features
    target_col : str
        Name of binary target column
        
    Returns
    -------
    pd.DataFrame
        Dataframe with additional overlap zone features
    """
    print("\n" + "="*80)
    print("🎯 OVERLAP ZONE INTERACTION FEATURES")
    print("="*80)
    print("\nStrategy: When economic features collapse (healthy economy),")
    print("use behavioral/timing signals to disambiguate subscribers.")
    print("="*80)
    
    original_cols = set(df.columns)
    
    # =================================================================
    # DEFINE OVERLAP ZONE MASKS
    # =================================================================
    """
    These masks identify observations where economic features have poor
    separation. We use the thresholds visible in the violin plots.
    
    The "low stress zone" is where economic_stress_integral < 0.2,
    which corresponds roughly to:
    - euribor3m > ~2.5  (high rates, stable economy)
    - emp.var.rate > ~0  (employment growing)
    - nr.employed > ~5150 (high employment)
    
    ~60-65% of the dataset falls in this zone.
    """
    print("\n📊 Part A: Overlap Zone Definitions")
    print("-"*80)
    
    # Primary overlap zone: low economic stress (integral-based)
    if 'economic_stress_integral' in df.columns:
        low_stress = (df['economic_stress_integral'] < 0.2).astype(int)
        moderate_stress = ((df['economic_stress_integral'] >= 0.2) & 
                          (df['economic_stress_integral'] < 0.5)).astype(int)
    else:
        # Fallback to raw features
        low_stress = ((df['euribor3m'] > 2.5) & (df['emp.var.rate'] > 0)).astype(int)
        moderate_stress = ((df['euribor3m'] >= 1.0) & (df['euribor3m'] <= 2.5)).astype(int)
    
    df['in_overlap_zone'] = low_stress
    df['in_moderate_zone'] = moderate_stress
    
    n_low = low_stress.sum()
    n_mod = moderate_stress.sum()
    n_total = len(df)
    
    if target_col in df.columns:
        rate_low = df.loc[low_stress == 1, target_col].mean()
        rate_mod = df.loc[moderate_stress == 1, target_col].mean()
        rate_high = df.loc[(low_stress == 0) & (moderate_stress == 0), target_col].mean()
        print(f"   Low stress zone:      {n_low:>6} samples ({n_low/n_total:.1%}), sub rate: {rate_low:.4f}")
        print(f"   Moderate stress zone:  {n_mod:>6} samples ({n_mod/n_total:.1%}), sub rate: {rate_mod:.4f}")
        print(f"   High stress zone:     {n_total-n_low-n_mod:>6} samples ({(n_total-n_low-n_mod)/n_total:.1%}), sub rate: {rate_high:.4f}")
    else:
        print(f"   Low stress zone:      {n_low:>6} samples ({n_low/n_total:.1%})")
        print(f"   Moderate stress zone:  {n_mod:>6} samples ({n_mod/n_total:.1%})")
    
    # =================================================================
    # PART B: BEHAVIORAL SIGNALS IN OVERLAP ZONES
    # =================================================================
    """
    In the overlap zone, economic features are ~useless.
    These features activate behavioral/timing signals ONLY in that zone.
    
    Mathematically: f(x) = behavioral_signal × 𝟙[in_overlap_zone]
    
    This is a piecewise activation: the model gets a DIFFERENT feature 
    subspace depending on economic regime.
    """
    print("\n📊 Part B: Behavioral Overlap Zone Interactions")
    print("-"*80)
    
    # B1: Contact method in overlap zone
    # Cellular converts at ~14.5% vs telephone at ~5% — this 3x lift
    # is independent of economic conditions and valuable in overlap zone
    if 'contact' in df.columns:
        df['overlap_cellular'] = low_stress * (1 - df['contact'])
        print(f"   ✅ overlap_cellular: cellular contact in low-stress zone")
        print(f"      Logic: When economics can't help, contact method matters most")
    
    # B2: Good month in overlap zone
    # March (51%), Dec (49%), Sept (45%), Oct (44%) vs May (6.5%)
    # These conversion rate differences are enormous and orthogonal to economics
    if 'month' in df.columns:
        good_months = df['month'].isin([3, 9, 10, 12]).astype(int)
        df['overlap_good_month'] = low_stress * good_months
        print(f"   ✅ overlap_good_month: high-conversion month in low-stress zone")
        
        # Also: avoid months signal in overlap zone
        avoid_months = df['month'].isin([5, 6, 7]).astype(int)
        df['overlap_avoid_month'] = low_stress * avoid_months
        print(f"   ✅ overlap_avoid_month: low-conversion month in low-stress zone")
    
    # B3: Prior contact in overlap zone
    # has_prior_contact captures the 10% → 21% jump at previous=1
    if 'has_prior_contact' in df.columns:
        df['overlap_prior_contact'] = low_stress * df['has_prior_contact']
        print(f"   ✅ overlap_prior_contact: prior relationship in low-stress zone")
    
    # B4: Default unknown in overlap zone (strong negative signal)
    # default=-1 (unknown) has ~5% subscribe rate — much lower than baseline
    if 'default' in df.columns:
        default_unknown = (df['default'] == -1).astype(int)
        default_clean = (df['default'] == 0).astype(int)
        df['overlap_default_unknown'] = low_stress * default_unknown
        df['overlap_default_clean'] = low_stress * default_clean
        print(f"   ✅ overlap_default_unknown: unknown default in low-stress (double negative)")
        print(f"   ✅ overlap_default_clean: clean default in low-stress (mitigating signal)")
    
    # =================================================================
    # PART C: TOP MI INTERACTIONS (from interaction analysis)
    # =================================================================
    """
    The MI analysis identified specific feature pairs with high synergistic
    lift. We implement the top interactions, particularly those involving
    features that retain signal in the overlap zone.
    
    Top interactions by lift %:
    1. cons.price.idx × contact:    50.2% lift
    2. month × cons.price.idx:      35.1% lift  
    3. cons.price.idx × cons.conf.idx: 28.6% lift
    4. month × cons.conf.idx:       23.1% lift
    5. age × education:             22.7% lift
    
    Note: cons.price.idx appears in 4 of the top 5 interactions.
    This makes sense from the distributions — CPI has moderate separation
    (d=-0.435) but its interaction with other features is where the real
    value lies. CPI acts as a CONTEXT variable that modulates other signals.
    """
    print("\n📊 Part C: High-MI Interactions")
    print("-"*80)
    
    # C1: cons.price.idx × contact (50.2% MI lift — HIGHEST)
    # CPI modulates the contact method effect
    # Low CPI + cellular = favorable macro + good channel = best combo
    if 'cons.price.idx' in df.columns and 'contact' in df.columns:
        # Threshold from distribution: CPI < 93.5 is favorable
        cpi_low = (df['cons.price.idx'] < 93.5).astype(int)
        cpi_high = (df['cons.price.idx'] >= 93.5).astype(int)
        cellular = (1 - df['contact'])  # contact=0 is cellular
        
        df['cpi_low_cellular'] = cpi_low * cellular
        df['cpi_high_cellular'] = cpi_high * cellular
        print(f"   ✅ cpi_low_cellular: low CPI × cellular (best combo)")
        print(f"   ✅ cpi_high_cellular: high CPI × cellular")
    
    # C2: month × cons.price.idx (35.1% MI lift)
    # Good months under favorable CPI = campaign timing sweet spot
    if 'month' in df.columns and 'cons.price.idx' in df.columns:
        # Continuous interaction: month_rate × CPI_deviation
        # Use smoothed month encoding for continuous version
        if 'dow_target_encoded' in df.columns:
            # We already have month target rates from earlier
            pass
        
        # Binary version: good month + low CPI
        df['good_month_low_cpi'] = good_months * cpi_low
        print(f"   ✅ good_month_low_cpi: high-conversion month × favorable CPI")
    
    # C3: cons.price.idx × cons.conf.idx (28.6% MI lift)
    # When both price and confidence are in "favorable" territory
    if 'cons.price.idx' in df.columns and 'cons.conf.idx' in df.columns:
        # cons.conf.idx < -40 is roughly where the ECDF curves diverge slightly
        conf_low = (df['cons.conf.idx'] < -40).astype(int)
        
        # CPI and confidence align (both in favorable territory)
        df['cpi_conf_aligned'] = cpi_low * conf_low
        
        # Continuous ratio: captures the joint macro sentiment
        # Normalize both to [0,1] range first
        cpi_norm = (df['cons.price.idx'] - 92.2) / (94.8 - 92.2)  # approx range
        conf_norm = (df['cons.conf.idx'] - (-50.8)) / ((-26.9) - (-50.8))  # approx range
        cpi_norm = cpi_norm.clip(0, 1)
        conf_norm = conf_norm.clip(0, 1)
        
        df['macro_sentiment'] = (1 - cpi_norm) * (1 - conf_norm)
        # High when BOTH CPI is low AND confidence is low (crisis → subscribe)
        
        print(f"   ✅ cpi_conf_aligned: both CPI and confidence in favorable territory")
        print(f"   ✅ macro_sentiment: (1-CPI_norm) × (1-conf_norm), crisis sentiment score")
    
    # C4: month × cons.conf.idx (23.1% MI lift)
    if 'month' in df.columns and 'cons.conf.idx' in df.columns:
        df['good_month_low_conf'] = good_months * conf_low
        print(f"   ✅ good_month_low_conf: high-conversion month × low consumer confidence")
    
    # C5: age × education (22.7% MI lift)
    # Demographic cross — certain age-education combos convert differently
    if 'age' in df.columns and 'education' in df.columns:
        # Age buckets from the distribution: <30, 30-50 (bulk), 50+ (subscriber tail)
        age_young = (df['age'] < 30).astype(int)
        age_senior = (df['age'] >= 50).astype(int)
        
        # Education: 0=illiterate(22%), -1=unknown(14%), 6=university(14%)
        edu_low = (df['education'] <= 1).astype(int)
        edu_high = (df['education'] >= 5).astype(int)
        
        df['senior_high_edu'] = age_senior * edu_high
        df['senior_low_edu'] = age_senior * edu_low
        df['young_high_edu'] = age_young * edu_high
        
        print(f"   ✅ senior_high_edu: age≥50 × high education")
        print(f"   ✅ senior_low_edu: age≥50 × low education")
        print(f"   ✅ young_high_edu: age<30 × high education")
    
    # =================================================================
    # PART D: DEFAULT AS NEGATIVE FILTER
    # =================================================================
    """
    default=-1 (unknown) is a surprisingly strong negative signal (~5% sub rate).
    This is likely a data quality proxy — incomplete records correlate with
    less engaged or less qualified prospects.
    
    We create interactions that amplify or mitigate this signal.
    """
    print("\n📊 Part D: Default-Unknown Interactions")
    print("-"*80)
    
    if 'default' in df.columns:
        default_unknown = (df['default'] == -1).astype(int)
        
        # D1: Unknown default × contact method
        # Does cellular contact mitigate the unknown-default penalty?
        if 'contact' in df.columns:
            df['default_unk_cellular'] = default_unknown * (1 - df['contact'])
            df['default_unk_telephone'] = default_unknown * df['contact']
            print(f"   ✅ default_unk_cellular: unknown default + cellular (mitigating?)")
            print(f"   ✅ default_unk_telephone: unknown default + telephone (compounding?)")
        
        # D2: Unknown default × economic stress
        # Does crisis override the unknown-default penalty?
        if 'economic_stress_integral' in df.columns:
            df['default_unk_crisis'] = default_unknown * df['economic_stress_integral']
            print(f"   ✅ default_unk_crisis: unknown default × economic stress")
        
        # D3: Unknown default × prior contact
        # If we've contacted them before, does that override the unknown default?
        if 'has_prior_contact' in df.columns:
            df['default_unk_prior'] = default_unknown * df['has_prior_contact']
            print(f"   ✅ default_unk_prior: unknown default + prior contact (mitigating?)")
    
    # =================================================================
    # PART E: MODERATE ZONE TIE-BREAKERS
    # =================================================================
    """
    The moderate stress zone (integral 0.2-0.5) has SOME economic signal
    but not enough for clean separation. Here, cons.conf.idx might break
    ties — it's useless alone but the MI analysis shows it has interaction
    value specifically with CPI (28.6% lift) and month (23.1% lift).
    """
    print("\n📊 Part E: Moderate Zone Tie-Breakers")
    print("-"*80)
    
    if 'cons.conf.idx' in df.columns:
        # E1: Consumer confidence direction in moderate zone
        df['moderate_zone_low_conf'] = moderate_stress * conf_low
        print(f"   ✅ moderate_zone_low_conf: moderate stress × low consumer confidence")
        
        # E2: Macro sentiment in moderate zone
        if 'macro_sentiment' in df.columns:
            df['moderate_zone_sentiment'] = moderate_stress * df['macro_sentiment']
            print(f"   ✅ moderate_zone_sentiment: moderate stress × crisis sentiment score")
    
    # E3: Campaign timing in moderate zone
    if 'month' in df.columns:
        df['moderate_zone_good_month'] = moderate_stress * good_months
        print(f"   ✅ moderate_zone_good_month: moderate stress × high-conversion month")
    
    # =================================================================
    # PART F: COMPOUND BEHAVIORAL SCORES
    # =================================================================
    """
    Combine multiple behavioral signals into compound scores for the
    overlap zone. Instead of individual binary interactions, create
    an aggregate "behavioral favorability" score.
    
    This is analogous to the economic_stress_integral but for 
    behavioral dimensions.
    """
    print("\n📊 Part F: Compound Behavioral Scores")
    print("-"*80)
    
    # Behavioral favorability score (overlap zone version)
    # Each component contributes when it's in the "favorable" direction
    behavioral_components = []
    weights = []
    
    # Contact method (strong signal: ~3x lift for cellular)
    if 'contact' in df.columns:
        behavioral_components.append((1 - df['contact']).values)
        weights.append(3)
    
    # Month (strong signal: 5-8x lift for good months)
    if 'month' in df.columns:
        behavioral_components.append(df['month'].isin([3, 9, 10, 12]).astype(float).values)
        weights.append(3)
    
    # Prior contact (moderate signal: ~2x lift)
    if 'has_prior_contact' in df.columns:
        behavioral_components.append(df['has_prior_contact'].astype(float).values)
        weights.append(2)
    
    # Default clean (moderate signal: 13% vs 5%)
    if 'default' in df.columns:
        behavioral_components.append((df['default'] == 0).astype(float).values)
        weights.append(1)
    
    # Age > 50 (weak signal but additive)
    if 'age' in df.columns:
        behavioral_components.append((df['age'] >= 50).astype(float).values)
        weights.append(1)
    
    if behavioral_components:
        total_weight = sum(weights)
        behavioral_score = sum(
            w * c for w, c in zip(weights, behavioral_components)
        ) / total_weight
        
        df['behavioral_favorability'] = behavioral_score
        
        # Overlap zone version: only fires in low-stress zone
        df['overlap_behavioral_score'] = low_stress * behavioral_score
        
        print(f"   ✅ behavioral_favorability: weighted behavioral score [0,1]")
        print(f"      Components: contact(w=3), month(w=3), prior(w=2), default(w=1), age(w=1)")
        print(f"      Range: [{df['behavioral_favorability'].min():.3f}, "
              f"{df['behavioral_favorability'].max():.3f}]")
        print(f"   ✅ overlap_behavioral_score: behavioral score × 𝟙[low_stress]")
        print(f"      Fires ONLY when economic features can't disambiguate")
    
    # =================================================================
    # PART G: REGIME-CONDITIONAL DENSITY
    # =================================================================
    """
    The neighborhood_subscription_density feature uses economic dimensions
    for distance. In the overlap zone, these distances are compressed 
    (everyone looks similar economically). 
    
    Create a regime-conditional version: density × regime indicator.
    This tells the model "in the overlap zone, how dense are subscribers
    among your economic neighbors?"
    """
    print("\n📊 Part G: Regime-Conditional Features")
    print("-"*80)
    
    if 'neighborhood_subscription_density' in df.columns:
        df['overlap_density'] = low_stress * df['neighborhood_subscription_density']
        df['high_stress_density'] = (1 - low_stress) * (1 - moderate_stress) * df['neighborhood_subscription_density']
        print(f"   ✅ overlap_density: subscription density in low-stress zone")
        print(f"   ✅ high_stress_density: subscription density in high-stress zone")
    
    # Decay features conditioned on regime
    if 'joint_economic_decay' in df.columns:
        df['overlap_decay'] = low_stress * df['joint_economic_decay']
        print(f"   ✅ overlap_decay: economic decay in low-stress zone")
    
    # =================================================================
    # SUMMARY & CLEANUP
    # =================================================================
    new_cols = set(df.columns) - original_cols
    
    # Clean inf/nan
    for c in new_cols:
        if df[c].isna().any() or np.isinf(df[c]).any():
            df[c] = df[c].replace([np.inf, -np.inf], np.nan)
            med = df[c].median()
            df[c] = df[c].fillna(med if not np.isnan(med) else 0.0)
            print(f"   Cleaned {c}")
    
    overlap_feats = [c for c in new_cols if 'overlap' in c or 'moderate_zone' in c]
    mi_feats = [c for c in new_cols if c.startswith(('cpi_', 'good_month_', 'macro_', 'senior_', 'young_'))]
    default_feats = [c for c in new_cols if 'default_unk' in c]
    compound_feats = [c for c in new_cols if c in ('behavioral_favorability', 'overlap_behavioral_score')]
    other_feats = [c for c in new_cols if c not in overlap_feats + mi_feats + default_feats + compound_feats]
    
    print(f"\n{'='*80}")
    print(f"✅ OVERLAP ZONE FEATURES COMPLETE: {len(new_cols)} new features")
    print(f"   Overlap zone conditionals: {len(overlap_feats)}")
    print(f"   High-MI interactions:       {len(mi_feats)}")
    print(f"   Default interactions:       {len(default_feats)}")
    print(f"   Compound scores:            {len(compound_feats)}")
    print(f"   Other:                      {len(other_feats)}")
    print(f"{'='*80}")
    
    return df


# ============================================================
# APPLY OVERLAP ZONE FEATURES
# ============================================================
print("\n🔧 Applying overlap zone interaction features...")
df_engineered = create_overlap_zone_interactions(df_engineered)
print(f"\n📊 Total features now: {len(df_engineered.columns)}")



🔧 Applying overlap zone interaction features...

🎯 OVERLAP ZONE INTERACTION FEATURES

Strategy: When economic features collapse (healthy economy),
use behavioral/timing signals to disambiguate subscribers.

📊 Part A: Overlap Zone Definitions
--------------------------------------------------------------------------------
   Low stress zone:       27690 samples (67.2%), sub rate: 0.0484
   Moderate stress zone:    8534 samples (20.7%), sub rate: 0.1280
   High stress zone:       4964 samples (12.1%), sub rate: 0.4450

📊 Part B: Behavioral Overlap Zone Interactions
--------------------------------------------------------------------------------
   ✅ overlap_cellular: cellular contact in low-stress zone
      Logic: When economics can't help, contact method matters most
   ✅ overlap_good_month: high-conversion month in low-stress zone
   ✅ overlap_avoid_month: low-conversion month in low-stress zone
   ✅ overlap_prior_contact: prior relationship in low-stress zone
   ✅ overlap_default_un

In [63]:
# ============================================================
# CELL 10G: DEFINE STAGE-SPECIFIC FEATURE SETS
# ============================================================
# Each cascade stage has its own feature set, optimized for that
# model's strengths:
#   LR  (Stage 1): 5 features  — linear interactions, pre-engineered
#   RF  (Stage 2): 8 features  — tree-friendly composites, raw counts
#   EBM (Stage 3): 14 features — non-linear shapes, overlap zone signals
#
# Feature selection driven by the EBM Feature Selection Hierarchy:
#   1. Shape Complexity (R²) — lower = more EBM value
#   2. Effect Magnitude (range) — bigger = more impact
#   3. Class Separation (r / Cohen's d) — sanity check
#   4. Global Importance — least reliable for correlated features
#
# History: Started at 57 features, reduced to 14 over 13 rounds
# of iterative simplification with AUC monitored at each step.
# Net AUC change: 0.8015 → ~0.801 (within noise).
# ============================================================

print("\n" + "="*80)
print("🎯 DEFINING STAGE-SPECIFIC FEATURE SETS")
print("="*80)

# ── Stage 1: Logistic Regression (5 features) ────────────────
# LR needs pre-engineered interactions since it can't learn them.
# All features are binary or smoothed-continuous to stay linear.
LR_FEATURES = [
    # cellular (contact==0) × crisis score ≥3 (low euribor + neg emp.var + low nr.employed)
    # Captures: best channel during economic downturns
    # Source: Cell 10F Part B (overlap zone interactions)
    'cellular_crisis',

    # Empirical P(subscribe) within each euribor3m quantile bin (20 bins)
    # Captures: local subscription density along the euribor rate curve
    # Source: Cell 10C (calculus-based integral features)
    'euribor3m_local_rate',

    # Smoothed target-encoded day_of_week × month interaction (factor=100)
    # Captures: campaign timing sweet spots (day+month combos)
    # Source: Cell 10D (day_of_week analysis)
    'dow_month_encoded',

    # high CPI (≥93.5) × cellular contact
    # Captures: contact method effect modulated by price environment
    # Source: Cell 10F Part C1 — top MI interaction (50.2% lift)
    # Note: weak LR coefficient (0.1275) but retained for cascade coverage
   # 'cpi_high_cellular',

    # high-conversion month (Mar/Sep/Oct/Dec) × low consumer confidence (<-40)
    # Captures: timing × sentiment alignment
    # Source: Cell 10F Part C4 — MI interaction (23.1% lift)
    # Note: weak LR coefficient — candidate for future removal
    #'good_month_low_conf',
]

# ── Stage 2: Random Forest (8 features) ──────────────────────
# RF handles non-linearity natively, so features are composites
# and raw values rather than pre-linearized interactions.
RF_FEATURES = [
    # Raw UCI: number of contacts during this campaign for this client
    # Strong tree-split feature — RF naturally finds the "too many calls" threshold
    'campaign',

    # Gaussian KDE estimate of P(subscribe) over euribor3m × nr.employed × emp.var.rate
    # Captures: spatial clustering of subscribers in 3D economic space
    # Source: Cell 10C (calculus-based integral features)
    'neighborhood_subscription_density',

    # Raw UCI: consumer confidence index (monthly, Banco de Portugal)
    # Range: [-50.8, -26.9]. Lower = more pessimistic consumers
    # RF uses this for regime-splitting; pairs with economic composites
    'cons.conf.idx',

    # Mean |∂²P/∂x²| across euribor3m, nr.employed, emp.var.rate
    # Captures: curvature/inflection points where subscription probability changes fastest
    # High values = transition zones between economic regimes
    # Source: Cell 10D (derivative features)
    'economic_curvature_intensity',

    # Product of decay functions: Π(decay_i) for euribor, emp.var, nr.employed
    # Captures: joint economic favorability — high when ALL indicators signal crisis
    # Source: Cell 10D (derivative features)
    'joint_economic_decay',

    # Smoothed target-encoded day_of_week × month interaction (factor=100)
    # Same as LR — RF gets additional splits on timing patterns
    # Source: Cell 10D (day_of_week analysis)
    'dow_month_encoded',

    # Weighted behavioral composite: cellular(3) + good_month(3) + prior(2) + default_clean(1) + age≥50(1)
    # Captures: aggregate prospect quality independent of economics
    # Source: Cell 10F Part F (compound behavioral scores)
    'behavioral_favorability',

    # high CPI (≥93.5) × cellular contact
    # Source: Cell 10F Part C1
    'cpi_high_cellular',
]

# ── Stage 3: Explainable Boosting Machine (14 features) ──────
# EBM learns non-linear shape functions per feature + pairwise
# interactions. Feature set optimized for:
#   - Complex shapes (low R²) where EBM adds unique value
#   - Features not covered by upstream LR/RF stages
#   - Minimal correlated pairs (3 remaining, all justified)
#
# Optuna-tuned: 14 main effects + 1 interaction (campaign × emp_var_rate_sigmoid_slope)
#
# Remaining correlated pairs (all retained with justification):
#   euribor3m_sigmoid_slope ↔ emp_var_rate_sigmoid_slope  r=+0.724  (emp in sole interaction)
#   economic_curvature_intensity ↔ overlap_behavioral_score  r=-0.716  (different signal types)
EBM_FEATURES = [

    # ── Core non-linear: EBM adds unique value (R² < 0.50) ────

    # Product of economic decay functions × neighborhood subscription density
    # R²=0.127 (deeply non-linear), Imp=#1 every round, range=1.51
    # Captures: joint economic crisis signal weighted by local subscriber clustering
    # Source: Cell 10D (derivative features) × Cell 10C (integral features)
    'decay_x_density',

    # Sigmoid derivative peaking at euribor3m=1.5 (the "cliff" in subscription rates)
    # R²=0.245 (complex), Imp=#2-6, range=2.12
    # Captures: rate of change in subscription probability at the euribor transition point
    # Won head-to-head vs economic_stress_integral (Session 3 experiment)
    # Source: Cell 10D — σ'(-(euribor3m - 1.5) / 0.5)
    'euribor3m_sigmoid_slope',

    # Mean |∂²P/∂x²| across euribor3m, nr.employed, emp.var.rate
    # R²=0.073 (deeply non-linear), range=2.69 (highest in set)
    # Captures: economic regime transition zones — where probability curves bend
    # Source: Cell 10D (derivative features)
    # Note: correlated with overlap_behavioral_score (r=-0.716) but orthogonal signal type
    'economic_curvature_intensity',

    # Raw UCI: consumer confidence index (monthly, Banco de Portugal)
    # R²=0.002 (most complex shape in entire set), range=0.78
    # EBM learns a highly non-linear mapping — confidence affects subscription
    # probability in ways no linear model can capture
    'cons.conf.idx',

    # Raw UCI: client age
    # R²=0.080 (complex non-linear), range=0.99
    # EBM captures the U-shaped pattern: young (<30) and senior (>60) subscribe more
    'age',

    # ── Moderate non-linearity (0.50 < R² < 0.90) ─────────────

    # Smoothed target-encoded day_of_week × month interaction (factor=100)
    # R²=0.853 (moderate), Imp=#1-2 every round, range=1.34
    # Untouchable: consistently top-2 importance across all simplification rounds
    # Source: Cell 10D (day_of_week analysis)
    'dow_month_encoded',

    # Raw UCI: has credit in default? Encoded: -1=unknown, 0=no, 1=yes
    # R²=0.531, range=1.73
    # 80% of values are "unknown" (-1) which has ~5% sub rate (strong negative signal)
    # EBM learns the step function across the three categories
    'default',

    # has_prior_contact (binary) × economic_stress_integral (continuous [0,1])
    # R²=0.450, range=1.32
    # Captures: prior relationship value amplified during economic crisis
    # Absorbed cumulative_campaign_pressure perfectly in Round 10 (imp nearly doubled)
    # Source: Cell 10E (previous interactions) × Cell 10C (integral features)
    'prior_x_stress',

    # Raw UCI: number of contacts during this campaign for this client
    # R²=0.812 (moderate), range=5.21 (highest range in set)
    # In the sole Optuna-selected interaction: campaign × emp_var_rate_sigmoid_slope
    # Captures: diminishing returns of repeated contact, modulated by economic conditions
    'campaign',

    # ── Linear but EBM-only handlers (not in LR or RF) ────────
    # These have high R² but no upstream coverage — EBM is the sole model using them.

    # low_stress_zone × (default == 0)
    # R²=1.000 (linear), Imp=#5, range=0.42
    # Captures: clean credit history signal specifically in the overlap zone
    # where economic features can't disambiguate
    # Source: Cell 10F Part B4 (overlap zone interactions)
    'overlap_default_clean',

    # low_stress_zone × behavioral_favorability composite
    # R²=0.956, Imp=#8, range=0.63
    # Captures: aggregate behavioral quality score, activated ONLY when
    # economic features have poor separation (healthy economy regime)
    # Source: Cell 10F Part F (compound behavioral scores)
    # Note: correlated with economic_curvature_intensity (r=-0.716) — different signal type
    'overlap_behavioral_score',

    # ── Linear with partial upstream coverage ──────────────────
    # High importance justifies retention despite upstream overlap.

    # high CPI (≥93.5) × cellular contact
    # R²=1.000 (linear), Imp=#3, range=0.75
    # In both LR and RF, but consistently #3 importance in EBM
    # May capture residual signal that upstream stages don't fully extract
    # Source: Cell 10F Part C1 — top MI interaction (50.2% lift)
    'cpi_high_cellular',

    # Weighted behavioral composite: cellular(3) + good_month(3) + prior(2) + default_clean(1) + age≥50(1)
    # R²=0.941, Imp=#4, range=0.98
    # In RF upstream, but absorbed contact signal in Round 12 (imp nearly doubled to #4)
    # Source: Cell 10F Part F (compound behavioral scores)
    'behavioral_favorability',

    # Sigmoid derivative peaking at emp.var.rate=-1.0 (employment transition cliff)
    # R²=0.966 (linear), Imp=#11, range=0.41
    # Protected: participates in the sole Optuna-selected interaction with campaign
    # Source: Cell 10D — σ'(-(emp.var.rate - (-1.0)) / 0.5)
    # Note: correlated with euribor3m_sigmoid_slope (r=+0.724) — protected by interaction
    'emp_var_rate_sigmoid_slope',
]

print(f"\n📊 Feature Set Sizes:")
print(f"   LR (Stage 1):   {len(LR_FEATURES):2d} features")
print(f"   RF (Stage 2):   {len(RF_FEATURES):2d} features")
print(f"   EBM (Stage 3):  {len(EBM_FEATURES):2d} features")
print(f"\n✅ Stage-specific features defined")
print("="*80)


🎯 DEFINING STAGE-SPECIFIC FEATURE SETS

📊 Feature Set Sizes:
   LR (Stage 1):    3 features
   RF (Stage 2):    8 features
   EBM (Stage 3):  14 features

✅ Stage-specific features defined


In [64]:
# ============================================================
# CELL 11: Does df_engineered actually have NaN/inf values?
# ============================================================
"""
Quick diagnostic to check if NaN/inf values exist after feature engineering.
The user is correct that there shouldn't be any since engineer_bank_features()
includes cleanup code at the end.
"""

print("="*80)
print("🔍 NaN/INF VERIFICATION FOR df_engineered")
print("="*80)

if 'df_engineered' not in globals():
    print("\n❌ df_engineered not found in globals()")
    print("   Please run feature engineering cells first")
else:
    print(f"\n✅ Found df_engineered with shape: {df_engineered.shape}")
    
    # Check for NaN
    nan_check = df_engineered.isnull().sum()
    nan_cols = nan_check[nan_check > 0]
    
    # Check for inf
    inf_check = np.isinf(df_engineered.select_dtypes(include=[np.number])).sum()
    inf_cols = inf_check[inf_check > 0]
    
    print("\n" + "="*80)
    print("RESULTS:")
    print("="*80)
    
    if len(nan_cols) == 0 and len(inf_cols) == 0:
        print("\n✅ CLEAN DATA - No NaN or inf values detected!")
        print("   The user is correct - data is clean after feature engineering")
    else:
        print("\n⚠️ FOUND PROBLEMATIC VALUES:")
        
        if len(nan_cols) > 0:
            print(f"\n   NaN values in {len(nan_cols)} columns:")
            for col, count in nan_cols.items():
                pct = (count / len(df_engineered)) * 100
                print(f"      {col:40s} {count:6d} ({pct:5.2f}%)")
        
        if len(inf_cols) > 0:
            print(f"\n   Inf values in {len(inf_cols)} columns:")
            for col, count in inf_cols.items():
                pct = (count / len(df_engineered)) * 100
                print(f"      {col:40s} {count:6d} ({pct:5.2f}%)")
    
    # Also check the specific feature sets
    print("\n" + "="*80)
    print("FEATURE SET VERIFICATION:")
    print("="*80)
    
    for feature_set_name in ['LR_FEATURES', 'RF_FEATURES', 'EBM_FEATURES']:
        if feature_set_name in globals():
            features = globals()[feature_set_name]
            subset = df_engineered[features]
            
            nan_count = subset.isnull().sum().sum()
            inf_count = np.isinf(subset.select_dtypes(include=[np.number])).sum().sum()
            
            status = "✅ CLEAN" if (nan_count == 0 and inf_count == 0) else "⚠️ HAS ISSUES"
            
            print(f"\n{feature_set_name:15s} ({len(features):2d} features): {status}")
            if nan_count > 0:
                print(f"   NaN values: {nan_count}")
            if inf_count > 0:
                print(f"   Inf values: {inf_count}")


🔍 NaN/INF VERIFICATION FOR df_engineered

✅ Found df_engineered with shape: (41188, 131)

RESULTS:

✅ CLEAN DATA - No NaN or inf values detected!
   The user is correct - data is clean after feature engineering

FEATURE SET VERIFICATION:

LR_FEATURES     ( 3 features): ✅ CLEAN

RF_FEATURES     ( 8 features): ✅ CLEAN

EBM_FEATURES    (14 features): ✅ CLEAN


In [65]:
# ============================================================
# CELL 12A: LOGISTIC REGRESSION EXPERIMENT 
# ============================================================

print("\n" + "="*80)
print("📊 EXPERIMENT 1: LOGISTIC REGRESSION")
print("="*80)

# --- Data Prep ---
X = df_engineered[LR_FEATURES].copy()
y = df_engineered[TARGET_COL]

print(f"\n📊 Dataset: {X.shape[0]:,} samples, {X.shape[1]} features (LR-optimized)")
print(f"   Features: {LR_FEATURES}")
print(f"   Class distribution: {y.value_counts().to_dict()}")

# Safety check: inf/NaN
print("\n🔍 Pre-scaling data quality check...")
inf_mask = np.isinf(X).any()
nan_mask = X.isnull().any()

if inf_mask.any():
    inf_cols = X.columns[inf_mask].tolist()
    print(f"   ⚠️  Infinity in: {inf_cols} → replacing with NaN")
    X = X.replace([np.inf, -np.inf], np.nan)

if X.isnull().any().any():
    nan_cols = X.columns[X.isnull().any()].tolist()
    print(f"   ⚠️  NaN in: {nan_cols} → imputing with median")
    for col in nan_cols:
        median_val = X[col].median()
        X[col] = X[col].fillna(median_val)
        print(f"      {col}: filled with {median_val:.4f}")
    print(f"   ✅ Data cleaned")
else:
    print(f"   ✅ No inf/NaN values detected")

assert not np.isinf(X).any().any(), "Infinity values still present!"
assert not X.isnull().any().any(), "NaN values still present!"

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- GridSearchCV ---
# NOTE: class_weight pinned to 'balanced' to prevent recall collapse
#       when comparing feature sets. Remove pin only for final tuning.
print("\n🔍 Hyperparameter Tuning (GridSearchCV)...")
param_grid = {
    'C': [0.001, 0.01, 0.1, 1.0, 10.0],
    'penalty': ['l2'],
    'solver': ['lbfgs'],
    'max_iter': [1000],
    'class_weight': ['balanced'],  # PINNED — do not change during feature reduction
}

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

lr = LogisticRegression(random_state=42)
grid_search = GridSearchCV(
    lr, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1, verbose=0
)
grid_search.fit(X_scaled, y)

print(f"\n✅ Best parameters: {grid_search.best_params_}")
print(f"✅ Best CV AUC: {grid_search.best_score_:.4f}")

best_lr = grid_search.best_estimator_

# --- 10-Fold Evaluation ---
print("\n📊 10-Fold Cross-Validation...")

auc_scores = []
recall_scores = []
precision_scores = []
f1_scores = []
recall_at_10fpr_scores = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_scaled, y), 1):
    X_train, X_val = X_scaled[train_idx], X_scaled[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    best_lr.fit(X_train, y_train)
    y_pred_proba = best_lr.predict_proba(X_val)[:, 1]
    y_pred = best_lr.predict(X_val)
    
    auc = roc_auc_score(y_val, y_pred_proba)
    recall = recall_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    
    # Recall @ 10% FPR per fold
    fpr, tpr, _ = roc_curve(y_val, y_pred_proba)
    idx_10 = np.argmin(np.abs(fpr - 0.10))
    r10 = tpr[idx_10]
    
    auc_scores.append(auc)
    recall_scores.append(recall)
    precision_scores.append(precision)
    f1_scores.append(f1)
    recall_at_10fpr_scores.append(r10)
    
    print(f"   Fold {fold:2d}: AUC={auc:.4f}  Recall={recall:.4f}  "
          f"Precision={precision:.4f}  F1={f1:.4f}  R@10%FPR={r10:.4f}")

# --- Summary ---
print("\n" + "-"*80)
print("📊 LOGISTIC REGRESSION SUMMARY")
print("-"*80)
print(f"AUC:           {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
print(f"Recall:        {np.mean(recall_scores):.4f} ± {np.std(recall_scores):.4f}")
print(f"Precision:     {np.mean(precision_scores):.4f} ± {np.std(precision_scores):.4f}")
print(f"F1:            {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"Recall@10%FPR: {np.mean(recall_at_10fpr_scores):.4f} ± {np.std(recall_at_10fpr_scores):.4f}")

# --- Feature Coefficients ---
print(f"\n🏆 Features by |Coefficient| ({len(LR_FEATURES)} features):")
best_lr.fit(X_scaled, y)
coefs = best_lr.coef_[0]
feat_imp = pd.DataFrame({
    'feature': X.columns,
    'coefficient': coefs,
    'abs_coef': np.abs(coefs)
}).sort_values('abs_coef', ascending=False)

for _, row in feat_imp.iterrows():
    bar = '█' * int(row['abs_coef'] * 3)
    print(f"   {row['feature']:35s} {row['coefficient']:+7.4f}  {bar}")

# --- Baseline Comparison ---
BASELINE_AUC = 0.7875  # ← UPDATE with your 14-feature baseline
print(f"\n📊 vs Baseline ({BASELINE_AUC:.4f}):")
current_auc = np.mean(auc_scores)
delta = current_auc - BASELINE_AUC
status = "✅ HOLD" if current_auc >= 0.79 else "⚠️  BELOW TARGET" if current_auc >= 0.785 else "❌ REGRESSED"
print(f"   Current AUC: {current_auc:.4f} (Δ = {delta:+.4f}) → {status}")
print(f"   Features: {len(LR_FEATURES)} (was 11)")

print("\n" + "="*80)


📊 EXPERIMENT 1: LOGISTIC REGRESSION

📊 Dataset: 41,188 samples, 3 features (LR-optimized)
   Features: ['cellular_crisis', 'euribor3m_local_rate', 'dow_month_encoded']
   Class distribution: {0: 36548, 1: 4640}

🔍 Pre-scaling data quality check...
   ✅ No inf/NaN values detected

🔍 Hyperparameter Tuning (GridSearchCV)...

✅ Best parameters: {'C': 0.001, 'class_weight': 'balanced', 'max_iter': 1000, 'penalty': 'l2', 'solver': 'lbfgs'}
✅ Best CV AUC: 0.7822

📊 10-Fold Cross-Validation...
   Fold  1: AUC=0.7828  Recall=0.6466  Precision=0.3145  F1=0.4231  R@10%FPR=0.5172
   Fold  2: AUC=0.7628  Recall=0.6250  Precision=0.3125  F1=0.4167  R@10%FPR=0.4914
   Fold  3: AUC=0.7872  Recall=0.6272  Precision=0.3089  F1=0.4139  R@10%FPR=0.5129
   Fold  4: AUC=0.7734  Recall=0.6466  Precision=0.3058  F1=0.4152  R@10%FPR=0.5194
   Fold  5: AUC=0.7881  Recall=0.6724  Precision=0.3358  F1=0.4480  R@10%FPR=0.5560
   Fold  6: AUC=0.7981  Recall=0.6638  Precision=0.3229  F1=0.4344  R@10%FPR=0.5733
   F

In [66]:
# ============================================================
# CELL 12B: LOGISTIC REGRESSION — BAYESIAN OPTIMIZATION (BALANCED)
# ============================================================
print("\n" + "=" * 80)
print("📊 EXPERIMENT 1: LOGISTIC REGRESSION — BALANCED")
print("=" * 80)

# ── Prepare data ──────────────────────────────────────────────
lr_X = df_engineered[LR_FEATURES].copy()
lr_y = df_engineered[TARGET_COL].copy()
print(f"\n📊 Dataset: {lr_X.shape[0]:,} samples, {lr_X.shape[1]} features (LR-optimized)")
print(f"   Features used: {LR_FEATURES}")
print(f"   Class distribution: {lr_y.value_counts().to_dict()}")

# ── Safety check ──────────────────────────────────────────────
print("\n🔍 Pre-scaling data quality check...")
lr_inf_mask = np.isinf(lr_X).any()
if lr_inf_mask.any():
    lr_inf_cols = lr_X.columns[lr_inf_mask].tolist()
    print(f"   ⚠️  Infinity in: {lr_inf_cols} → replacing with NaN")
    lr_X = lr_X.replace([np.inf, -np.inf], np.nan)
if lr_X.isnull().any().any():
    lr_nan_cols = lr_X.columns[lr_X.isnull().any()].tolist()
    print(f"   ⚠️  NaN in: {lr_nan_cols} → imputing with median")
    for col in lr_nan_cols:
        lr_median_val = lr_X[col].median()
        lr_X[col] = lr_X[col].fillna(lr_median_val)
        print(f"      {col}: filled with {lr_median_val:.4f}")
    print(f"   ✅ Data cleaned")
else:
    print(f"   ✅ No inf/NaN values detected")
assert not np.isinf(lr_X).any().any(), "Infinity values still present!"
assert not lr_X.isnull().any().any(), "NaN values still present!"

# ── Shared CV strategy ────────────────────────────────────────
LR_N_FOLDS = 5
LR_RANDOM_STATE = 42
lr_cv = StratifiedKFold(n_splits=LR_N_FOLDS, shuffle=True, random_state=LR_RANDOM_STATE)

# ==============================================================
# BAYESIAN OPTIMIZATION (Optuna) — 100 trials, BALANCED
# ==============================================================
print("\n" + "─" * 80)
print("🧠 Bayesian Optimization (Optuna) — 100 trials, 5-fold Stratified CV")
print("   ⚖️  class_weight = 'balanced' (fixed)")
print("─" * 80)

def lr_optuna_objective(trial: optuna.Trial) -> float:
    """Optuna objective for LR — balanced class weight."""
    C = trial.suggest_float('C', 1e-4, 1e3, log=True)
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2', 'elasticnet'])
    if penalty == 'elasticnet':
        l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1.0)
    else:
        l1_ratio = None

    lr_kwargs = dict(
        C=C, penalty=penalty, solver='saga', max_iter=5000,
        random_state=LR_RANDOM_STATE,
        class_weight='balanced',              # ← FIXED: always balanced
    )
    if penalty == 'elasticnet':
        lr_kwargs['l1_ratio'] = l1_ratio

    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(**lr_kwargs))
    ])

    fold_scores = []
    for fold_idx, (train_idx, val_idx) in enumerate(lr_cv.split(lr_X, lr_y)):
        X_train, X_val = lr_X.iloc[train_idx], lr_X.iloc[val_idx]
        y_train, y_val = lr_y.iloc[train_idx], lr_y.iloc[val_idx]
        try:
            pipe.fit(X_train, y_train)
            y_proba = pipe.predict_proba(X_val)[:, 1]
            auc = roc_auc_score(y_val, y_proba)
        except Exception:
            return float('-inf')
        fold_scores.append(auc)
        trial.report(np.mean(fold_scores), fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(fold_scores)

lr_study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=LR_RANDOM_STATE),
    pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=1),
    study_name='lr_bayesian_opt_balanced',
)

lr_t0 = time.perf_counter()
lr_study.optimize(lr_optuna_objective, n_trials=100, show_progress_bar=True)
lr_runtime = time.perf_counter() - lr_t0

# ── Extract results ───────────────────────────────────────────
lr_optuna_params = lr_study.best_params.copy()
lr_best_auc = lr_study.best_value
lr_n_complete = len([t for t in lr_study.trials
                     if t.state == optuna.trial.TrialState.COMPLETE])
lr_n_pruned = len([t for t in lr_study.trials
                   if t.state == optuna.trial.TrialState.PRUNED])

# ── Reconstruct best model ───────────────────────────────────
lr_bo_kwargs = dict(
    C=lr_optuna_params['C'],
    penalty=lr_optuna_params['penalty'],
    solver='saga', max_iter=5000, random_state=LR_RANDOM_STATE,
    class_weight='balanced',                  # ← FIXED: always balanced
)
if lr_optuna_params['penalty'] == 'elasticnet':
    lr_bo_kwargs['l1_ratio'] = lr_optuna_params['l1_ratio']

lr_best_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(**lr_bo_kwargs))
])

# ── Verify with full CV ──────────────────────────────────────
lr_verify_scores = cross_val_score(
    lr_best_pipe, lr_X, lr_y, cv=lr_cv, scoring='roc_auc', n_jobs=-1
)
lr_verify_mean = np.mean(lr_verify_scores)
lr_verify_std = np.std(lr_verify_scores)

# ── Print Optuna summary ─────────────────────────────────────
print(f"\n   ✅ Completed in {lr_runtime:.1f}s")
print(f"   Trials completed: {lr_n_complete} | Pruned: {lr_n_pruned}")
print(f"   Best CV ROC-AUC:  {lr_best_auc:.6f}")
print(f"   Verified Mean±Std: {lr_verify_mean:.6f} ± {lr_verify_std:.6f}")
print(f"\n   Best hyperparameters:")
print(f"      {'C':>15s} = {lr_optuna_params['C']:.6f}")
print(f"      {'penalty':>15s} = {lr_optuna_params['penalty']}")
if lr_optuna_params['penalty'] == 'elasticnet':
    print(f"      {'l1_ratio':>15s} = {lr_optuna_params['l1_ratio']:.4f}")
print(f"      {'class_weight':>15s} = balanced")
print(f"\n   Per-fold ROC-AUC: {['%.4f' % s for s in lr_verify_scores]}")

# ==============================================================
# 10-FOLD DETAILED EVALUATION
# ==============================================================
print("\n" + "─" * 80)
print("📊 10-Fold Cross-Validation Results (Final Evaluation)...")
print("─" * 80)

lr_cv_eval = StratifiedKFold(n_splits=10, shuffle=True, random_state=LR_RANDOM_STATE)
lr_auc_scores = []
lr_recall_scores = []
lr_precision_scores = []
lr_f1_scores = []

for fold, (train_idx, val_idx) in enumerate(lr_cv_eval.split(lr_X, lr_y), 1):
    X_train, X_val = lr_X.iloc[train_idx], lr_X.iloc[val_idx]
    y_train, y_val = lr_y.iloc[train_idx], lr_y.iloc[val_idx]

    lr_best_pipe.fit(X_train, y_train)
    y_pred_proba = lr_best_pipe.predict_proba(X_val)[:, 1]
    y_pred = lr_best_pipe.predict(X_val)

    auc = roc_auc_score(y_val, y_pred_proba)
    recall = recall_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)

    lr_auc_scores.append(auc)
    lr_recall_scores.append(recall)
    lr_precision_scores.append(precision)
    lr_f1_scores.append(f1)

    print(f"   Fold {fold:2d}: AUC={auc:.4f}, Recall={recall:.4f}, "
          f"Precision={precision:.4f}, F1={f1:.4f}")

# ── Summary statistics ────────────────────────────────────────
print("\n" + "-" * 80)
print("📊 LOGISTIC REGRESSION SUMMARY — BALANCED")
print("-" * 80)
print(f"AUC:       {np.mean(lr_auc_scores):.4f} ± {np.std(lr_auc_scores):.4f}")
print(f"Recall:    {np.mean(lr_recall_scores):.4f} ± {np.std(lr_recall_scores):.4f}")
print(f"Precision: {np.mean(lr_precision_scores):.4f} ± {np.std(lr_precision_scores):.4f}")
print(f"F1:        {np.mean(lr_f1_scores):.4f} ± {np.std(lr_f1_scores):.4f}")

# ── Feature coefficients ─────────────────────────────────────
print("\n🏆 Features by |Coefficient|:")
lr_best_pipe.fit(lr_X, lr_y)
lr_best_model = lr_best_pipe.named_steps['clf']
lr_coefs = lr_best_model.coef_[0]
lr_feature_importance = pd.DataFrame({
    'feature': lr_X.columns,
    'coefficient': lr_coefs,
    'abs_coef': np.abs(lr_coefs)
}).sort_values('abs_coef', ascending=False)

for _, row in lr_feature_importance.iterrows():
    status = "🔥 ACTIVE" if row['abs_coef'] >= 0.15 else "⚪ WEAK" if row['abs_coef'] >= 0.10 else "💀 DEAD"
    print(f"   {row['feature']:30s} {row['coefficient']:+.4f}  {status}")

# ── Recall at 10% FPR ────────────────────────────────────────
print("\n📊 Recall at 10% FPR:")
lr_y_pred_proba = lr_best_pipe.predict_proba(lr_X)[:, 1]
lr_fpr, lr_tpr, _ = roc_curve(lr_y, lr_y_pred_proba)
lr_idx_10fpr = np.argmin(np.abs(lr_fpr - 0.10))
lr_recall_at_10fpr = lr_tpr[lr_idx_10fpr]
print(f"   Recall @ 10% FPR: {lr_recall_at_10fpr:.4f}")

# ── Structured summary ───────────────────────────────────────
lr_cv_std = np.std(lr_auc_scores)
print("\n" + "=" * 80)
print("📊 LOGISTIC REGRESSION — TUNING SUMMARY (BALANCED)")
print("=" * 80)
print(f"""
┌────────────────────────────────────────────────────────────┐
│  Bayesian Optimization (Optuna) — LR BALANCED              │
├────────────────────────────────────────────────────────────┤
│  Trials:           100 ({lr_n_complete} completed, {lr_n_pruned} pruned)       │
│  Runtime:          {lr_runtime:>7.1f}s                                  │
│  Best ROC-AUC:     {lr_best_auc:.6f}                                │
│  10-Fold Mean±Std: {np.mean(lr_auc_scores):.4f} ± {lr_cv_std:.4f}                          │
├────────────────────────────────────────────────────────────┤
│  C:                {lr_optuna_params['C']:<10.6f}                              │
│  penalty:          {lr_optuna_params['penalty']:<10}                              │
│  class_weight:     balanced                                │
└────────────────────────────────────────────────────────────┘
💡 Stability: CV std = {lr_cv_std:.4f} ({'Low variance — stable' if lr_cv_std < 0.015 else 'Moderate variance' if lr_cv_std < 0.03 else 'High variance'})
""")
print(f"   💾 lr_optuna_params saved: {lr_optuna_params}")
print("=" * 80)


📊 EXPERIMENT 1: LOGISTIC REGRESSION — BALANCED

📊 Dataset: 41,188 samples, 3 features (LR-optimized)
   Features used: ['cellular_crisis', 'euribor3m_local_rate', 'dow_month_encoded']
   Class distribution: {0: 36548, 1: 4640}

🔍 Pre-scaling data quality check...
   ✅ No inf/NaN values detected

────────────────────────────────────────────────────────────────────────────────
🧠 Bayesian Optimization (Optuna) — 100 trials, 5-fold Stratified CV
   ⚖️  class_weight = 'balanced' (fixed)
────────────────────────────────────────────────────────────────────────────────


Best trial: 7. Best value: 0.782236: 100%|██████████| 100/100 [00:24<00:00,  4.16it/s]



   ✅ Completed in 24.1s
   Trials completed: 37 | Pruned: 63
   Best CV ROC-AUC:  0.782236
   Verified Mean±Std: 0.782236 ± 0.006641

   Best hyperparameters:
                    C = 0.000285
              penalty = l2
         class_weight = balanced

   Per-fold ROC-AUC: ['0.7727', '0.7802', '0.7934', '0.7827', '0.7822']

────────────────────────────────────────────────────────────────────────────────
📊 10-Fold Cross-Validation Results (Final Evaluation)...
────────────────────────────────────────────────────────────────────────────────
   Fold  1: AUC=0.7828, Recall=0.6466, Precision=0.3148, F1=0.4234
   Fold  2: AUC=0.7624, Recall=0.6250, Precision=0.3125, F1=0.4167
   Fold  3: AUC=0.7872, Recall=0.6272, Precision=0.3089, F1=0.4139
   Fold  4: AUC=0.7734, Recall=0.6466, Precision=0.3058, F1=0.4152
   Fold  5: AUC=0.7881, Recall=0.6724, Precision=0.3358, F1=0.4480
   Fold  6: AUC=0.7986, Recall=0.6638, Precision=0.3229, F1=0.4344
   Fold  7: AUC=0.7764, Recall=0.6272, Precision=0.3

In [67]:
# ============================================================
# CELL 12C: LOGISTIC REGRESSION — BAYESIAN OPTIMIZATION (RECALL-BIASED)
# ============================================================
print("\n" + "=" * 80)
print("📊 EXPERIMENT 1: LOGISTIC REGRESSION — RECALL-BIASED")
print("=" * 80)

# ── Prepare data ──────────────────────────────────────────────
lr_recall_X = df_engineered[LR_FEATURES].copy()
lr_recall_y = df_engineered[TARGET_COL].copy()
print(f"\n📊 Dataset: {lr_recall_X.shape[0]:,} samples, {lr_recall_X.shape[1]} features (LR-optimized)")
print(f"   Features used: {LR_FEATURES}")
print(f"   Class distribution: {lr_recall_y.value_counts().to_dict()}")
print(f"   🎯 RECALL-BIASED: Optuna tunes minority_weight (1.0–10.0x)")
print(f"      1.0 = no weighting, 'balanced' ≈ {36548/4640:.1f}x — full spectrum covered")

# ── Safety check ──────────────────────────────────────────────
print("\n🔍 Pre-scaling data quality check...")
lr_recall_inf_mask = np.isinf(lr_recall_X).any()
if lr_recall_inf_mask.any():
    lr_recall_X = lr_recall_X.replace([np.inf, -np.inf], np.nan)
if lr_recall_X.isnull().any().any():
    lr_recall_nan_cols = lr_recall_X.columns[lr_recall_X.isnull().any()].tolist()
    print(f"   ⚠️  NaN in: {lr_recall_nan_cols} → imputing with median")
    for col in lr_recall_nan_cols:
        lr_recall_X[col] = lr_recall_X[col].fillna(lr_recall_X[col].median())
    print(f"   ✅ Data cleaned")
else:
    print(f"   ✅ No inf/NaN values detected")
assert not np.isinf(lr_recall_X).any().any(), "Infinity values still present!"
assert not lr_recall_X.isnull().any().any(), "NaN values still present!"

# ── Shared CV strategy ────────────────────────────────────────
LR_RECALL_N_FOLDS = 5
LR_RECALL_RANDOM_STATE = 42
lr_recall_cv = StratifiedKFold(n_splits=LR_RECALL_N_FOLDS, shuffle=True, random_state=LR_RECALL_RANDOM_STATE)

# ==============================================================
# BAYESIAN OPTIMIZATION (Optuna) — 100 trials, RECALL-BIASED
# ==============================================================
print("\n" + "─" * 80)
print("🧠 Bayesian Optimization (Optuna) — 100 trials, 5-fold Stratified CV")
print("   🎯 class_weight = {0: 1, 1: minority_weight} — Optuna tunes minority_weight")
print("─" * 80)

def lr_recall_optuna_objective(trial: optuna.Trial) -> float:
    """
    Optuna objective for LR — recall-biased.
    Tunes minority_weight alongside regularization params.
    Still optimizes AUC so ranking quality doesn't collapse.
    """
    C = trial.suggest_float('C', 1e-4, 1e3, log=True)
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2', 'elasticnet'])
    if penalty == 'elasticnet':
        l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1.0)
    else:
        l1_ratio = None

    # ── Recall bias: tune minority class weight ──────────────
    # 'balanced' would be ~7.9x for this dataset (36548/4640)
    # We search 1x–10x: 1.0=no weighting, 7.9≈balanced, 10=aggressive
    minority_weight = trial.suggest_float('minority_weight', 1.0, 10.0)
    class_weight = {0: 1, 1: minority_weight}

    lr_kwargs = dict(
        C=C, penalty=penalty, solver='saga', max_iter=5000,
        random_state=LR_RECALL_RANDOM_STATE,
        class_weight=class_weight,
    )
    if penalty == 'elasticnet':
        lr_kwargs['l1_ratio'] = l1_ratio

    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(**lr_kwargs))
    ])

    fold_scores = []
    for fold_idx, (train_idx, val_idx) in enumerate(lr_recall_cv.split(lr_recall_X, lr_recall_y)):
        X_train, X_val = lr_recall_X.iloc[train_idx], lr_recall_X.iloc[val_idx]
        y_train, y_val = lr_recall_y.iloc[train_idx], lr_recall_y.iloc[val_idx]
        try:
            pipe.fit(X_train, y_train)
            y_proba = pipe.predict_proba(X_val)[:, 1]
            auc = roc_auc_score(y_val, y_proba)
        except Exception:
            return float('-inf')
        fold_scores.append(auc)
        trial.report(np.mean(fold_scores), fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(fold_scores)

lr_recall_study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=LR_RECALL_RANDOM_STATE),
    pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=1),
    study_name='lr_bayesian_opt_recall',
)

lr_recall_t0 = time.perf_counter()
lr_recall_study.optimize(lr_recall_optuna_objective, n_trials=100, show_progress_bar=True)
lr_recall_runtime = time.perf_counter() - lr_recall_t0

# ── Extract results ───────────────────────────────────────────
lr_recall_optuna_params = lr_recall_study.best_params.copy()
lr_recall_best_auc = lr_recall_study.best_value
lr_recall_n_complete = len([t for t in lr_recall_study.trials
                            if t.state == optuna.trial.TrialState.COMPLETE])
lr_recall_n_pruned = len([t for t in lr_recall_study.trials
                          if t.state == optuna.trial.TrialState.PRUNED])

# ── Reconstruct best model ───────────────────────────────────
lr_recall_best_minority_weight = lr_recall_optuna_params['minority_weight']
lr_recall_best_class_weight = {0: 1, 1: lr_recall_best_minority_weight}

lr_recall_kwargs = dict(
    C=lr_recall_optuna_params['C'],
    penalty=lr_recall_optuna_params['penalty'],
    solver='saga', max_iter=5000, random_state=LR_RECALL_RANDOM_STATE,
    class_weight=lr_recall_best_class_weight,
)
if lr_recall_optuna_params['penalty'] == 'elasticnet':
    lr_recall_kwargs['l1_ratio'] = lr_recall_optuna_params['l1_ratio']

lr_recall_best_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(**lr_recall_kwargs))
])

# ── Verify with full CV ──────────────────────────────────────
lr_recall_verify_scores = cross_val_score(
    lr_recall_best_pipe, lr_recall_X, lr_recall_y,
    cv=lr_recall_cv, scoring='roc_auc', n_jobs=-1
)
lr_recall_verify_mean = np.mean(lr_recall_verify_scores)
lr_recall_verify_std = np.std(lr_recall_verify_scores)

# ── Print Optuna summary ─────────────────────────────────────
print(f"\n   ✅ Completed in {lr_recall_runtime:.1f}s")
print(f"   Trials completed: {lr_recall_n_complete} | Pruned: {lr_recall_n_pruned}")
print(f"   Best CV ROC-AUC:  {lr_recall_best_auc:.6f}")
print(f"   Verified Mean±Std: {lr_recall_verify_mean:.6f} ± {lr_recall_verify_std:.6f}")
print(f"\n   Best hyperparameters:")
print(f"      {'C':>15s} = {lr_recall_optuna_params['C']:.6f}")
print(f"      {'penalty':>15s} = {lr_recall_optuna_params['penalty']}")
if lr_recall_optuna_params['penalty'] == 'elasticnet':
    print(f"      {'l1_ratio':>15s} = {lr_recall_optuna_params['l1_ratio']:.4f}")
print(f"      {'minority_weight':>15s} = {lr_recall_best_minority_weight:.2f}")
print(f"      {'class_weight':>15s} = {{0: 1, 1: {lr_recall_best_minority_weight:.2f}}}")
print(f"\n   Per-fold ROC-AUC: {['%.4f' % s for s in lr_recall_verify_scores]}")

# ==============================================================
# 10-FOLD DETAILED EVALUATION
# ==============================================================
print("\n" + "─" * 80)
print("📊 10-Fold Cross-Validation Results (Final Evaluation)...")
print("─" * 80)

lr_recall_cv_eval = StratifiedKFold(n_splits=10, shuffle=True, random_state=LR_RECALL_RANDOM_STATE)
lr_recall_auc_scores = []
lr_recall_recall_scores = []
lr_recall_precision_scores = []
lr_recall_f1_scores = []

for fold, (train_idx, val_idx) in enumerate(lr_recall_cv_eval.split(lr_recall_X, lr_recall_y), 1):
    X_train, X_val = lr_recall_X.iloc[train_idx], lr_recall_X.iloc[val_idx]
    y_train, y_val = lr_recall_y.iloc[train_idx], lr_recall_y.iloc[val_idx]

    lr_recall_best_pipe.fit(X_train, y_train)
    y_pred_proba = lr_recall_best_pipe.predict_proba(X_val)[:, 1]
    y_pred = lr_recall_best_pipe.predict(X_val)

    auc = roc_auc_score(y_val, y_pred_proba)
    recall = recall_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)

    lr_recall_auc_scores.append(auc)
    lr_recall_recall_scores.append(recall)
    lr_recall_precision_scores.append(precision)
    lr_recall_f1_scores.append(f1)

    print(f"   Fold {fold:2d}: AUC={auc:.4f}, Recall={recall:.4f}, "
          f"Precision={precision:.4f}, F1={f1:.4f}")

# ── Summary statistics ────────────────────────────────────────
print("\n" + "-" * 80)
print("📊 LOGISTIC REGRESSION SUMMARY — RECALL-BIASED")
print("-" * 80)
print(f"AUC:       {np.mean(lr_recall_auc_scores):.4f} ± {np.std(lr_recall_auc_scores):.4f}")
print(f"Recall:    {np.mean(lr_recall_recall_scores):.4f} ± {np.std(lr_recall_recall_scores):.4f}")
print(f"Precision: {np.mean(lr_recall_precision_scores):.4f} ± {np.std(lr_recall_precision_scores):.4f}")
print(f"F1:        {np.mean(lr_recall_f1_scores):.4f} ± {np.std(lr_recall_f1_scores):.4f}")

# ── Feature coefficients ─────────────────────────────────────
print("\n🏆 Features by |Coefficient|:")
lr_recall_best_pipe.fit(lr_recall_X, lr_recall_y)
lr_recall_best_model = lr_recall_best_pipe.named_steps['clf']
lr_recall_coefs = lr_recall_best_model.coef_[0]
lr_recall_feature_importance = pd.DataFrame({
    'feature': lr_recall_X.columns,
    'coefficient': lr_recall_coefs,
    'abs_coef': np.abs(lr_recall_coefs)
}).sort_values('abs_coef', ascending=False)

for _, row in lr_recall_feature_importance.iterrows():
    status = "🔥 ACTIVE" if row['abs_coef'] >= 0.15 else "⚪ WEAK" if row['abs_coef'] >= 0.10 else "💀 DEAD"
    print(f"   {row['feature']:30s} {row['coefficient']:+.4f}  {status}")

# ── Recall at 10% FPR ────────────────────────────────────────
print("\n📊 Recall at 10% FPR:")
lr_recall_y_pred_proba = lr_recall_best_pipe.predict_proba(lr_recall_X)[:, 1]
lr_recall_fpr, lr_recall_tpr, _ = roc_curve(lr_recall_y, lr_recall_y_pred_proba)
lr_recall_idx_10fpr = np.argmin(np.abs(lr_recall_fpr - 0.10))
lr_recall_at_10fpr = lr_recall_tpr[lr_recall_idx_10fpr]
print(f"   Recall @ 10% FPR: {lr_recall_at_10fpr:.4f}")

# ── Structured summary ───────────────────────────────────────
lr_recall_cv_std = np.std(lr_recall_auc_scores)
print("\n" + "=" * 80)
print("📊 LOGISTIC REGRESSION — TUNING SUMMARY (RECALL-BIASED)")
print("=" * 80)
print(f"""
┌────────────────────────────────────────────────────────────┐
│  Bayesian Optimization (Optuna) — LR RECALL-BIASED         │
├────────────────────────────────────────────────────────────┤
│  Trials:           100 ({lr_recall_n_complete} completed, {lr_recall_n_pruned} pruned)       │
│  Runtime:          {lr_recall_runtime:>7.1f}s                                  │
│  Best ROC-AUC:     {lr_recall_best_auc:.6f}                                │
│  10-Fold Mean±Std: {np.mean(lr_recall_auc_scores):.4f} ± {lr_recall_cv_std:.4f}                          │
├────────────────────────────────────────────────────────────┤
│  C:                {lr_recall_optuna_params['C']:<10.6f}                              │
│  penalty:          {lr_recall_optuna_params['penalty']:<10}                              │
│  minority_weight:  {lr_recall_best_minority_weight:<10.2f}                              │
│  class_weight:     {{0: 1, 1: {lr_recall_best_minority_weight:.2f}}}                       │
└────────────────────────────────────────────────────────────┘
💡 Stability: CV std = {lr_recall_cv_std:.4f} ({'Low variance — stable' if lr_recall_cv_std < 0.015 else 'Moderate variance' if lr_recall_cv_std < 0.03 else 'High variance'})
""")

# ── Compare with balanced ─────────────────────────────────────
print("─" * 80)
print("📊 COMPARISON: Recall-Biased vs Balanced")
print("─" * 80)
print(f"   {'Metric':<20s} {'Recall-Biased':>15s} {'Balanced (12B)':>15s}")
print(f"   {'─'*50}")
print(f"   {'AUC':<20s} {np.mean(lr_recall_auc_scores):>15.4f} {'(run 12B first)':>15s}")
print(f"   {'Recall':<20s} {np.mean(lr_recall_recall_scores):>15.4f} {'':>15s}")
print(f"   {'Precision':<20s} {np.mean(lr_recall_precision_scores):>15.4f} {'':>15s}")
print(f"   {'F1':<20s} {np.mean(lr_recall_f1_scores):>15.4f} {'':>15s}")
print(f"   {'minority_weight':<20s} {lr_recall_best_minority_weight:>15.2f} {'~7.9 (auto)':>15s}")
print(f"\n   💡 If minority_weight lands near 7.9, recall-biased ≈ balanced")
print(f"   💡 If it lands near 1.0, no weighting is optimal (class_weight=None)")
print(f"   💡 If it lands 2-5, balanced was over-weighting the minority class")
print(f"\n   🎯 CASCADE STRATEGY NOTES:")
print(f"      If LR recall-biased outperforms → LR = wide net, RF/EBM = refinement")
print(f"      If LR balanced is better → consider recall bias on RF or EBM instead")
print(f"      Key metric to watch: Recall@10%FPR (ranking quality at low false positive)")

print(f"\n   💾 lr_recall_optuna_params saved: {lr_recall_optuna_params}")
print("=" * 80)


📊 EXPERIMENT 1: LOGISTIC REGRESSION — RECALL-BIASED

📊 Dataset: 41,188 samples, 3 features (LR-optimized)
   Features used: ['cellular_crisis', 'euribor3m_local_rate', 'dow_month_encoded']
   Class distribution: {0: 36548, 1: 4640}
   🎯 RECALL-BIASED: Optuna tunes minority_weight (1.0–10.0x)
      1.0 = no weighting, 'balanced' ≈ 7.9x — full spectrum covered

🔍 Pre-scaling data quality check...
   ✅ No inf/NaN values detected

────────────────────────────────────────────────────────────────────────────────
🧠 Bayesian Optimization (Optuna) — 100 trials, 5-fold Stratified CV
   🎯 class_weight = {0: 1, 1: minority_weight} — Optuna tunes minority_weight
────────────────────────────────────────────────────────────────────────────────


Best trial: 61. Best value: 0.782197: 100%|██████████| 100/100 [01:20<00:00,  1.24it/s]



   ✅ Completed in 80.7s
   Trials completed: 33 | Pruned: 67
   Best CV ROC-AUC:  0.782197
   Verified Mean±Std: 0.782197 ± 0.006463

   Best hyperparameters:
                    C = 0.199848
              penalty = elasticnet
             l1_ratio = 0.9376
      minority_weight = 9.87
         class_weight = {0: 1, 1: 9.87}

   Per-fold ROC-AUC: ['0.7730', '0.7801', '0.7931', '0.7825', '0.7823']

────────────────────────────────────────────────────────────────────────────────
📊 10-Fold Cross-Validation Results (Final Evaluation)...
────────────────────────────────────────────────────────────────────────────────
   Fold  1: AUC=0.7830, Recall=0.6897, Precision=0.2844, F1=0.4028
   Fold  2: AUC=0.7627, Recall=0.6681, Precision=0.2773, F1=0.3919
   Fold  3: AUC=0.7870, Recall=0.6659, Precision=0.2754, F1=0.3897
   Fold  4: AUC=0.7734, Recall=0.6875, Precision=0.2796, F1=0.3975
   Fold  5: AUC=0.7882, Recall=0.7047, Precision=0.2935, F1=0.4144
   Fold  6: AUC=0.7980, Recall=0.7004, Preci

In [68]:
# ============================================================
# CELL 12D: LR FEATURE DIAGNOSTICS (CORRELATION, VIF, CLASS SEPARATION, COEFFICIENTS)
# ============================================================

print("\n" + "="*80)
print("🔬 LR FEATURE DIAGNOSTICS")
print(f"   Feature set: {len(LR_FEATURES)} features")
print("="*80)

# --- Setup ---
diag_X = df_engineered[LR_FEATURES].copy()
diag_y = df_engineered[TARGET_COL]

# Get model + scaler from 12B (falls back to 12A if 12B hasn't run)
try:
    _lr_model = lr_best_pipe.named_steps['clf']
    _lr_scaler = lr_best_pipe.named_steps['scaler']
    lr_best_pipe.fit(diag_X, diag_y)
    _lr_coefs = _lr_model.coef_[0]
    _source = "12B (Optuna)"
except NameError:
    try:
        best_lr.fit(StandardScaler().fit_transform(diag_X), diag_y)
        _lr_coefs = best_lr.coef_[0]
        _source = "12A (GridSearch)"
    except NameError:
        print("❌ No LR model found. Run Cell 12A or 12B first.")
        raise

print(f"   Model source: {_source}")

# ============================================================
# 1️⃣  CORRELATION MATRIX
# ============================================================
print("\n" + "-"*80)
print("1️⃣  CORRELATION ANALYSIS")
print("-"*80)

corr_matrix = diag_X.corr()

# Find high correlation pairs
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.7:
            high_corr_pairs.append({
                'Feature 1': corr_matrix.columns[i],
                'Feature 2': corr_matrix.columns[j],
                'Correlation': r
            })

if high_corr_pairs:
    high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlation', key=abs, ascending=False)
    print(f"\n📊 High Correlation Pairs (|r| > 0.7): {len(high_corr_df)} pairs")
    for _, row in high_corr_df.iterrows():
        flag = "🔴" if abs(row['Correlation']) > 0.9 else "🟡"
        print(f"   {flag} {row['Feature 1']:35s} ↔ {row['Feature 2']:35s}  r={row['Correlation']:+.3f}")
else:
    print("\n✅ No high correlation pairs (|r| > 0.7) — multicollinearity resolved!")

# ============================================================
# 2️⃣  VIF ANALYSIS
# ============================================================
print("\n" + "-"*80)
print("2️⃣  VARIANCE INFLATION FACTOR (VIF)")
print("-"*80)

# Scale for VIF (VIF needs standardized data)
from sklearn.preprocessing import StandardScaler
diag_X_scaled = pd.DataFrame(
    StandardScaler().fit_transform(diag_X),
    columns=diag_X.columns
)

vif_data = []
for i, col in enumerate(diag_X_scaled.columns):
    try:
        vif_val = variance_inflation_factor(diag_X_scaled.values, i)
    except Exception:
        vif_val = float('inf')
    vif_data.append({'feature': col, 'VIF': vif_val})

vif_df = pd.DataFrame(vif_data).sort_values('VIF', ascending=False)

high_vif_count = 0
for _, row in vif_df.iterrows():
    if row['VIF'] > 10:
        flag = "❌ HIGH"
        high_vif_count += 1
    elif row['VIF'] > 5:
        flag = "⚠️  MOD"
    else:
        flag = "✅ LOW"
    print(f"   {row['feature']:40s} VIF={row['VIF']:>8.2f}  {flag}")

if high_vif_count == 0:
    print(f"\n✅ All VIF < 10 — multicollinearity resolved!")
else:
    print(f"\n⚠️  {high_vif_count} features with VIF > 10")

# ============================================================
# 3️⃣  CLASS SEPARATION
# ============================================================
print("\n" + "-"*80)
print("3️⃣  CLASS SEPARATION ANALYSIS")
print("-"*80)

pos_mask = diag_y == 1
neg_mask = diag_y == 0

separation_data = []
for col in LR_FEATURES:
    vals_pos = diag_X.loc[pos_mask, col]
    vals_neg = diag_X.loc[neg_mask, col]
    
    # Cohen's d
    pooled_std = np.sqrt((vals_neg.std()**2 + vals_pos.std()**2) / 2)
    cohens_d = (vals_pos.mean() - vals_neg.mean()) / pooled_std if pooled_std > 0 else 0
    
    # KS statistic
    ks_stat, _ = ks_2samp(vals_pos, vals_neg)
    
    # Point-biserial correlation
    r_pb, _ = pointbiserialr(diag_y, diag_X[col])
    
    separation_data.append({
        'feature': col,
        'cohens_d': cohens_d,
        'ks_stat': ks_stat,
        'r_pb': r_pb,
        'abs_r': abs(r_pb)
    })

sep_df = pd.DataFrame(separation_data).sort_values('abs_r', ascending=False)

for _, row in sep_df.iterrows():
    if row['abs_r'] >= 0.2:
        flag = "✅ STRONG"
    elif row['abs_r'] >= 0.05:
        flag = "⚠️  MOD"
    else:
        flag = "❌ WEAK"
    print(f"   {row['feature']:40s} d={row['cohens_d']:+.3f}  KS={row['ks_stat']:.3f}  r={row['r_pb']:+.3f}  {flag}")

weak_seps = sep_df[sep_df['abs_r'] < 0.05]['feature'].tolist()
if weak_seps:
    print(f"\n⚠️  Weak separators (|r| < 0.05): {weak_seps}")

# ============================================================
# 4️⃣  COEFFICIENTS (from current model)
# ============================================================
print("\n" + "-"*80)
print(f"4️⃣  MODEL COEFFICIENTS ({_source})")
print("-"*80)

coef_df = pd.DataFrame({
    'feature': LR_FEATURES,
    'coefficient': _lr_coefs,
    'abs_coef': np.abs(_lr_coefs)
}).sort_values('abs_coef', ascending=False)

for _, row in coef_df.iterrows():
    bar = '█' * int(row['abs_coef'] * 5)
    if row['abs_coef'] < 0.1:
        flag = "💀 DEAD"
    elif row['abs_coef'] < 0.3:
        flag = "⚪ WEAK"
    else:
        flag = "🔥 ACTIVE"
    print(f"   {row['feature']:40s} {row['coefficient']:+7.4f}  {bar}  {flag}")

dead_feats = coef_df[coef_df['abs_coef'] < 0.1]['feature'].tolist()
if dead_feats:
    print(f"\n💀 Near-zero coefficients (candidates to drop): {dead_feats}")

# ============================================================
# 5️⃣  SUMMARY & RECOMMENDATIONS
# ============================================================
print("\n" + "-"*80)
print("5️⃣  SUMMARY")
print("-"*80)

print(f"\n   Features:          {len(LR_FEATURES)}")
print(f"   High VIF (>10):    {high_vif_count}")
print(f"   High corr (>0.7):  {len(high_corr_pairs) if high_corr_pairs else 0} pairs")
print(f"   Dead coefs (<0.1): {len(dead_feats)}")
print(f"   Weak sep (<0.05):  {len(weak_seps)}")

if high_vif_count == 0 and len(dead_feats) == 0 and len(weak_seps) == 0:
    print(f"\n   ✅ Feature set looks clean! Ready for final Optuna validation.")
elif high_vif_count == 0 and len(dead_feats) > 0:
    print(f"\n   ⚠️  VIF resolved but {len(dead_feats)} dead features remain.")
    print(f"      Consider dropping: {dead_feats}")
else:
    print(f"\n   ⚠️  Still has multicollinearity issues. Continue reducing.")

# --- Heatmap ---
print("\n📊 Generating correlation heatmap...")

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(
    corr_matrix, 
    annot=True, fmt='.2f', 
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    mask=mask,
    square=True, ax=ax,
    cbar_kws={'shrink': 0.8}
)
ax.set_title(f'LR Features Correlation Matrix ({len(LR_FEATURES)} features)', fontsize=14)
plt.tight_layout()
plt.show()

# --- Violin plots for top separators ---
top_seps = sep_df.head(6)['feature'].tolist()
if len(top_seps) > 0:
    print(f"\n📊 Violin plots for top {len(top_seps)} separators...")
    n_plots = len(top_seps)
    n_cols = 3
    n_rows = (n_plots + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    axes = axes.flatten() if n_plots > 1 else [axes]
    
    for idx, feat in enumerate(top_seps):
        ax = axes[idx]
        data_plot = pd.DataFrame({
            'Value': diag_X[feat],
            'Class': diag_y.map({0: 'No Subscribe', 1: 'Subscribe'})
        })
        sns.violinplot(data=data_plot, x='Class', y='Value', ax=ax,
                      palette={'No Subscribe': '#e74c3c', 'Subscribe': '#2ecc71'})
        feat_row = sep_df[sep_df['feature'] == feat].iloc[0]
        ax.set_title(f"{feat}\nd={feat_row['cohens_d']:.3f}, r={feat_row['r_pb']:.3f}", fontsize=10)
        ax.set_xlabel('')
    
    # Hide empty subplots
    for idx in range(len(top_seps), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()

print("\n" + "="*80)
print("✅ LR Diagnostics complete")
print("="*80)


🔬 LR FEATURE DIAGNOSTICS
   Feature set: 3 features
   Model source: 12B (Optuna)

--------------------------------------------------------------------------------
1️⃣  CORRELATION ANALYSIS
--------------------------------------------------------------------------------

✅ No high correlation pairs (|r| > 0.7) — multicollinearity resolved!

--------------------------------------------------------------------------------
2️⃣  VARIANCE INFLATION FACTOR (VIF)
--------------------------------------------------------------------------------
   euribor3m_local_rate                     VIF=    1.97  ✅ LOW
   cellular_crisis                          VIF=    1.59  ✅ LOW
   dow_month_encoded                        VIF=    1.35  ✅ LOW

✅ All VIF < 10 — multicollinearity resolved!

--------------------------------------------------------------------------------
3️⃣  CLASS SEPARATION ANALYSIS
--------------------------------------------------------------------------------
   euribor3m_local_rate  

In [69]:
# ============================================================
# CELL 13A: RANDOM FOREST EXPERIMENT
# ============================================================
print("\n" + "="*80)
print("🌲 EXPERIMENT 2: RANDOM FOREST")
print("="*80)
# Prepare data with RF-specific features 
X = df_engineered[RF_FEATURES]
y = df_engineered[TARGET_COL]
print(f"\n📊 Dataset: {X.shape[0]:,} samples, {X.shape[1]} features (RF-optimized)")
print(f"   Using all engineered features - RF handles non-linearity naturally")
print(f"   Class distribution: {y.value_counts().to_dict()}")
print("\n🔍 Hyperparameter Tuning (GridSearchCV)...")
# Tight grid around Optuna-validated params
param_grid = {
    'n_estimators': [750, 1000],
    'max_depth': [7, 9, 11],
    'min_samples_leaf': [13, 18, 23],
    'max_features': ['sqrt', 'log2'],
    'class_weight': ['balanced']
}
# Setup cross-validation
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
# Grid search
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
grid_search = GridSearchCV(
    rf,
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)
grid_search.fit(X, y)
print(f"\n✅ Best parameters: {grid_search.best_params_}")
print(f"✅ Best CV AUC: {grid_search.best_score_:.4f}")
# Get best model
best_rf = grid_search.best_estimator_
# 10-fold evaluation with best model
print("\n📊 10-Fold Cross-Validation Results...")
auc_scores = []
recall_scores = []
precision_scores = []
f1_scores = []
for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # Train
    best_rf.fit(X_train, y_train)
    
    # Predict
    y_pred_proba = best_rf.predict_proba(X_val)[:, 1]
    y_pred = best_rf.predict(X_val)
    
    # Metrics
    auc = roc_auc_score(y_val, y_pred_proba)
    recall = recall_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    
    auc_scores.append(auc)
    recall_scores.append(recall)
    precision_scores.append(precision)
    f1_scores.append(f1)
    
    print(f"   Fold {fold:2d}: AUC={auc:.4f}, Recall={recall:.4f}, Precision={precision:.4f}, F1={f1:.4f}")
# Summary statistics
print("\n" + "-"*80)
print("🌲 RANDOM FOREST SUMMARY")
print("-"*80)
print(f"AUC:       {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
print(f"Recall:    {np.mean(recall_scores):.4f} ± {np.std(recall_scores):.4f}")
print(f"Precision: {np.mean(precision_scores):.4f} ± {np.std(precision_scores):.4f}")
print(f"F1:        {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
# Feature importance
print("\n🏆 Top 15 Features by Importance:")
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': best_rf.feature_importances_
}).sort_values('importance', ascending=False)
for i, row in feature_importance.head(15).iterrows():
    print(f"   {row['feature']:30s} {row['importance']:.4f}")
# Calculate Recall at 10% FPR
print("\n📊 Recall at 10% FPR:")
best_rf.fit(X, y)
y_pred_proba = best_rf.predict_proba(X)[:, 1]
fpr, tpr, thresholds = roc_curve(y, y_pred_proba)
idx_10fpr = np.argmin(np.abs(fpr - 0.10))
recall_at_10fpr = tpr[idx_10fpr]
print(f"   Recall @ 10% FPR: {recall_at_10fpr:.4f}")
print("\n" + "="*80)


🌲 EXPERIMENT 2: RANDOM FOREST

📊 Dataset: 41,188 samples, 8 features (RF-optimized)
   Using all engineered features - RF handles non-linearity naturally
   Class distribution: {0: 36548, 1: 4640}

🔍 Hyperparameter Tuning (GridSearchCV)...

✅ Best parameters: {'class_weight': 'balanced', 'max_depth': 7, 'max_features': 'log2', 'min_samples_leaf': 18, 'n_estimators': 750}
✅ Best CV AUC: 0.1317

📊 10-Fold Cross-Validation Results...
   Fold  1: AUC=0.8075, Recall=0.6228, Precision=0.3813, F1=0.4730
   Fold  2: AUC=0.7873, Recall=0.6013, Precision=0.3545, F1=0.4460
   Fold  3: AUC=0.8105, Recall=0.6358, Precision=0.3753, F1=0.4720
   Fold  4: AUC=0.7958, Recall=0.6121, Precision=0.3688, F1=0.4603
   Fold  5: AUC=0.8121, Recall=0.6466, Precision=0.4021, F1=0.4959
   Fold  6: AUC=0.8129, Recall=0.6336, Precision=0.3838, F1=0.4780
   Fold  7: AUC=0.7946, Recall=0.6272, Precision=0.3544, F1=0.4529
   Fold  8: AUC=0.8035, Recall=0.6552, Precision=0.3819, F1=0.4825
   Fold  9: AUC=0.8001, Reca

In [105]:
# ============================================================
# CELL 13B: RANDOM FOREST — BAYESIAN OPTIMIZATION (BALANCED)
# ============================================================
print("\n" + "=" * 80)
print("🌲 EXPERIMENT 2: RANDOM FOREST — BALANCED")
print("=" * 80)
# ── Prepare data ──────────────────────────────────────────────
rf_X = df_engineered[RF_FEATURES].copy()
rf_y = df_engineered[TARGET_COL].copy()
print(f"\n📊 Dataset: {rf_X.shape[0]:,} samples, {rf_X.shape[1]} features (RF-optimized)")
print(f"   Using all engineered features — RF handles non-linearity naturally")
print(f"   Class distribution: {rf_y.value_counts().to_dict()}")

# ── Safety check ──────────────────────────────────────────────
print("\n🔍 Pre-modeling data quality check...")
rf_inf_mask = np.isinf(rf_X).any()
if rf_inf_mask.any():
    rf_X = rf_X.replace([np.inf, -np.inf], np.nan)
if rf_X.isnull().any().any():
    rf_nan_cols = rf_X.columns[rf_X.isnull().any()].tolist()
    print(f"   ⚠️  NaN values in: {rf_nan_cols} — imputing with median")
    for col in rf_nan_cols:
        rf_X[col] = rf_X[col].fillna(rf_X[col].median())
    print(f"   ✅ Data cleaned")
else:
    print(f"   ✅ No inf/NaN values detected")
assert not np.isinf(rf_X).any().any(), "Infinity values still present!"
assert not rf_X.isnull().any().any(), "NaN values still present!"

# ── Shared CV strategy ────────────────────────────────────────
RF_N_FOLDS = 5
RF_RANDOM_STATE = 42
rf_cv = StratifiedKFold(n_splits=RF_N_FOLDS, shuffle=True, random_state=RF_RANDOM_STATE)

# ==============================================================
# BAYESIAN OPTIMIZATION (Optuna) — 200 trials, BALANCED
# ==============================================================
print("\n" + "─" * 80)
print("🧠 Bayesian Optimization (Optuna) — 200 trials, 5-fold Stratified CV")
print("   ⚖️  class_weight = 'balanced' (fixed)")
print("─" * 80)

def rf_optuna_objective(trial: optuna.Trial) -> float:
    """
    Optuna objective for Random Forest — balanced class weight.
    """
    n_estimators = trial.suggest_int('n_estimators', 300, 1000, step=50)
    use_max_depth = trial.suggest_categorical('use_max_depth', [True, False])
    if use_max_depth:
        max_depth = trial.suggest_int('max_depth', 3, 20)
    else:
        max_depth = None

    min_samples_leaf = trial.suggest_int('min_samples_leaf', 2, 20)

    max_features_type = trial.suggest_categorical(
        'max_features_type', ['sqrt', 'log2', 'fraction']
    )
    if max_features_type == 'fraction':
        max_features = trial.suggest_float('max_features_fraction', 0.3, 1.0)
    else:
        max_features = max_features_type

    pipe = Pipeline([
        ('clf', RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            class_weight='balanced',          
            random_state=RF_RANDOM_STATE,
            n_jobs=-1,
        ))
    ])

    fold_scores = []
    for fold_idx, (train_idx, val_idx) in enumerate(rf_cv.split(rf_X, rf_y)):
        X_train, X_val = rf_X.iloc[train_idx], rf_X.iloc[val_idx]
        y_train, y_val = rf_y.iloc[train_idx], rf_y.iloc[val_idx]
        try:
            pipe.fit(X_train, y_train)
            y_proba = pipe.predict_proba(X_val)[:, 1]
            auc = roc_auc_score(y_val, y_proba)
        except Exception:
            return float('-inf')
        fold_scores.append(auc)
        trial.report(np.mean(fold_scores), fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(fold_scores)

rf_study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=RF_RANDOM_STATE),
    pruner=MedianPruner(n_startup_trials=15, n_warmup_steps=1),
    study_name='rf_bayesian_opt_balanced',
)

rf_t0 = time.perf_counter()
rf_study.optimize(rf_optuna_objective, n_trials=200, show_progress_bar=True)
rf_runtime = time.perf_counter() - rf_t0

# ── Extract results ───────────────────────────────────────────
rf_optuna_params = rf_study.best_params.copy()
rf_best_auc = rf_study.best_value
rf_n_complete = len([t for t in rf_study.trials
                     if t.state == optuna.trial.TrialState.COMPLETE])
rf_n_pruned = len([t for t in rf_study.trials
                   if t.state == optuna.trial.TrialState.PRUNED])

# ── Reconstruct best model for downstream use ────────────────
rf_best_max_depth = rf_optuna_params.get('max_depth', None) if rf_optuna_params.get('use_max_depth', False) else None
if rf_optuna_params['max_features_type'] == 'fraction':
    rf_best_max_features = rf_optuna_params['max_features_fraction']
else:
    rf_best_max_features = rf_optuna_params['max_features_type']

rf_best_pipe = Pipeline([
    ('clf', RandomForestClassifier(
        n_estimators=rf_optuna_params['n_estimators'],
        max_depth=rf_best_max_depth,
        min_samples_leaf=rf_optuna_params['min_samples_leaf'],
        max_features=rf_best_max_features,
        class_weight='balanced',              
        random_state=RF_RANDOM_STATE,
        n_jobs=-1,
    ))
])

# ── Verify with full CV to get per-fold scores ───────────────
rf_verify_scores = cross_val_score(
    rf_best_pipe, rf_X, rf_y, cv=rf_cv, scoring='roc_auc', n_jobs=-1
)
rf_verify_mean = np.mean(rf_verify_scores)
rf_verify_std = np.std(rf_verify_scores)

# ── Print Optuna summary ─────────────────────────────────────
print(f"\n   ✅ Completed in {rf_runtime:.1f}s")
print(f"   Trials completed: {rf_n_complete} | Pruned: {rf_n_pruned}")
print(f"   Best CV ROC-AUC:  {rf_best_auc:.6f}")
print(f"   Verified Mean±Std: {rf_verify_mean:.6f} ± {rf_verify_std:.6f}")
print(f"\n   Best hyperparameters:")
print(f"      {'n_estimators':>20s} = {rf_optuna_params['n_estimators']}")
print(f"      {'max_depth':>20s} = {rf_best_max_depth}")
print(f"      {'min_samples_leaf':>20s} = {rf_optuna_params['min_samples_leaf']}")
print(f"      {'max_features':>20s} = {rf_best_max_features}")
print(f"      {'class_weight':>20s} = balanced")
print(f"\n   Per-fold ROC-AUC: {['%.4f' % s for s in rf_verify_scores]}")

# ==============================================================
# 10-FOLD DETAILED EVALUATION
# ==============================================================
print("\n" + "─" * 80)
print("📊 10-Fold Cross-Validation Results (Final Evaluation)...")
print("─" * 80)

rf_cv_eval = StratifiedKFold(n_splits=10, shuffle=True, random_state=RF_RANDOM_STATE)
rf_auc_scores = []
rf_recall_scores = []
rf_precision_scores = []
rf_f1_scores = []

for fold, (train_idx, val_idx) in enumerate(rf_cv_eval.split(rf_X, rf_y), 1):
    X_train, X_val = rf_X.iloc[train_idx], rf_X.iloc[val_idx]
    y_train, y_val = rf_y.iloc[train_idx], rf_y.iloc[val_idx]

    rf_best_pipe.fit(X_train, y_train)
    y_pred_proba = rf_best_pipe.predict_proba(X_val)[:, 1]
    y_pred = rf_best_pipe.predict(X_val)

    auc = roc_auc_score(y_val, y_pred_proba)
    recall = recall_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)

    rf_auc_scores.append(auc)
    rf_recall_scores.append(recall)
    rf_precision_scores.append(precision)
    rf_f1_scores.append(f1)

    print(f"   Fold {fold:2d}: AUC={auc:.4f}, Recall={recall:.4f}, "
          f"Precision={precision:.4f}, F1={f1:.4f}")

# ── Summary statistics ────────────────────────────────────────
print("\n" + "-" * 80)
print("🌲 RANDOM FOREST SUMMARY — BALANCED")
print("-" * 80)
print(f"AUC:       {np.mean(rf_auc_scores):.4f} ± {np.std(rf_auc_scores):.4f}")
print(f"Recall:    {np.mean(rf_recall_scores):.4f} ± {np.std(rf_recall_scores):.4f}")
print(f"Precision: {np.mean(rf_precision_scores):.4f} ± {np.std(rf_precision_scores):.4f}")
print(f"F1:        {np.mean(rf_f1_scores):.4f} ± {np.std(rf_f1_scores):.4f}")

# ── Feature importance ────────────────────────────────────────
print("\n🏆 Top 15 Features by Importance:")
rf_best_pipe.fit(rf_X, rf_y)
rf_best_model = rf_best_pipe.named_steps['clf']
rf_feature_importance = pd.DataFrame({
    'feature': rf_X.columns,
    'importance': rf_best_model.feature_importances_
}).sort_values('importance', ascending=False)

for _, row in rf_feature_importance.head(15).iterrows():
    print(f"   {row['feature']:30s} {row['importance']:.4f}")

# ── Recall at 10% FPR ────────────────────────────────────────
print("\n📊 Recall at 10% FPR:")
rf_y_pred_proba = rf_best_pipe.predict_proba(rf_X)[:, 1]
rf_fpr, rf_tpr, rf_thresholds = roc_curve(rf_y, rf_y_pred_proba)
rf_idx_10fpr = np.argmin(np.abs(rf_fpr - 0.10))
rf_recall_at_10fpr = rf_tpr[rf_idx_10fpr]
print(f"   Recall @ 10% FPR: {rf_recall_at_10fpr:.4f}")

# ── Structured summary ───────────────────────────────────────
rf_cv_std = np.std(rf_auc_scores)
print("\n" + "=" * 80)
print("📊 RANDOM FOREST — TUNING SUMMARY (BALANCED)")
print("=" * 80)
print(f"""
┌────────────────────────────────────────────────────────────┐
│  Bayesian Optimization (Optuna) — Random Forest BALANCED   │
├────────────────────────────────────────────────────────────┤
│  Trials:           200 ({rf_n_complete} completed, {rf_n_pruned} pruned)       │
│  Runtime:          {rf_runtime:>7.1f}s                                  │
│  Best ROC-AUC:     {rf_best_auc:.6f}                                │
│  10-Fold Mean±Std: {np.mean(rf_auc_scores):.4f} ± {rf_cv_std:.4f}                          │
├────────────────────────────────────────────────────────────┤
│  n_estimators:     {rf_optuna_params['n_estimators']:<10}                              │
│  max_depth:        {str(rf_best_max_depth):<10}                              │
│  min_samples_leaf: {rf_optuna_params['min_samples_leaf']:<10}                              │
│  max_features:     {str(rf_best_max_features):<10}                              │
│  class_weight:     balanced                                │
└────────────────────────────────────────────────────────────┘
💡 Stability: CV std = {rf_cv_std:.4f} ({'Low variance — stable' if rf_cv_std < 0.015 else 'Moderate variance — consider more regularization' if rf_cv_std < 0.03 else 'High variance — overfitting risk'})
""")
print(f"   💾 rf_optuna_params saved: {rf_optuna_params}")
print("=" * 80)


🌲 EXPERIMENT 2: RANDOM FOREST — BALANCED

📊 Dataset: 41,188 samples, 29 features (RF-optimized)
   Using all engineered features — RF handles non-linearity naturally
   Class distribution: {0: 36548, 1: 4640}

🔍 Pre-modeling data quality check...
   ✅ No inf/NaN values detected

────────────────────────────────────────────────────────────────────────────────
🧠 Bayesian Optimization (Optuna) — 200 trials, 5-fold Stratified CV
   ⚖️  class_weight = 'balanced' (fixed)
────────────────────────────────────────────────────────────────────────────────


Best trial: 182. Best value: 0.793375: 100%|██████████| 200/200 [42:43<00:00, 12.82s/it]



   ✅ Completed in 2563.8s
   Trials completed: 121 | Pruned: 79
   Best CV ROC-AUC:  0.793375
   Verified Mean±Std: 0.793375 ± 0.005401

   Best hyperparameters:
              n_estimators = 550
                 max_depth = 8
          min_samples_leaf = 9
              max_features = 0.4725664486326693
              class_weight = balanced

   Per-fold ROC-AUC: ['0.7937', '0.7902', '0.8033', '0.7873', '0.7923']

────────────────────────────────────────────────────────────────────────────────
📊 10-Fold Cross-Validation Results (Final Evaluation)...
────────────────────────────────────────────────────────────────────────────────
   Fold  1: AUC=0.8053, Recall=0.6099, Precision=0.3748, F1=0.4643
   Fold  2: AUC=0.7824, Recall=0.5927, Precision=0.3599, F1=0.4479
   Fold  3: AUC=0.7976, Recall=0.6078, Precision=0.3701, F1=0.4600
   Fold  4: AUC=0.7814, Recall=0.5970, Precision=0.3538, F1=0.4443
   Fold  5: AUC=0.8013, Recall=0.6336, Precision=0.3925, F1=0.4847
   Fold  6: AUC=0.8004, Reca

In [71]:
# ============================================================
# CELL 13C: RF LIFT ANALYSIS — BINNING STRATEGY
# ============================================================
print("=" * 80)
print("📈 RF LIFT ANALYSIS — BINNING STRATEGY")
print("=" * 80)
print("   Using trained RF to identify natural breakpoints for feature binning")
print("=" * 80)

import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

FIGURES_DIR = r"C:\Users\phill\projects\ai-business-coach\prototype\glass_pipeline\research_logs\figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

# ============================================================
# 1. SETUP — use RF model + full feature set
# ============================================================
rf_best_pipe.fit(rf_X, rf_y)
_rf_model     = rf_best_pipe.named_steps['clf']
_overall_rate = rf_y.mean()

print(f"\n   Overall conversion rate : {_overall_rate:.4f} ({_overall_rate:.1%})")
print(f"   Features in analysis    : {len(rf_X.columns)}")

# ============================================================
# 2. GINI + PERMUTATION IMPORTANCE
# ============================================================
print(f"\n{'─'*80}")
print("📊 FEATURE IMPORTANCE")
print(f"{'─'*80}")

from sklearn.inspection import permutation_importance as _perm_imp

_split_idx = int(len(rf_X) * 0.8)
_perm_X = rf_X.iloc[_split_idx:]
_perm_y = rf_y.iloc[_split_idx:]

_gini_imp = pd.DataFrame({
    'feature': rf_X.columns,
    'gini':    _rf_model.feature_importances_
}).sort_values('gini', ascending=False).reset_index(drop=True)

print("\n   🔝 Gini Importance (Top 20):")
print(f"   {'Rank':<6} {'Feature':<40} {'Gini':>8}")
print(f"   {'-'*57}")
for i, row in _gini_imp.head(20).iterrows():
    bar = "█" * int(row['gini'] * 200)
    print(f"   {i+1:<6} {row['feature']:<40} {row['gini']:>8.4f}  {bar}")

print("\n   🔀 Computing Permutation Importance (this takes ~30s)...")
_perm_result = _perm_imp(
    rf_best_pipe, _perm_X, _perm_y,
    n_repeats=10, random_state=42, n_jobs=-1, scoring='roc_auc'
)
_perm_df = pd.DataFrame({
    'feature': rf_X.columns,
    'perm':    _perm_result.importances_mean,
    'std':     _perm_result.importances_std
}).sort_values('perm', ascending=False).reset_index(drop=True)

print(f"\n   {'Rank':<6} {'Feature':<40} {'Perm AUC Drop':>14} {'Std':>8}")
print(f"   {'-'*70}")
for i, row in _perm_df.head(20).iterrows():
    flag = "🔥" if row['perm'] > 0.01 else "⚪" if row['perm'] > 0 else "💀"
    print(f"   {i+1:<6} {row['feature']:<40} {row['perm']:>14.4f} {row['std']:>8.4f}  {flag}")

_combined = _gini_imp.merge(_perm_df, on='feature')
_combined['avg_rank'] = (
    _combined['gini'].rank(ascending=False) +
    _combined['perm'].rank(ascending=False)
) / 2
_combined = _combined.sort_values('avg_rank').reset_index(drop=True)

print(f"\n   🏆 Combined Ranking:")
print(f"   {'Rank':<6} {'Feature':<40} {'Gini':>8} {'Perm':>8} {'AvgRank':>8}")
print(f"   {'-'*75}")
for i, row in _combined.iterrows():
    print(f"   {i+1:<6} {row['feature']:<40} {row['gini']:>8.4f} {row['perm']:>8.4f} {row['avg_rank']:>8.1f}")

# ============================================================
# 3. LIFT BY DECILE — RF FEATURES
# ============================================================
CONTINUOUS_FEATURES = [f for f in [
    'campaign',
    'neighborhood_subscription_density',
    'cons.conf.idx',
    'economic_curvature_intensity',
    'joint_economic_decay',
    'dow_month_encoded',
    'behavioral_favorability',
    'cpi_high_cellular',
] if f in rf_X.columns]

CATEGORICAL_FEATURES = []  # no raw categoricals in RF feature set

print(f"\n{'─'*80}")
print("📈 CONTINUOUS FEATURE LIFT BY DECILE")
print(f"{'─'*80}")

_lift_results = {}

for feat in CONTINUOUS_FEATURES:
    _df_tmp = pd.DataFrame({'val': rf_X[feat], 'y': rf_y})
    n_unique = _df_tmp['val'].nunique()
    try:
        if n_unique < 10:
            _df_tmp['bin'] = _df_tmp['val']
        else:
            _df_tmp['bin'] = pd.qcut(_df_tmp['val'], q=10, duplicates='drop')
    except Exception:
        continue

    _lift_df = _df_tmp.groupby('bin').agg(
        count=('y', 'count'),
        conversions=('y', 'sum'),
        conv_rate=('y', 'mean')
    ).reset_index()
    _lift_df['lift']    = _lift_df['conv_rate'] / _overall_rate
    _lift_df['feature'] = feat
    _lift_results[feat] = _lift_df

    print(f"\n   🔹 {feat}  (range: [{rf_X[feat].min():.4f}, {rf_X[feat].max():.4f}])")
    print(f"   {'Bin':<35} {'Count':>7} {'ConvRate':>10} {'Lift':>7}")
    print(f"   {'-'*62}")
    for _, row in _lift_df.iterrows():
        flag = "🔥" if row['lift'] > 1.5 else "🟢" if row['lift'] > 1.0 else "🔴"
        print(f"   {str(row['bin']):<35} {row['count']:>7,} {row['conv_rate']:>10.4f} {row['lift']:>7.2f}x  {flag}")

# ============================================================
# 4. PLOT — IMPORTANCE
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle("RF Feature Importance — Binning Strategy\n(UCI Bank Marketing)",
             fontsize=13, fontweight='bold')

top_gini = _gini_imp.head(8)
axes[0].barh(top_gini['feature'], top_gini['gini'], color='forestgreen')
axes[0].set_xlabel('Gini Importance')
axes[0].set_title('Gini Importance', fontweight='bold')
axes[0].invert_yaxis()
axes[0].grid(axis='x', alpha=0.3)

top_perm = _perm_df.head(8)
axes[1].barh(top_perm['feature'], top_perm['perm'],
             xerr=top_perm['std'], color='steelblue')
axes[1].set_xlabel('Permutation Importance (AUC drop)')
axes[1].set_title('Permutation Importance', fontweight='bold')
axes[1].invert_yaxis()
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
_path = os.path.join(FIGURES_DIR, "rf_feature_importance.png")
plt.savefig(_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"\n   ✅ Importance plot saved → {_path}")

# ============================================================
# 5. PLOT — LIFT CURVES
# ============================================================
if _lift_results:
    _n_feats = len(_lift_results)
    _n_cols  = 3
    _n_rows  = (_n_feats + _n_cols - 1) // _n_cols

    fig, axes = plt.subplots(_n_rows, _n_cols, figsize=(18, _n_rows * 4))
    axes = axes.flatten()
    fig.suptitle("Conversion Rate by Decile — Lift Analysis\n(Green=High Lift >1.5x, Red=Low Lift <0.7x)",
                 fontsize=13, fontweight='bold')

    for idx, (feat, ldf) in enumerate(_lift_results.items()):
        ax = axes[idx]
        bars = ax.bar(range(len(ldf)), ldf['conv_rate'], color='steelblue', alpha=0.7)
        for bar, lift in zip(bars, ldf['lift']):
            if lift > 1.5:
                bar.set_color('darkgreen')
            elif lift < 0.7:
                bar.set_color('darkred')
        ax.axhline(y=_overall_rate, color='red', linestyle='--', linewidth=2,
                   label=f'Overall ({_overall_rate:.3f})')
        ax.set_xticks(range(len(ldf)))
        ax.set_xticklabels([str(b)[:15] for b in ldf['bin']], rotation=45, ha='right', fontsize=7)
        ax.set_ylabel('Conversion Rate')
        ax.set_title(f'{feat}\n(Max: {ldf["lift"].max():.2f}x, Min: {ldf["lift"].min():.2f}x)',
                     fontweight='bold', fontsize=9)
        ax.legend(fontsize=7)
        ax.grid(axis='y', alpha=0.3)

    for idx in range(len(_lift_results), len(axes)):
        axes[idx].axis('off')

    plt.tight_layout()
    _path = os.path.join(FIGURES_DIR, "rf_lift_curves.png")
    plt.savefig(_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"   ✅ Lift curves saved → {_path}")

# ============================================================
# 6. BINNING RECOMMENDATIONS
# ============================================================
print(f"\n{'='*80}")
print("🎯 BINNING RECOMMENDATIONS")
print(f"{'='*80}")
print(f"\n   Overall conversion rate: {_overall_rate:.4f}\n")

for feat, ldf in _lift_results.items():
    high_lift = ldf[ldf['lift'] > 1.5]
    low_lift  = ldf[ldf['lift'] < 0.7]
    print(f"   {feat}:")
    if len(high_lift):
        print(f"      🔥 HIGH LIFT bins (>1.5x): {list(high_lift['bin'].astype(str))}")
    if len(low_lift):
        print(f"      🔴 LOW LIFT bins  (<0.7x): {list(low_lift['bin'].astype(str))}")
    if not len(high_lift) and not len(low_lift):
        print(f"      ⚪ Flat lift — low binning value")

print(f"\n{'='*80}")
print("🎉 LIFT ANALYSIS COMPLETE — Review figures then define binning strategy")
print(f"{'='*80}")

📈 RF LIFT ANALYSIS — BINNING STRATEGY
   Using trained RF to identify natural breakpoints for feature binning

   Overall conversion rate : 0.1127 (11.3%)
   Features in analysis    : 8

────────────────────────────────────────────────────────────────────────────────
📊 FEATURE IMPORTANCE
────────────────────────────────────────────────────────────────────────────────

   🔝 Gini Importance (Top 20):
   Rank   Feature                                      Gini
   ---------------------------------------------------------
   1      neighborhood_subscription_density          0.2872  █████████████████████████████████████████████████████████
   2      joint_economic_decay                       0.2522  ██████████████████████████████████████████████████
   3      economic_curvature_intensity               0.1280  █████████████████████████
   4      dow_month_encoded                          0.1220  ████████████████████████
   5      behavioral_favorability                    0.0930  ████████████

In [76]:
# ============================================================
# CELL 13D: FEATURE BINNING — LIFT-DRIVEN BINARY ENCODING
# ============================================================
# Transforms 8 continuous RF features → 8 binary flags based on
# lift analysis breakpoints from Cell 13C.
# Goal: Force RF into combinatorial (2^8=256 pattern) space
#        so it can't smooth-interpolate same curves as LR/EBM.
# ============================================================

print("\n" + "=" * 80)
print("🔀 FEATURE BINNING — LIFT-DRIVEN BINARY ENCODING")
print("=" * 80)
print("   Converting continuous features → binary flags using lift breakpoints")
print("   Target: Create disagreement between models by eliminating shared")
print("           continuous feature space")

# ── Define binning thresholds from lift analysis ──────────────
BINNING_STRATEGY = {
    'nsd_high': {
        'source': 'neighborhood_subscription_density',
        'rule': 'threshold_above',
        'threshold': 0.23,
        'rationale': 'Top decile 4.08x lift; clean single breakpoint'
    },
    'jed_elevated': {
        'source': 'joint_economic_decay',
        'rule': 'threshold_above',
        'threshold': 0.04,
        'rationale': 'Deliberately lower than NSD to create asymmetric firing — '
                     'captures 1.43x zone that NSD misses'
    },
    'cci_favorable': {
        'source': 'cons.conf.idx',
        'rule': 'disjunction',
        'conditions': [(-41.8, -40.0), (-36.1, -26.9)],
        'rationale': 'Two non-contiguous hot zones at 4.11x and 3.78x; '
                     'dead valley between them'
    },
    'eci_high': {
        'source': 'economic_curvature_intensity',
        'rule': 'threshold_above',
        'threshold': 0.0948,
        'rationale': 'Top decile 3.53x lift; smaller pocket at 0.0777–0.0797 '
                     'too narrow/noisy to include'
    },
    'dow_high': {
        'source': 'dow_month_encoded',
        'rule': 'threshold_above',
        'threshold': 0.127,
        'rationale': 'Top decile 3.55x lift; monotonic ramp below'
    },
    'behav_high': {
        'source': 'behavioral_favorability',
        'rule': 'threshold_above',
        'threshold': 0.5,
        'rationale': 'Clear inflection at 0.5 — bins above are 1.87x and 3.97x'
    },
    'campaign_fresh': {
        'source': 'campaign',
        'rule': 'threshold_below_eq',
        'threshold': 2,
        'rationale': 'Only bin above baseline is (0.999, 2.0] at 1.10x; '
                     'everything above 2 contacts decays sharply'
    },
    'cpi_high_cellular': {
        'source': 'cpi_high_cellular',
        'rule': 'passthrough',
        'rationale': 'Already binary — 1.50x lift when =1'
    },
}

# ── Apply binning ─────────────────────────────────────────────
print("\n" + "─" * 80)
print("📐 Applying binning rules...")
print("─" * 80)

df_binned = df_engineered.copy()

for bin_name, config in BINNING_STRATEGY.items():
    source = config['source']
    rule = config['rule']

    if rule == 'threshold_above':
        df_binned[bin_name] = (df_binned[source] > config['threshold']).astype(int)

    elif rule == 'threshold_below_eq':
        df_binned[bin_name] = (df_binned[source] <= config['threshold']).astype(int)

    elif rule == 'disjunction':
        mask = pd.Series(False, index=df_binned.index)
        for lo, hi in config['conditions']:
            mask = mask | ((df_binned[source] > lo) & (df_binned[source] <= hi))
        df_binned[bin_name] = mask.astype(int)

    elif rule == 'passthrough':
        df_binned[bin_name] = df_binned[source].astype(int)

    else:
        raise ValueError(f"Unknown rule: {rule}")

    # Report
    n_hot = df_binned[bin_name].sum()
    pct_hot = n_hot / len(df_binned) * 100
    conv_hot = df_binned.loc[df_binned[bin_name] == 1, TARGET_COL].mean()
    conv_cold = df_binned.loc[df_binned[bin_name] == 0, TARGET_COL].mean()
    lift = conv_hot / conv_cold if conv_cold > 0 else float('inf')

    print(f"\n   ✅ {bin_name:25s} ← {source}")
    print(f"      Rule: {rule} | {config.get('threshold', config.get('conditions', 'N/A'))}")
    print(f"      Hot: {n_hot:>6,} ({pct_hot:5.1f}%) | "
          f"Conv hot: {conv_hot:.4f} vs cold: {conv_cold:.4f} | "
          f"Lift: {lift:.2f}x")

# ── Define new RF feature array ───────────────────────────────
RF_FEATURES_BINARY = [
    'nsd_high',
    'jed_elevated',
    'cci_favorable',
    'eci_high',
    'dow_high',
    'behav_high',
    'campaign_fresh',
    'cpi_high_cellular',
]

# ── Validate binary feature set ──────────────────────────────
print("\n" + "─" * 80)
print("🔍 BINARY FEATURE VALIDATION")
print("─" * 80)

rf_X_binary = df_binned[RF_FEATURES_BINARY].copy()
rf_y_binary = df_binned[TARGET_COL].copy()

print(f"\n   Shape: {rf_X_binary.shape}")
print(f"   All binary: {(rf_X_binary.isin([0, 1]).all()).all()}")
print(f"   Unique patterns: {rf_X_binary.drop_duplicates().shape[0]} / {2**len(RF_FEATURES_BINARY)} possible")

# ── Correlation matrix — check NSD vs JED asymmetry ──────────
print("\n   📊 Pairwise correlations (checking for redundancy):")
corr_matrix = rf_X_binary.corr()
high_corr_pairs = []
for i in range(len(RF_FEATURES_BINARY)):
    for j in range(i + 1, len(RF_FEATURES_BINARY)):
        c = corr_matrix.iloc[i, j]
        f1, f2 = RF_FEATURES_BINARY[i], RF_FEATURES_BINARY[j]
        marker = "⚠️ " if abs(c) > 0.7 else "   "
        if abs(c) > 0.3:
            print(f"      {marker}{f1:25s} × {f2:25s} = {c:+.4f}")
            if abs(c) > 0.7:
                high_corr_pairs.append((f1, f2, c))

if not high_corr_pairs:
    print("      ✅ No high-correlation pairs (>0.7) — good asymmetry!")
else:
    print(f"\n      ⚠️  {len(high_corr_pairs)} high-correlation pair(s) found — "
          f"may need threshold adjustment")

# ── Class-conditional activation rates ────────────────────────
print("\n   📊 Class-conditional activation rates:")
print(f"   {'Feature':25s} {'P(hot|y=0)':>12s} {'P(hot|y=1)':>12s} {'Ratio':>8s}")
print(f"   {'-'*25} {'-'*12} {'-'*12} {'-'*8}")
for feat in RF_FEATURES_BINARY:
    rate_neg = df_binned.loc[df_binned[TARGET_COL] == 0, feat].mean()
    rate_pos = df_binned.loc[df_binned[TARGET_COL] == 1, feat].mean()
    ratio = rate_pos / rate_neg if rate_neg > 0 else float('inf')
    marker = "🔥" if ratio > 3 else "🟢" if ratio > 1.5 else "  "
    print(f"   {feat:25s} {rate_neg:12.4f} {rate_pos:12.4f} {ratio:7.2f}x {marker}")

# ── Combinatorial pattern analysis ────────────────────────────
print("\n" + "─" * 80)
print("🧮 COMBINATORIAL PATTERN ANALYSIS")
print("─" * 80)

pattern_df = rf_X_binary.copy()
pattern_df['_target'] = rf_y_binary.values
pattern_df['_pattern'] = rf_X_binary.apply(
    lambda row: ''.join(str(int(v)) for v in row), axis=1
)

pattern_stats = pattern_df.groupby('_pattern').agg(
    count=('_target', 'size'),
    conv_rate=('_target', 'mean')
).sort_values('count', ascending=False)

n_patterns = len(pattern_stats)
n_hot_patterns = (pattern_stats['conv_rate'] > 0.113 * 1.5).sum()  # >1.5x lift
n_cold_patterns = (pattern_stats['conv_rate'] < 0.113 * 0.7).sum()  # <0.7x lift

print(f"   Observed patterns:  {n_patterns:>4} / {2**len(RF_FEATURES_BINARY)} possible")
print(f"   Hot patterns (>1.5x):  {n_hot_patterns}")
print(f"   Cold patterns (<0.7x): {n_cold_patterns}")
print(f"\n   Top 10 patterns by frequency:")
print(f"   {'Pattern':>12s} {'Count':>8s} {'ConvRate':>10s} {'Lift':>8s}")
print(f"   {'-'*12} {'-'*8} {'-'*10} {'-'*8}")
for pattern, row in pattern_stats.head(10).iterrows():
    lift = row['conv_rate'] / 0.1127
    marker = "🔥" if lift > 1.5 else "🔴" if lift < 0.7 else "  "
    print(f"   {pattern:>12s} {row['count']:>8,.0f} {row['conv_rate']:>10.4f} "
          f"{lift:>7.2f}x {marker}")

# ── Summary ───────────────────────────────────────────────────
print("\n" + "=" * 80)
print("✅ BINNING COMPLETE")
print("=" * 80)
print(f"""
   8 continuous features → 8 binary flags
   RF_FEATURES_BINARY = {RF_FEATURES_BINARY}

   Key design choices:
   • NSD threshold 0.23 (strict) vs JED threshold 0.04 (loose)
     → Asymmetric firing creates disagreement between correlated features
   • CCI uses disjunction for two non-contiguous hot zones
   • ECI uses high-only cut (0.0948) — ignoring narrow noisy pocket
   • Campaign inverted: fresh (≤2) = positive signal

   Next: Re-run Optuna on binary features (Cell 13E)
""")
print("=" * 80)


🔀 FEATURE BINNING — LIFT-DRIVEN BINARY ENCODING
   Converting continuous features → binary flags using lift breakpoints
   Target: Create disagreement between models by eliminating shared
           continuous feature space

────────────────────────────────────────────────────────────────────────────────
📐 Applying binning rules...
────────────────────────────────────────────────────────────────────────────────

   ✅ nsd_high                  ← neighborhood_subscription_density
      Rule: threshold_above | 0.23
      Hot:  4,242 ( 10.3%) | Conv hot: 0.4576 vs cold: 0.0731 | Lift: 6.26x

   ✅ jed_elevated              ← joint_economic_decay
      Rule: threshold_above | 0.04
      Hot:  9,128 ( 22.2%) | Conv hot: 0.2832 vs cold: 0.0641 | Lift: 4.42x

   ✅ cci_favorable             ← cons.conf.idx
      Rule: disjunction | [(-41.8, -40.0), (-36.1, -26.9)]
      Hot:  3,934 (  9.6%) | Conv hot: 0.4380 vs cold: 0.0783 | Lift: 5.59x

   ✅ eci_high                  ← economic_curvature_int

In [83]:
# ============================================================
# CELL 13D: FEATURE BINNING — LIFT-DRIVEN MULTI-BIN ENCODING
# ============================================================
# 8 RF features → multi-level binary bins from Cell 13C lift analysis.
# BRW-style: mutually exclusive bins per feature (cold/warm/hot).
# ============================================================

print("\n" + "=" * 80)
print("🔀 FEATURE BINNING — LIFT-DRIVEN MULTI-BIN ENCODING")
print("=" * 80)

df_binned = df_engineered.copy()

# ─────────────────────────────────────────────────────────────
# 1. neighborhood_subscription_density  [0.048, 0.350]
#    Dead ≤0.052 (0.27–0.49x), transition to 0.19 (~0.9x),
#    warm 0.19–0.23 (1.04–1.43x), hot >0.23 (4.08x)
# ─────────────────────────────────────────────────────────────
df_binned['nsd_cold']     = (df_binned['neighborhood_subscription_density'] <= 0.0524).astype('int8')
df_binned['nsd_warm']     = ((df_binned['neighborhood_subscription_density'] > 0.0524) &
                             (df_binned['neighborhood_subscription_density'] <= 0.192)).astype('int8')
df_binned['nsd_elevated'] = ((df_binned['neighborhood_subscription_density'] > 0.192) &
                              (df_binned['neighborhood_subscription_density'] <= 0.23)).astype('int8')
df_binned['nsd_hot']      = (df_binned['neighborhood_subscription_density'] > 0.23).astype('int8')

# ─────────────────────────────────────────────────────────────
# 2. joint_economic_decay  [0.0, 0.430]
#    Dead ≤0.00035 (0.27–0.49x), transition to 0.038,
#    warm 0.038–0.086 (1.04–1.43x), hot >0.086 (4.08x)
#    Cuts offset from NSD to break correlation.
# ─────────────────────────────────────────────────────────────
df_binned['jed_cold']       = (df_binned['joint_economic_decay'] <= 0.000352).astype('int8')
df_binned['jed_transition'] = ((df_binned['joint_economic_decay'] > 0.000352) &
                                (df_binned['joint_economic_decay'] <= 0.038)).astype('int8')
df_binned['jed_warm']       = ((df_binned['joint_economic_decay'] > 0.038) &
                                (df_binned['joint_economic_decay'] <= 0.0856)).astype('int8')
df_binned['jed_hot']        = (df_binned['joint_economic_decay'] > 0.0856).astype('int8')

# ─────────────────────────────────────────────────────────────
# 3. cons.conf.idx  [-50.8, -26.9]  NON-MONOTONIC
#    ≤-46.2: modest 1.24x | -46.2 to -41.8: dead (0.38–0.54x)
#    -41.8 to -40.0: HOT1 4.11x | -40.0 to -36.1: valley 0.46–0.64x
#    >-36.1: HOT2 3.78x
# ─────────────────────────────────────────────────────────────
df_binned['cci_low']      = (df_binned['cons.conf.idx'] <= -46.2).astype('int8')
df_binned['cci_dead_mid'] = ((df_binned['cons.conf.idx'] > -46.2) &
                              (df_binned['cons.conf.idx'] <= -41.8)).astype('int8')
df_binned['cci_hot1']     = ((df_binned['cons.conf.idx'] > -41.8) &
                              (df_binned['cons.conf.idx'] <= -40.0)).astype('int8')
df_binned['cci_valley']   = ((df_binned['cons.conf.idx'] > -40.0) &
                              (df_binned['cons.conf.idx'] <= -36.1)).astype('int8')
df_binned['cci_hot2']     = (df_binned['cons.conf.idx'] > -36.1).astype('int8')

# ─────────────────────────────────────────────────────────────
# 4. economic_curvature_intensity  [0.008, 0.153]
#    Dead ≤0.029 (0.44–0.52x), mid 0.029–0.076 (mixed),
#    warm 0.076–0.095 (1.49–1.67x), hot >0.095 (3.53x)
# ─────────────────────────────────────────────────────────────
df_binned['eci_cold'] = (df_binned['economic_curvature_intensity'] <= 0.0291).astype('int8')
df_binned['eci_mid']  = ((df_binned['economic_curvature_intensity'] > 0.0291) &
                          (df_binned['economic_curvature_intensity'] <= 0.0758)).astype('int8')
df_binned['eci_warm'] = ((df_binned['economic_curvature_intensity'] > 0.0758) &
                          (df_binned['economic_curvature_intensity'] <= 0.0948)).astype('int8')
df_binned['eci_hot']  = (df_binned['economic_curvature_intensity'] > 0.0948).astype('int8')

# ─────────────────────────────────────────────────────────────
# 5. dow_month_encoded  [0.055, 0.397]  MONOTONIC
#    Cold ≤0.065 (0.52–0.56x), low 0.065–0.090 (0.63–0.78x),
#    mid 0.090–0.127 (0.83–1.07x), hot >0.127 (3.55x)
# ─────────────────────────────────────────────────────────────
df_binned['dow_cold'] = (df_binned['dow_month_encoded'] <= 0.0653).astype('int8')
df_binned['dow_low']  = ((df_binned['dow_month_encoded'] > 0.0653) &
                          (df_binned['dow_month_encoded'] <= 0.0895)).astype('int8')
df_binned['dow_mid']  = ((df_binned['dow_month_encoded'] > 0.0895) &
                          (df_binned['dow_month_encoded'] <= 0.127)).astype('int8')
df_binned['dow_hot']  = (df_binned['dow_month_encoded'] > 0.127).astype('int8')

# ─────────────────────────────────────────────────────────────
# 6. behavioral_favorability  [0.0, 1.0]  STEP FUNCTION
#    Cold ≤0.3 (0.36–0.58x), baseline 0.3–0.4 (0.95x),
#    warm 0.4–0.5 (1.20x), hot >0.5 (1.87–3.97x)
# ─────────────────────────────────────────────────────────────
df_binned['behav_cold']     = (df_binned['behavioral_favorability'] <= 0.3).astype('int8')
df_binned['behav_baseline'] = ((df_binned['behavioral_favorability'] > 0.3) &
                                (df_binned['behavioral_favorability'] <= 0.4)).astype('int8')
df_binned['behav_warm']     = ((df_binned['behavioral_favorability'] > 0.4) &
                                (df_binned['behavioral_favorability'] <= 0.5)).astype('int8')
df_binned['behav_hot']      = (df_binned['behavioral_favorability'] > 0.5).astype('int8')

# ─────────────────────────────────────────────────────────────
# 7. campaign  [1, 56]
#    Fresh ≤2 (1.10x), moderate 3–5 (0.77–0.95x), heavy >5 (0.49x)
# ─────────────────────────────────────────────────────────────
df_binned['campaign_fresh']    = (df_binned['campaign'] <= 2).astype('int8')
df_binned['campaign_moderate'] = ((df_binned['campaign'] > 2) &
                                   (df_binned['campaign'] <= 5)).astype('int8')
df_binned['campaign_heavy']    = (df_binned['campaign'] > 5).astype('int8')

# ─────────────────────────────────────────────────────────────
# 8. cpi_high_cellular — already binary, passthrough
# ─────────────────────────────────────────────────────────────
df_binned['cpi_cellular'] = df_binned['cpi_high_cellular'].astype('int8')

# ── Reassign RF_FEATURES ──────────────────────────────────────
RF_FEATURES_BINARY = [
    'nsd_cold', 'nsd_warm', 'nsd_elevated', 'nsd_hot',
    'jed_cold', 'jed_transition', 'jed_warm', 'jed_hot',
    'cci_low', 'cci_dead_mid', 'cci_hot1', 'cci_valley', 'cci_hot2',
    'eci_cold', 'eci_mid', 'eci_warm', 'eci_hot',
    'dow_cold', 'dow_low', 'dow_mid', 'dow_hot',
    'behav_cold', 'behav_baseline', 'behav_warm', 'behav_hot',
    'campaign_fresh', 'campaign_moderate', 'campaign_heavy',
    'cpi_cellular',
]

# ── Mutual exclusivity check ─────────────────────────────────
feature_groups = {
    'NSD':      ['nsd_cold', 'nsd_warm', 'nsd_elevated', 'nsd_hot'],
    'JED':      ['jed_cold', 'jed_transition', 'jed_warm', 'jed_hot'],
    'CCI':      ['cci_low', 'cci_dead_mid', 'cci_hot1', 'cci_valley', 'cci_hot2'],
    'ECI':      ['eci_cold', 'eci_mid', 'eci_warm', 'eci_hot'],
    'DOW':      ['dow_cold', 'dow_low', 'dow_mid', 'dow_hot'],
    'BEHAV':    ['behav_cold', 'behav_baseline', 'behav_warm', 'behav_hot'],
    'CAMPAIGN': ['campaign_fresh', 'campaign_moderate', 'campaign_heavy'],
}

print(f"\n   8 features → {len(RF_FEATURES_BINARY)} binary bins")
print(f"\n   Mutual exclusivity:")
for grp, cols in feature_groups.items():
    sums = df_binned[cols].sum(axis=1)
    ok = "✅" if (sums == 1).all() else "⚠️"
    print(f"      {ok} {grp:10s}: {len(cols)} bins, range=[{sums.min()}, {sums.max()}]")

# ── Quick lift sanity ─────────────────────────────────────────
overall_rate = df_binned[TARGET_COL].mean()
print(f"\n   Per-bin lift (base rate: {overall_rate:.4f}):")
print(f"   {'Bin':25s} {'N':>7s} {'Conv':>8s} {'Lift':>7s}")
print(f"   {'-'*25} {'-'*7} {'-'*8} {'-'*7}")
for feat in RF_FEATURES_BINARY:
    n = df_binned[feat].sum()
    conv = df_binned.loc[df_binned[feat] == 1, TARGET_COL].mean() if n > 0 else 0
    lift = conv / overall_rate
    tag = "🔥" if lift > 1.5 else "🔴" if lift < 0.7 else "  "
    print(f"   {feat:25s} {n:>7,} {conv:>8.4f} {lift:>6.2f}x {tag}")

print(f"\n✅ RF_FEATURES_BINARY = {len(RF_FEATURES_BINARY)} features ready")
print("=" * 80)


🔀 FEATURE BINNING — LIFT-DRIVEN MULTI-BIN ENCODING

   8 features → 29 binary bins

   Mutual exclusivity:
      ✅ NSD       : 4 bins, range=[1, 1]
      ✅ JED       : 4 bins, range=[1, 1]
      ✅ CCI       : 5 bins, range=[1, 1]
      ✅ ECI       : 4 bins, range=[1, 1]
      ✅ DOW       : 4 bins, range=[1, 1]
      ✅ BEHAV     : 4 bins, range=[1, 1]
      ✅ CAMPAIGN  : 3 bins, range=[1, 1]

   Per-bin lift (base rate: 0.1127):
   Bin                             N     Conv    Lift
   ------------------------- ------- -------- -------
   nsd_cold                   24,122   0.0485   0.43x 🔴
   nsd_warm                    8,430   0.1078   0.96x   
   nsd_elevated                4,394   0.1411   1.25x   
   nsd_hot                     4,242   0.4576   4.06x 🔥
   jed_cold                   24,122   0.0485   0.43x 🔴
   jed_transition              4,406   0.0885   0.79x   
   jed_warm                    8,547   0.1392   1.24x   
   jed_hot                     4,113   0.4595   4.08x 🔥
   cci_

In [84]:
# ============================================================
# CELL 13E: RANDOM FOREST — BAYESIAN OPTIMIZATION (BINARY FEATURES)
# ============================================================
# Re-tunes RF on the 8 binary features from Cell 13D.
# Key differences from Cell 13B:
#   - Feature space is now 2^8 = 256 combinatorial patterns
#   - Shallower trees expected (binary splits are trivial)
#   - max_depth range tightened since features are pre-discretized
#   - max_features search adjusted for 8 binary columns
# ============================================================

print("\n" + "=" * 80)
print("🌲 RANDOM FOREST — BAYESIAN OPTIMIZATION (BINARY FEATURES)")
print("=" * 80)

# ── Prepare data ──────────────────────────────────────────────
rf_X_bin = df_binned[RF_FEATURES_BINARY].copy()
rf_y_bin = df_binned[TARGET_COL].copy()

print(f"\n📊 Dataset: {rf_X_bin.shape[0]:,} samples, {rf_X_bin.shape[1]} binary features")
print(f"   Feature space: 2^{rf_X_bin.shape[1]} = {2**rf_X_bin.shape[1]} possible patterns")
print(f"   Observed patterns: {rf_X_bin.drop_duplicates().shape[0]}")
print(f"   Class distribution: {rf_y_bin.value_counts().to_dict()}")

# ── Safety check ──────────────────────────────────────────────
print("\n🔍 Pre-modeling data quality check...")
assert rf_X_bin.isin([0, 1]).all().all(), "Non-binary values detected!"
assert not rf_X_bin.isnull().any().any(), "NaN values detected!"
print(f"   ✅ All values binary, no missing data")

# ── Shared CV strategy ────────────────────────────────────────
RF_BIN_N_FOLDS = 5
RF_BIN_RANDOM_STATE = 42
rf_bin_cv = StratifiedKFold(n_splits=RF_BIN_N_FOLDS, shuffle=True,
                            random_state=RF_BIN_RANDOM_STATE)

# ==============================================================
# BAYESIAN OPTIMIZATION (Optuna) — 200 trials, BINARY FEATURES
# ==============================================================
print("\n" + "─" * 80)
print("🧠 Bayesian Optimization (Optuna) — 200 trials, 5-fold Stratified CV")
print("   ⚖️  class_weight = 'balanced' (fixed)")
print("   📐 Binary feature space — adjusted search ranges")
print("─" * 80)


def rf_binary_optuna_objective(trial: optuna.Trial) -> float:
    """
    Optuna objective for Random Forest on binary features.
    Adjusted ranges:
      - max_depth capped lower (binary splits need fewer levels)
      - min_samples_leaf can be higher (coarser patterns)
      - max_features explores more of the 8-feature space
    """
    n_estimators = trial.suggest_int('n_estimators', 200, 1200, step=50)

    # Binary features need shallower trees — 8 features means
    # at most 8 meaningful splits, but interactions need depth
    use_max_depth = trial.suggest_categorical('use_max_depth', [True, False])
    if use_max_depth:
        max_depth = trial.suggest_int('max_depth', 2, 12)
    else:
        max_depth = None

    min_samples_leaf = trial.suggest_int('min_samples_leaf', 5, 50)

    # With only 8 binary features, test specific counts too
    max_features_type = trial.suggest_categorical(
        'max_features_type', ['sqrt', 'log2', 'fraction']
    )
    if max_features_type == 'fraction':
        max_features = trial.suggest_float('max_features_fraction', 0.3, 1.0)
    else:
        max_features = max_features_type

    # Minority weight — let Optuna find the right balance for binary space
    minority_weight = trial.suggest_float('minority_weight', 1.0, 15.0)
    sample_weight_map = {0: 1.0, 1: minority_weight}

    pipe = Pipeline([
        ('clf', RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            class_weight='balanced',
            random_state=RF_BIN_RANDOM_STATE,
            n_jobs=-1,
        ))
    ])

    fold_scores = []
    for fold_idx, (train_idx, val_idx) in enumerate(rf_bin_cv.split(rf_X_bin, rf_y_bin)):
        X_train, X_val = rf_X_bin.iloc[train_idx], rf_X_bin.iloc[val_idx]
        y_train, y_val = rf_y_bin.iloc[train_idx], rf_y_bin.iloc[val_idx]

        # Apply sample weights on top of class_weight
        sw = y_train.map(sample_weight_map).values

        try:
            pipe.fit(X_train, y_train, clf__sample_weight=sw)
            y_proba = pipe.predict_proba(X_val)[:, 1]
            auc = roc_auc_score(y_val, y_proba)
        except Exception:
            return float('-inf')

        fold_scores.append(auc)
        trial.report(np.mean(fold_scores), fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(fold_scores)


rf_bin_study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=RF_BIN_RANDOM_STATE),
    pruner=MedianPruner(n_startup_trials=15, n_warmup_steps=1),
    study_name='rf_binary_features_opt',
)

rf_bin_t0 = time.perf_counter()
rf_bin_study.optimize(rf_binary_optuna_objective, n_trials=200, show_progress_bar=True)
rf_bin_runtime = time.perf_counter() - rf_bin_t0

# ── Extract results ───────────────────────────────────────────
rf_bin_optuna_params = rf_bin_study.best_params.copy()
rf_bin_best_auc = rf_bin_study.best_value
rf_bin_n_complete = len([t for t in rf_bin_study.trials
                         if t.state == optuna.trial.TrialState.COMPLETE])
rf_bin_n_pruned = len([t for t in rf_bin_study.trials
                       if t.state == optuna.trial.TrialState.PRUNED])

# ── Reconstruct best model ───────────────────────────────────
rf_bin_best_max_depth = (rf_bin_optuna_params.get('max_depth', None)
                         if rf_bin_optuna_params.get('use_max_depth', False)
                         else None)

if rf_bin_optuna_params['max_features_type'] == 'fraction':
    rf_bin_best_max_features = rf_bin_optuna_params['max_features_fraction']
else:
    rf_bin_best_max_features = rf_bin_optuna_params['max_features_type']

rf_bin_best_minority_weight = rf_bin_optuna_params.get('minority_weight', 1.0)

rf_bin_best_pipe = Pipeline([
    ('clf', RandomForestClassifier(
        n_estimators=rf_bin_optuna_params['n_estimators'],
        max_depth=rf_bin_best_max_depth,
        min_samples_leaf=rf_bin_optuna_params['min_samples_leaf'],
        max_features=rf_bin_best_max_features,
        class_weight='balanced',
        random_state=RF_BIN_RANDOM_STATE,
        n_jobs=-1,
    ))
])

# ── Verify with full CV ──────────────────────────────────────
rf_bin_verify_scores = cross_val_score(
    rf_bin_best_pipe, rf_X_bin, rf_y_bin, cv=rf_bin_cv,
    scoring='roc_auc', n_jobs=-1
)
rf_bin_verify_mean = np.mean(rf_bin_verify_scores)
rf_bin_verify_std = np.std(rf_bin_verify_scores)

# ── Print Optuna summary ─────────────────────────────────────
print(f"\n   ✅ Completed in {rf_bin_runtime:.1f}s")
print(f"   Trials completed: {rf_bin_n_complete} | Pruned: {rf_bin_n_pruned}")
print(f"   Best CV ROC-AUC:  {rf_bin_best_auc:.6f}")
print(f"   Verified Mean±Std: {rf_bin_verify_mean:.6f} ± {rf_bin_verify_std:.6f}")

print(f"\n   Best hyperparameters:")
print(f"      {'n_estimators':>20s} = {rf_bin_optuna_params['n_estimators']}")
print(f"      {'max_depth':>20s} = {rf_bin_best_max_depth}")
print(f"      {'min_samples_leaf':>20s} = {rf_bin_optuna_params['min_samples_leaf']}")
print(f"      {'max_features':>20s} = {rf_bin_best_max_features}")
print(f"      {'minority_weight':>20s} = {rf_bin_best_minority_weight:.4f}")
print(f"      {'class_weight':>20s} = balanced")

print(f"\n   Per-fold ROC-AUC: {['%.4f' % s for s in rf_bin_verify_scores]}")

# ==============================================================
# 10-FOLD DETAILED EVALUATION
# ==============================================================
print("\n" + "─" * 80)
print("📊 10-Fold Cross-Validation Results (Binary Features)...")
print("─" * 80)

rf_bin_cv_eval = StratifiedKFold(n_splits=10, shuffle=True,
                                  random_state=RF_BIN_RANDOM_STATE)

rf_bin_auc_scores = []
rf_bin_recall_scores = []
rf_bin_precision_scores = []
rf_bin_f1_scores = []

for fold, (train_idx, val_idx) in enumerate(rf_bin_cv_eval.split(rf_X_bin, rf_y_bin), 1):
    X_train, X_val = rf_X_bin.iloc[train_idx], rf_X_bin.iloc[val_idx]
    y_train, y_val = rf_y_bin.iloc[train_idx], rf_y_bin.iloc[val_idx]

    sw = y_train.map({0: 1.0, 1: rf_bin_best_minority_weight}).values
    rf_bin_best_pipe.fit(X_train, y_train, clf__sample_weight=sw)

    y_pred_proba = rf_bin_best_pipe.predict_proba(X_val)[:, 1]
    y_pred = rf_bin_best_pipe.predict(X_val)

    auc = roc_auc_score(y_val, y_pred_proba)
    recall = recall_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)

    rf_bin_auc_scores.append(auc)
    rf_bin_recall_scores.append(recall)
    rf_bin_precision_scores.append(precision)
    rf_bin_f1_scores.append(f1)

    print(f"   Fold {fold:2d}: AUC={auc:.4f}, Recall={recall:.4f}, "
          f"Precision={precision:.4f}, F1={f1:.4f}")

# ── Feature importance (binary) ───────────────────────────────
print("\n🏆 Binary Feature Importance:")
rf_bin_best_pipe.fit(rf_X_bin, rf_y_bin)
rf_bin_model = rf_bin_best_pipe.named_steps['clf']
rf_bin_importance = pd.DataFrame({
    'feature': RF_FEATURES_BINARY,
    'importance': rf_bin_model.feature_importances_
}).sort_values('importance', ascending=False)

for _, row in rf_bin_importance.iterrows():
    bar = '█' * int(row['importance'] * 80)
    print(f"   {row['feature']:25s} {row['importance']:.4f}  {bar}")

# ── Recall at 10% FPR ────────────────────────────────────────
print("\n📊 Recall at 10% FPR:")
rf_bin_y_proba = rf_bin_best_pipe.predict_proba(rf_X_bin)[:, 1]
rf_bin_fpr, rf_bin_tpr, _ = roc_curve(rf_y_bin, rf_bin_y_proba)
rf_bin_idx_10fpr = np.argmin(np.abs(rf_bin_fpr - 0.10))
rf_bin_recall_at_10fpr = rf_bin_tpr[rf_bin_idx_10fpr]
print(f"   Recall @ 10% FPR: {rf_bin_recall_at_10fpr:.4f}")

# ── Comparison with continuous features ───────────────────────
print("\n" + "─" * 80)
print("⚖️  BINARY vs CONTINUOUS COMPARISON")
print("─" * 80)

# Check if continuous results exist from Cell 13B
try:
    cont_auc = np.mean(rf_auc_scores)
    cont_recall = np.mean(rf_recall_scores)
    cont_prec = np.mean(rf_precision_scores)
    cont_f1 = np.mean(rf_f1_scores)

    bin_auc = np.mean(rf_bin_auc_scores)
    bin_recall = np.mean(rf_bin_recall_scores)
    bin_prec = np.mean(rf_bin_precision_scores)
    bin_f1 = np.mean(rf_bin_f1_scores)

    print(f"   {'Metric':>12s} {'Continuous':>12s} {'Binary':>12s} {'Delta':>10s}")
    print(f"   {'-'*12} {'-'*12} {'-'*12} {'-'*10}")
    for name, c_val, b_val in [
        ('AUC', cont_auc, bin_auc),
        ('Recall', cont_recall, bin_recall),
        ('Precision', cont_prec, bin_prec),
        ('F1', cont_f1, bin_f1),
    ]:
        delta = b_val - c_val
        arrow = "↑" if delta > 0 else "↓" if delta < 0 else "="
        print(f"   {name:>12s} {c_val:>12.4f} {b_val:>12.4f} {delta:>+9.4f} {arrow}")

    if bin_auc < cont_auc:
        auc_drop = (cont_auc - bin_auc) / cont_auc * 100
        print(f"\n   📉 AUC dropped {auc_drop:.1f}% — expected trade-off for "
              f"disagreement gain")
        if auc_drop > 10:
            print(f"   ⚠️  >10% AUC drop — consider adjusting thresholds")
        else:
            print(f"   ✅ Acceptable drop — binary RF should create healthier "
                  f"disagreement topology")
    else:
        print(f"\n   📈 AUC maintained or improved — binary encoding captured "
              f"the right structure")
except NameError:
    print("   ℹ️  Continuous RF results not available — run Cell 13B first for comparison")

# ── Summary ───────────────────────────────────────────────────
rf_bin_cv_std = np.std(rf_bin_auc_scores)

print("\n" + "=" * 80)
print("📊 RANDOM FOREST — TUNING SUMMARY (BINARY FEATURES)")
print("=" * 80)
print(f"""
┌────────────────────────────────────────────────────────────┐
│  Bayesian Optimization — RF BINARY FEATURES                │
├────────────────────────────────────────────────────────────┤
│  Trials:           200 ({rf_bin_n_complete} completed, {rf_bin_n_pruned} pruned)       │
│  Runtime:          {rf_bin_runtime:>7.1f}s                                  │
│  Best ROC-AUC:     {rf_bin_best_auc:.6f}                                │
│  10-Fold Mean±Std: {np.mean(rf_bin_auc_scores):.4f} ± {rf_bin_cv_std:.4f}                          │
├────────────────────────────────────────────────────────────┤
│  n_estimators:     {rf_bin_optuna_params['n_estimators']:<10}                              │
│  max_depth:        {str(rf_bin_best_max_depth):<10}                              │
│  min_samples_leaf: {str(rf_bin_optuna_params['min_samples_leaf']):<10}                              │
│  max_features:     {str(rf_bin_best_max_features):<10}                              │
│  minority_weight:  {rf_bin_best_minority_weight:<10.4f}                              │
│  class_weight:     balanced                                │
│  Feature space:    8 binary → {2**8} patterns                  │
└────────────────────────────────────────────────────────────┘
💡 Stability: CV std = {rf_bin_cv_std:.4f} ({'Low variance — stable' if rf_bin_cv_std < 0.015 else 'Moderate variance' if rf_bin_cv_std < 0.03 else 'High variance — overfitting risk'})
""")

# ── Save params ───────────────────────────────────────────────
rf_bin_optuna_params_final = {
    'n_estimators': rf_bin_optuna_params['n_estimators'],
    'max_depth': rf_bin_best_max_depth,
    'min_samples_leaf': rf_bin_optuna_params['min_samples_leaf'],
    'max_features': rf_bin_best_max_features,
    'minority_weight': rf_bin_best_minority_weight,
    'class_weight': 'balanced',
    'features': RF_FEATURES_BINARY,
}

print(f"   💾 rf_bin_optuna_params_final saved: {rf_bin_optuna_params_final}")
print(f"\n   Next: Run Venn trace on binary RF vs LR vs EBM to check")
print(f"         agreement drop toward 65-75% target (Cell 13F)")
print("=" * 80)


🌲 RANDOM FOREST — BAYESIAN OPTIMIZATION (BINARY FEATURES)

📊 Dataset: 41,188 samples, 29 binary features
   Feature space: 2^29 = 536870912 possible patterns
   Observed patterns: 479
   Class distribution: {0: 36548, 1: 4640}

🔍 Pre-modeling data quality check...
   ✅ All values binary, no missing data

────────────────────────────────────────────────────────────────────────────────
🧠 Bayesian Optimization (Optuna) — 200 trials, 5-fold Stratified CV
   ⚖️  class_weight = 'balanced' (fixed)
   📐 Binary feature space — adjusted search ranges
────────────────────────────────────────────────────────────────────────────────


Best trial: 118. Best value: 0.793071: 100%|██████████| 200/200 [24:22<00:00,  7.31s/it]



   ✅ Completed in 1462.7s
   Trials completed: 138 | Pruned: 62
   Best CV ROC-AUC:  0.793071
   Verified Mean±Std: 0.792979 ± 0.005879

   Best hyperparameters:
              n_estimators = 200
                 max_depth = 9
          min_samples_leaf = 5
              max_features = sqrt
           minority_weight = 1.0234
              class_weight = balanced

   Per-fold ROC-AUC: ['0.7937', '0.7889', '0.8033', '0.7860', '0.7929']

────────────────────────────────────────────────────────────────────────────────
📊 10-Fold Cross-Validation Results (Binary Features)...
────────────────────────────────────────────────────────────────────────────────
   Fold  1: AUC=0.8044, Recall=0.6099, Precision=0.3642, F1=0.4561
   Fold  2: AUC=0.7822, Recall=0.5948, Precision=0.3494, F1=0.4402
   Fold  3: AUC=0.7973, Recall=0.6207, Precision=0.3504, F1=0.4479
   Fold  4: AUC=0.7795, Recall=0.5970, Precision=0.3411, F1=0.4342
   Fold  5: AUC=0.8022, Recall=0.6422, Precision=0.3850, F1=0.4814
   Fold

In [85]:
# ============================================================
# CELL 13E-RECALL: RF — RECALL-BIASED OPTUNA (29 BINARY FEATURES)
# ============================================================
# Same 29 binary features from Cell 13D, but optimizing for recall.
# Key differences from balanced 13E:
#   - Objective: 0.6 × Recall + 0.4 × AUC (recall-dominant)
#   - Recall measured at a 70%-target threshold from train set
#   - Pruning after fold 3 (n_warmup_steps=2) — recall is noisier
#   - Wider minority_weight range (1.0–20.0) to explore aggressive bias
#   - Comparison: recall-biased vs balanced at end
# ============================================================
print("\n" + "=" * 80)
print("🌲 RANDOM FOREST — RECALL-BIASED OPTUNA (29 BINARY FEATURES)")
print("=" * 80)

# ── Prepare data ──────────────────────────────────────────────
rf_X_bin = df_binned[RF_FEATURES_BINARY].copy()
rf_y_bin = df_binned[TARGET_COL].copy()

print(f"\n📊 Dataset: {rf_X_bin.shape[0]:,} samples, {rf_X_bin.shape[1]} binary features")
print(f"   Feature space: 2^{rf_X_bin.shape[1]} = {2**rf_X_bin.shape[1]:,} possible patterns")
print(f"   Observed patterns: {rf_X_bin.drop_duplicates().shape[0]}")
print(f"   Class distribution: {rf_y_bin.value_counts().to_dict()}")

# ── Safety check ──────────────────────────────────────────────
print("\n🔍 Pre-modeling data quality check...")
assert rf_X_bin.isin([0, 1]).all().all(), "Non-binary values detected!"
assert not rf_X_bin.isnull().any().any(), "NaN values detected!"
print(f"   ✅ All values binary, no missing data")

# ── Shared CV strategy ────────────────────────────────────────
RF_BIN_N_FOLDS = 5
RF_BIN_RANDOM_STATE = 42
rf_bin_cv = StratifiedKFold(n_splits=RF_BIN_N_FOLDS, shuffle=True,
                            random_state=RF_BIN_RANDOM_STATE)

# ==============================================================
# RECALL-BIASED BAYESIAN OPTIMIZATION
# ==============================================================
print("\n" + "─" * 80)
print("🧠 Bayesian Optimization (Optuna) — 200 trials, 5-fold Stratified CV")
print("   🎯 RECALL-BIASED: objective = 0.6 × Recall + 0.4 × AUC")
print("   📐 29 binary features — multi-bin encoding")
print("   ✂️  Pruning after fold 3 (n_warmup_steps=2)")
print("─" * 80)


def rf_bin_recall_objective(trial: optuna.Trial) -> float:
    """
    Recall-biased objective for RF on 29 binary features.

    Scoring: 0.6 × recall + 0.4 × AUC
      - Recall computed at a threshold targeting ~70% recall on train,
        then measured on validation fold
      - AUC keeps ranking quality in check so we don't just predict all 1s
    """
    n_estimators = trial.suggest_int('n_estimators', 200, 1200, step=50)

    use_max_depth = trial.suggest_categorical('use_max_depth', [True, False])
    if use_max_depth:
        max_depth = trial.suggest_int('max_depth', 2, 15)
    else:
        max_depth = None

    min_samples_leaf = trial.suggest_int('min_samples_leaf', 5, 50)

    max_features_type = trial.suggest_categorical(
        'max_features_type', ['sqrt', 'log2', 'fraction']
    )
    if max_features_type == 'fraction':
        max_features = trial.suggest_float('max_features_fraction', 0.3, 1.0)
    else:
        max_features = max_features_type

    # Wider minority weight range — push recall harder
    minority_weight = trial.suggest_float('minority_weight', 1.0, 20.0)
    sample_weight_map = {0: 1.0, 1: minority_weight}

    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        class_weight='balanced',
        random_state=RF_BIN_RANDOM_STATE,
        n_jobs=-1,
    )

    fold_scores = []
    for fold_idx, (train_idx, val_idx) in enumerate(rf_bin_cv.split(rf_X_bin, rf_y_bin)):
        X_train, X_val = rf_X_bin.iloc[train_idx], rf_X_bin.iloc[val_idx]
        y_train, y_val = rf_y_bin.iloc[train_idx], rf_y_bin.iloc[val_idx]

        sw = y_train.map(sample_weight_map).values

        try:
            clf.fit(X_train, y_train, sample_weight=sw)
            y_proba = clf.predict_proba(X_val)[:, 1]

            # AUC for ranking quality
            auc = roc_auc_score(y_val, y_proba)

            # Find recall-targeted threshold on train set
            train_proba = clf.predict_proba(X_train)[:, 1]
            precs_tr, recs_tr, threshs_tr = precision_recall_curve(y_train, train_proba)
            target_idx = np.argmin(np.abs(recs_tr[:-1] - 0.70))
            thresh = threshs_tr[target_idx]

            # Apply threshold on validation
            y_pred = (y_proba >= thresh).astype(int)
            rec = recall_score(y_val, y_pred, zero_division=0)

            # Weighted objective: recall-dominant
            score = 0.6 * rec + 0.4 * auc

        except Exception:
            return float('-inf')

        fold_scores.append(score)
        trial.report(np.mean(fold_scores), fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(fold_scores)


rf_bin_recall_study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=RF_BIN_RANDOM_STATE),
    pruner=MedianPruner(n_startup_trials=15, n_warmup_steps=2),  # prune after fold 3
    study_name='rf_binary_recall_opt',
)

rf_bin_recall_t0 = time.perf_counter()
rf_bin_recall_study.optimize(rf_bin_recall_objective, n_trials=200, show_progress_bar=True)
rf_bin_recall_runtime = time.perf_counter() - rf_bin_recall_t0

# ── Extract results ───────────────────────────────────────────
rf_bin_recall_params = rf_bin_recall_study.best_params.copy()
rf_bin_recall_best_score = rf_bin_recall_study.best_value
rf_bin_recall_n_complete = len([t for t in rf_bin_recall_study.trials
                                if t.state == optuna.trial.TrialState.COMPLETE])
rf_bin_recall_n_pruned = len([t for t in rf_bin_recall_study.trials
                              if t.state == optuna.trial.TrialState.PRUNED])

# ── Reconstruct best model params ─────────────────────────────
rf_bin_recall_max_depth = (rf_bin_recall_params.get('max_depth', None)
                           if rf_bin_recall_params.get('use_max_depth', False)
                           else None)

if rf_bin_recall_params['max_features_type'] == 'fraction':
    rf_bin_recall_max_features = rf_bin_recall_params['max_features_fraction']
else:
    rf_bin_recall_max_features = rf_bin_recall_params['max_features_type']

rf_bin_recall_minority_weight = rf_bin_recall_params.get('minority_weight', 1.0)

print(f"\n   ✅ Completed in {rf_bin_recall_runtime:.1f}s")
print(f"   Trials completed: {rf_bin_recall_n_complete} | Pruned: {rf_bin_recall_n_pruned}")
print(f"   Best CV Score (0.6×Rec + 0.4×AUC): {rf_bin_recall_best_score:.6f}")
print(f"\n   Best hyperparameters:")
print(f"      {'n_estimators':>20s} = {rf_bin_recall_params['n_estimators']}")
print(f"      {'max_depth':>20s} = {rf_bin_recall_max_depth}")
print(f"      {'min_samples_leaf':>20s} = {rf_bin_recall_params['min_samples_leaf']}")
print(f"      {'max_features':>20s} = {rf_bin_recall_max_features}")
print(f"      {'minority_weight':>20s} = {rf_bin_recall_minority_weight:.4f}")
print(f"      {'class_weight':>20s} = balanced")

# ==============================================================
# 10-FOLD DETAILED EVALUATION
# ==============================================================
print("\n" + "─" * 80)
print("📊 10-Fold Cross-Validation Results (Recall-Biased Binary RF)...")
print("─" * 80)

rf_bin_recall_clf = RandomForestClassifier(
    n_estimators=rf_bin_recall_params['n_estimators'],
    max_depth=rf_bin_recall_max_depth,
    min_samples_leaf=rf_bin_recall_params['min_samples_leaf'],
    max_features=rf_bin_recall_max_features,
    class_weight='balanced',
    random_state=RF_BIN_RANDOM_STATE,
    n_jobs=-1,
)

rf_bin_recall_cv_eval = StratifiedKFold(n_splits=10, shuffle=True,
                                         random_state=RF_BIN_RANDOM_STATE)

rf_binr_auc_scores = []
rf_binr_recall_scores = []
rf_binr_precision_scores = []
rf_binr_f1_scores = []
rf_binr_thresh_scores = []

for fold, (train_idx, val_idx) in enumerate(rf_bin_recall_cv_eval.split(rf_X_bin, rf_y_bin), 1):
    X_train, X_val = rf_X_bin.iloc[train_idx], rf_X_bin.iloc[val_idx]
    y_train, y_val = rf_y_bin.iloc[train_idx], rf_y_bin.iloc[val_idx]

    sw = y_train.map({0: 1.0, 1: rf_bin_recall_minority_weight}).values
    rf_bin_recall_clf.fit(X_train, y_train, sample_weight=sw)

    y_pred_proba = rf_bin_recall_clf.predict_proba(X_val)[:, 1]

    # Use recall-targeted threshold from train set
    train_proba = rf_bin_recall_clf.predict_proba(X_train)[:, 1]
    precs_tr, recs_tr, threshs_tr = precision_recall_curve(y_train, train_proba)
    target_idx = np.argmin(np.abs(recs_tr[:-1] - 0.70))
    thresh = threshs_tr[target_idx]

    y_pred = (y_pred_proba >= thresh).astype(int)

    auc = roc_auc_score(y_val, y_pred_proba)
    recall = recall_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)

    rf_binr_auc_scores.append(auc)
    rf_binr_recall_scores.append(recall)
    rf_binr_precision_scores.append(precision)
    rf_binr_f1_scores.append(f1)
    rf_binr_thresh_scores.append(thresh)

    print(f"   Fold {fold:2d}: AUC={auc:.4f}, Recall={recall:.4f}, "
          f"Precision={precision:.4f}, F1={f1:.4f}, thresh={thresh:.4f}")

# ── Feature importance ────────────────────────────────────────
print("\n🏆 Recall-Biased Binary Feature Importance:")
rf_bin_recall_clf.fit(rf_X_bin, rf_y_bin,
                       sample_weight=rf_y_bin.map({0: 1.0, 1: rf_bin_recall_minority_weight}).values)
rf_binr_importance = pd.DataFrame({
    'feature': RF_FEATURES_BINARY,
    'importance': rf_bin_recall_clf.feature_importances_
}).sort_values('importance', ascending=False)

for _, row in rf_binr_importance.iterrows():
    bar = '█' * int(row['importance'] * 80)
    print(f"   {row['feature']:25s} {row['importance']:.4f}  {bar}")

# ── Recall at 10% FPR ────────────────────────────────────────
print("\n📊 Recall at 10% FPR:")
rf_binr_y_proba = rf_bin_recall_clf.predict_proba(rf_X_bin)[:, 1]
rf_binr_fpr, rf_binr_tpr, _ = roc_curve(rf_y_bin, rf_binr_y_proba)
rf_binr_idx_10fpr = np.argmin(np.abs(rf_binr_fpr - 0.10))
rf_binr_recall_at_10fpr = rf_binr_tpr[rf_binr_idx_10fpr]
print(f"   Recall @ 10% FPR: {rf_binr_recall_at_10fpr:.4f}")

# ==============================================================
# COMPARISON: BALANCED vs RECALL-BIASED (both on 29 binary)
# ==============================================================
print("\n" + "─" * 80)
print("⚖️  BALANCED vs RECALL-BIASED COMPARISON (29 Binary Features)")
print("─" * 80)

try:
    # rf_bin_auc_scores etc. from balanced 13E should exist
    bal_auc = np.mean(rf_bin_auc_scores)
    bal_recall = np.mean(rf_bin_recall_scores)
    bal_prec = np.mean(rf_bin_precision_scores)
    bal_f1 = np.mean(rf_bin_f1_scores)

    rec_auc = np.mean(rf_binr_auc_scores)
    rec_recall = np.mean(rf_binr_recall_scores)
    rec_prec = np.mean(rf_binr_precision_scores)
    rec_f1 = np.mean(rf_binr_f1_scores)

    print(f"\n   {'Metric':>12s} {'Balanced':>12s} {'Recall-Biased':>14s} {'Delta':>10s}")
    print(f"   {'-'*12} {'-'*12} {'-'*14} {'-'*10}")
    for name, b_val, r_val in [
        ('AUC', bal_auc, rec_auc),
        ('Recall', bal_recall, rec_recall),
        ('Precision', bal_prec, rec_prec),
        ('F1', bal_f1, rec_f1),
    ]:
        delta = r_val - b_val
        arrow = "↑" if delta > 0.001 else "↓" if delta < -0.001 else "≈"
        print(f"   {name:>12s} {b_val:>12.4f} {r_val:>14.4f} {delta:>+9.4f} {arrow}")

    rec_bump = rec_recall - bal_recall
    auc_cost = bal_auc - rec_auc
    print(f"\n   📊 Trade-off: +{rec_bump:.4f} recall for -{auc_cost:.4f} AUC")
    if rec_bump > 0.02 and auc_cost < 0.02:
        print(f"   ✅ Good trade — recall-biased RF is the better arbiter input")
    elif rec_bump > 0 and auc_cost < 0.01:
        print(f"   🟡 Marginal gain — recall-biased is slightly better")
    elif rec_bump <= 0:
        print(f"   ❌ No recall gain — balanced RF is better, skip recall bias")
    else:
        print(f"   ⚠️  Costly trade — AUC drop may hurt arbiter probability calibration")

except NameError:
    print("   ℹ️  Balanced 13E results not available — run balanced first for comparison")

# ==============================================================
# COMPARISON: vs CONTINUOUS RF
# ==============================================================
print("\n" + "─" * 80)
print("⚖️  RECALL-BIASED BINARY vs CONTINUOUS COMPARISON")
print("─" * 80)

try:
    cont_auc = np.mean(rf_auc_scores)
    cont_recall = np.mean(rf_recall_scores)
    cont_prec = np.mean(rf_precision_scores)
    cont_f1 = np.mean(rf_f1_scores)

    print(f"\n   {'Metric':>12s} {'Continuous':>12s} {'RecBias-Bin':>12s} {'Delta':>10s}")
    print(f"   {'-'*12} {'-'*12} {'-'*12} {'-'*10}")
    for name, c_val, r_val in [
        ('AUC', cont_auc, rec_auc),
        ('Recall', cont_recall, rec_recall),
        ('Precision', cont_prec, rec_prec),
        ('F1', cont_f1, rec_f1),
    ]:
        delta = r_val - c_val
        arrow = "↑" if delta > 0.001 else "↓" if delta < -0.001 else "≈"
        print(f"   {name:>12s} {c_val:>12.4f} {r_val:>12.4f} {delta:>+9.4f} {arrow}")

    auc_drop = (cont_auc - rec_auc) / cont_auc * 100
    if auc_drop > 0:
        print(f"\n   📉 AUC dropped {auc_drop:.1f}% from continuous")
        if auc_drop > 10:
            print(f"   ⚠️  >10% AUC drop — may be too costly even for disagreement")
        else:
            print(f"   ✅ Acceptable — recall-biased binary RF is a viable arbiter input")
    else:
        print(f"\n   📈 AUC maintained or improved — best of both worlds")

except NameError:
    print("   ℹ️  Continuous RF results not available — run Cell 13B first")

# ── Summary ───────────────────────────────────────────────────
rf_binr_cv_std = np.std(rf_binr_auc_scores)
rf_binr_recall_std = np.std(rf_binr_recall_scores)

print("\n" + "=" * 80)
print("📊 RANDOM FOREST — RECALL-BIASED TUNING SUMMARY (29 BINARY)")
print("=" * 80)
print(f"""
┌────────────────────────────────────────────────────────────────┐
│  Bayesian Optimization — RF RECALL-BIASED BINARY              │
├────────────────────────────────────────────────────────────────┤
│  Trials:           200 ({rf_bin_recall_n_complete} completed, {rf_bin_recall_n_pruned} pruned)          │
│  Runtime:          {rf_bin_recall_runtime:>7.1f}s                                     │
│  Best CV Score:    {rf_bin_recall_best_score:.6f} (0.6×Rec + 0.4×AUC)               │
│  10-Fold AUC:      {np.mean(rf_binr_auc_scores):.4f} ± {rf_binr_cv_std:.4f}                             │
│  10-Fold Recall:   {np.mean(rf_binr_recall_scores):.4f} ± {rf_binr_recall_std:.4f}                             │
├────────────────────────────────────────────────────────────────┤
│  n_estimators:     {rf_bin_recall_params['n_estimators']:<10}                                 │
│  max_depth:        {str(rf_bin_recall_max_depth):<10}                                 │
│  min_samples_leaf: {str(rf_bin_recall_params['min_samples_leaf']):<10}                                 │
│  max_features:     {str(rf_bin_recall_max_features):<10}                                 │
│  minority_weight:  {rf_bin_recall_minority_weight:<10.4f}                                 │
│  class_weight:     balanced                                   │
│  Feature space:    29 binary → multi-bin encoded               │
│  Avg threshold:    {np.mean(rf_binr_thresh_scores):.4f} (targeting 70% recall)           │
└────────────────────────────────────────────────────────────────┘
💡 Stability: AUC std = {rf_binr_cv_std:.4f}, Recall std = {rf_binr_recall_std:.4f}
   {'Low variance — stable' if rf_binr_cv_std < 0.015 else 'Moderate variance' if rf_binr_cv_std < 0.03 else 'High variance — overfitting risk'}
""")

# ── Save params ───────────────────────────────────────────────
rf_bin_recall_optuna_params_final = {
    'n_estimators': rf_bin_recall_params['n_estimators'],
    'max_depth': rf_bin_recall_max_depth,
    'min_samples_leaf': rf_bin_recall_params['min_samples_leaf'],
    'max_features': rf_bin_recall_max_features,
    'minority_weight': rf_bin_recall_minority_weight,
    'class_weight': 'balanced',
    'features': RF_FEATURES_BINARY,
    'objective': '0.6*recall + 0.4*auc',
    'avg_threshold': float(np.mean(rf_binr_thresh_scores)),
}

print(f"   💾 rf_bin_recall_optuna_params_final saved")
print(f"      {rf_bin_recall_optuna_params_final}")
print(f"\n   Next steps:")
print(f"      1. Compare balanced vs recall-biased in Venn trace")
print(f"      2. Pick the one with better disagreement topology")
print(f"      3. Feed into Cell 16 bitmask arbiter")
print("=" * 80)


🌲 RANDOM FOREST — RECALL-BIASED OPTUNA (29 BINARY FEATURES)

📊 Dataset: 41,188 samples, 29 binary features
   Feature space: 2^29 = 536,870,912 possible patterns
   Observed patterns: 479
   Class distribution: {0: 36548, 1: 4640}

🔍 Pre-modeling data quality check...
   ✅ All values binary, no missing data

────────────────────────────────────────────────────────────────────────────────
🧠 Bayesian Optimization (Optuna) — 200 trials, 5-fold Stratified CV
   🎯 RECALL-BIASED: objective = 0.6 × Recall + 0.4 × AUC
   📐 29 binary features — multi-bin encoding
   ✂️  Pruning after fold 3 (n_warmup_steps=2)
────────────────────────────────────────────────────────────────────────────────


Best trial: 157. Best value: 0.746706: 100%|██████████| 200/200 [45:00<00:00, 13.50s/it]



   ✅ Completed in 2700.2s
   Trials completed: 109 | Pruned: 91
   Best CV Score (0.6×Rec + 0.4×AUC): 0.746706

   Best hyperparameters:
              n_estimators = 600
                 max_depth = 3
          min_samples_leaf = 28
              max_features = 0.9734244081085968
           minority_weight = 10.7902
              class_weight = balanced

────────────────────────────────────────────────────────────────────────────────
📊 10-Fold Cross-Validation Results (Recall-Biased Binary RF)...
────────────────────────────────────────────────────────────────────────────────
   Fold  1: AUC=0.7938, Recall=0.7263, Precision=0.2458, F1=0.3673, thresh=0.9311
   Fold  2: AUC=0.7724, Recall=0.7026, Precision=0.2395, F1=0.3573, thresh=0.9308
   Fold  3: AUC=0.7869, Recall=0.7134, Precision=0.2437, F1=0.3633, thresh=0.9314
   Fold  4: AUC=0.7805, Recall=0.7263, Precision=0.2435, F1=0.3647, thresh=0.9309
   Fold  5: AUC=0.7912, Recall=0.7414, Precision=0.2520, F1=0.3762, thresh=0.9313
   Fol

In [86]:
# =====================================================================================
# CELL 13C: RF FEATURE DIAGNOSTICS (GINI, PERMUTATION, CORRELATION, AGREEMENT RANKING)
# =====================================================================================

print("\n" + "="*80)
print("🔬 RF FEATURE DIAGNOSTICS")
print(f"   Feature set: {len(RF_FEATURES)} features")
print("="*80)

# --- Setup ---
diag_X = df_engineered[RF_FEATURES].copy()
diag_y = df_engineered[TARGET_COL]

# Get model from 13B (falls back to 13A if 13B hasn't run)
try:
    _rf_model = rf_best_pipe.named_steps['clf']
    rf_best_pipe.fit(diag_X, diag_y)
    _source = "13B (Optuna)"
except NameError:
    try:
        best_rf.fit(diag_X, diag_y)
        _rf_model = best_rf
        _source = "13A (GridSearch)"
    except NameError:
        print("❌ No RF model found. Run Cell 13A or 13B first.")
        raise

print(f"   Model source: {_source}")
print(f"   Model config: max_depth={_rf_model.max_depth}, "
      f"min_samples_leaf={_rf_model.min_samples_leaf}, "
      f"n_estimators={_rf_model.n_estimators}")

# ============================================================
# 1️⃣  GINI IMPORTANCE
# ============================================================
print("\n" + "-"*80)
print("1️⃣  GINI IMPORTANCE (Mean Decrease in Impurity)")
print("-"*80)

gini_df = pd.DataFrame({
    'feature': RF_FEATURES,
    'gini_importance': _rf_model.feature_importances_
}).sort_values('gini_importance', ascending=False).reset_index(drop=True)

gini_df['gini_rank'] = range(1, len(gini_df) + 1)
gini_df['gini_cumulative'] = gini_df['gini_importance'].cumsum()

for _, row in gini_df.iterrows():
    bar = '█' * int(row['gini_importance'] * 100)
    if row['gini_importance'] >= 0.03:
        flag = "🔥 TOP"
    elif row['gini_importance'] >= 0.01:
        flag = "⚪ MID"
    else:
        flag = "💀 LOW"
    print(f"   {row['gini_rank']:2d}. {row['feature']:40s} {row['gini_importance']:.4f}  {bar}  {flag}")

gini_dead = gini_df[gini_df['gini_importance'] < 0.01]['feature'].tolist()
# Find how many features capture 90% of importance
n_for_90 = (gini_df['gini_cumulative'] < 0.90).sum() + 1
print(f"\n   📊 Top {n_for_90} features capture 90% of Gini importance")
print(f"   💀 Features with Gini < 0.01: {len(gini_dead)}")

# ============================================================
# 2️⃣  PERMUTATION IMPORTANCE (model-agnostic, less biased)
# ============================================================
print("\n" + "-"*80)
print("2️⃣  PERMUTATION IMPORTANCE (5-fold CV, unbiased)")
print("-"*80)
print("   Computing permutation importance (this may take ~30-60s)...")

perm_t0 = time.perf_counter()

# Use CV-based permutation importance to avoid overfitting bias
perm_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
perm_scores_all = []

for fold_idx, (train_idx, val_idx) in enumerate(perm_cv.split(diag_X, diag_y)):
    X_train, X_val = diag_X.iloc[train_idx], diag_X.iloc[val_idx]
    y_train, y_val = diag_y.iloc[train_idx], diag_y.iloc[val_idx]
    
    _rf_model.fit(X_train, y_train)
    perm_result = permutation_importance(
        _rf_model, X_val, y_val,
        n_repeats=10,
        scoring='roc_auc',
        random_state=42,
        n_jobs=-1
    )
    perm_scores_all.append(perm_result.importances_mean)

perm_runtime = time.perf_counter() - perm_t0
perm_mean = np.mean(perm_scores_all, axis=0)
perm_std = np.std(perm_scores_all, axis=0)

# Refit on full data for downstream use
_rf_model.fit(diag_X, diag_y)

perm_df = pd.DataFrame({
    'feature': RF_FEATURES,
    'perm_importance': perm_mean,
    'perm_std': perm_std
}).sort_values('perm_importance', ascending=False).reset_index(drop=True)

perm_df['perm_rank'] = range(1, len(perm_df) + 1)

print(f"   ✅ Computed in {perm_runtime:.1f}s\n")

for _, row in perm_df.iterrows():
    bar = '█' * int(row['perm_importance'] * 200)
    if row['perm_importance'] >= 0.005:
        flag = "🔥 TOP"
    elif row['perm_importance'] >= 0.001:
        flag = "⚪ MID"
    elif row['perm_importance'] > 0:
        flag = "⚠️  MARGINAL"
    else:
        flag = "💀 ZERO/NEG"
    print(f"   {row['perm_rank']:2d}. {row['feature']:40s} "
          f"{row['perm_importance']:+.5f} ±{row['perm_std']:.5f}  {bar}  {flag}")

perm_dead = perm_df[perm_df['perm_importance'] <= 0]['feature'].tolist()
perm_marginal = perm_df[(perm_df['perm_importance'] > 0) & 
                         (perm_df['perm_importance'] < 0.001)]['feature'].tolist()
print(f"\n   💀 Zero/negative permutation importance: {len(perm_dead)}")
print(f"   ⚠️  Marginal (0 < perm < 0.001): {len(perm_marginal)}")

# ============================================================
# 3️⃣  GINI vs PERMUTATION AGREEMENT
# ============================================================
print("\n" + "-"*80)
print("3️⃣  GINI vs PERMUTATION AGREEMENT")
print("-"*80)

# Merge rankings
agreement_df = gini_df[['feature', 'gini_importance', 'gini_rank']].merge(
    perm_df[['feature', 'perm_importance', 'perm_rank']],
    on='feature'
)
agreement_df['rank_diff'] = abs(agreement_df['gini_rank'] - agreement_df['perm_rank'])
agreement_df = agreement_df.sort_values('gini_rank')

# Spearman rank correlation
rho, p_val = spearmanr(agreement_df['gini_rank'], agreement_df['perm_rank'])
print(f"\n   Spearman rank correlation: ρ = {rho:.3f} (p = {p_val:.4f})")
if rho > 0.8:
    print(f"   ✅ Strong agreement — Gini and permutation rankings are consistent")
elif rho > 0.5:
    print(f"   ⚠️  Moderate agreement — some features may be spuriously ranked by Gini")
else:
    print(f"   ❌ Weak agreement — Gini is likely biased (check for high-cardinality features)")

# Top-10 overlap
gini_top10 = set(gini_df.head(10)['feature'])
perm_top10 = set(perm_df.head(10)['feature'])
top10_overlap = gini_top10 & perm_top10
top10_gini_only = gini_top10 - perm_top10
top10_perm_only = perm_top10 - gini_top10

print(f"\n   📊 Top-10 overlap: {len(top10_overlap)}/10")
print(f"      Both:      {sorted(top10_overlap)}")
if top10_gini_only:
    print(f"      Gini only: {sorted(top10_gini_only)}  ← may be Gini-biased")
if top10_perm_only:
    print(f"      Perm only: {sorted(top10_perm_only)}  ← genuinely predictive")

# Features that rank low on BOTH — safest removal candidates
print(f"\n   🎯 SAFE REMOVAL CANDIDATES (low on BOTH rankings):")
agreement_df['avg_rank'] = (agreement_df['gini_rank'] + agreement_df['perm_rank']) / 2
safe_cuts = agreement_df[
    (agreement_df['gini_importance'] < 0.01) & 
    (agreement_df['perm_importance'] < 0.001)
].sort_values('avg_rank', ascending=False)

if len(safe_cuts) > 0:
    for _, row in safe_cuts.iterrows():
        print(f"      {row['feature']:40s} Gini={row['gini_importance']:.4f} (#{int(row['gini_rank'])}), "
              f"Perm={row['perm_importance']:.5f} (#{int(row['perm_rank'])})")
    safe_cut_list = safe_cuts['feature'].tolist()
    print(f"\n   → {len(safe_cuts)} features are safe to remove")
else:
    safe_cut_list = []
    print(f"      No features are low on both — all contribute something")

# ============================================================
# 4️⃣  CORRELATION ANALYSIS (redundancy → cleaner GLASS-BRW rules)
# ============================================================
# Highly correlated features produce redundant rule conditions
# and split importance across similar signals. Removing them
# yields fewer, more distinct rules with clearer interpretations.

print("\n" + "-"*80)
print("4️⃣  CORRELATION ANALYSIS (redundancy → cleaner GLASS-BRW rules)")
print("-"*80)

corr_matrix = diag_X.corr()

high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.7:
            feat_i = corr_matrix.columns[i]
            feat_j = corr_matrix.columns[j]
            # Determine which to keep (higher avg importance)
            imp_i = agreement_df[agreement_df['feature'] == feat_i]['avg_rank'].values
            imp_j = agreement_df[agreement_df['feature'] == feat_j]['avg_rank'].values
            rank_i = imp_i[0] if len(imp_i) > 0 else 999
            rank_j = imp_j[0] if len(imp_j) > 0 else 999
            keep = feat_i if rank_i < rank_j else feat_j
            drop = feat_j if rank_i < rank_j else feat_i
            high_corr_pairs.append({
                'Feature 1': feat_i,
                'Feature 2': feat_j,
                'Correlation': r,
                'Keep (higher rank)': keep,
                'Drop candidate': drop
            })

if high_corr_pairs:
    high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlation', key=abs, ascending=False)
    print(f"\n   📊 High Correlation Pairs (|r| > 0.7): {len(high_corr_df)} pairs")
    for _, row in high_corr_df.iterrows():
        flag = "🔴" if abs(row['Correlation']) > 0.9 else "🟡"
        print(f"   {flag} {row['Feature 1']:35s} ↔ {row['Feature 2']:35s}  r={row['Correlation']:+.3f}")
        print(f"      → Keep: {row['Keep (higher rank)']},  Drop candidate: {row['Drop candidate']}")
    
    corr_drop_candidates = list(set(high_corr_df['Drop candidate'].tolist()))
    print(f"\n   🎯 Correlation-based drop candidates: {corr_drop_candidates}")
else:
    high_corr_df = pd.DataFrame()
    corr_drop_candidates = []
    print("\n   ✅ No high correlation pairs (|r| > 0.7)")

# ============================================================
# 5️⃣  CLASS SEPARATION ANALYSIS
# ============================================================
print("\n" + "-"*80)
print("5️⃣  CLASS SEPARATION ANALYSIS")
print("-"*80)

pos_mask = diag_y == 1
neg_mask = diag_y == 0

separation_data = []
for col in RF_FEATURES:
    vals_pos = diag_X.loc[pos_mask, col]
    vals_neg = diag_X.loc[neg_mask, col]
    
    # Cohen's d
    pooled_std = np.sqrt((vals_neg.std()**2 + vals_pos.std()**2) / 2)
    cohens_d = (vals_pos.mean() - vals_neg.mean()) / pooled_std if pooled_std > 0 else 0
    
    # KS statistic
    ks_stat, _ = ks_2samp(vals_pos, vals_neg)
    
    # Point-biserial correlation
    r_pb, _ = pointbiserialr(diag_y, diag_X[col])
    
    separation_data.append({
        'feature': col,
        'cohens_d': cohens_d,
        'ks_stat': ks_stat,
        'r_pb': r_pb,
        'abs_r': abs(r_pb)
    })

sep_df = pd.DataFrame(separation_data).sort_values('abs_r', ascending=False)

for _, row in sep_df.iterrows():
    if row['abs_r'] >= 0.2:
        flag = "✅ STRONG"
    elif row['abs_r'] >= 0.05:
        flag = "⚠️  MOD"
    else:
        flag = "❌ WEAK"
    print(f"   {row['feature']:40s} d={row['cohens_d']:+.3f}  KS={row['ks_stat']:.3f}  r={row['r_pb']:+.3f}  {flag}")

weak_seps = sep_df[sep_df['abs_r'] < 0.05]['feature'].tolist()
if weak_seps:
    print(f"\n   ⚠️  Weak separators (|r| < 0.05): {weak_seps}")

# ============================================================
# 6️⃣  COMPOSITE SCORE & TIER ASSIGNMENT
# ============================================================
print("\n" + "-"*80)
print("6️⃣  COMPOSITE RANKING & FEATURE TIERS")
print("-"*80)

# Build composite dataframe
composite_df = agreement_df[['feature', 'gini_importance', 'gini_rank', 
                              'perm_importance', 'perm_rank', 'avg_rank']].copy()

# Merge class separation
sep_merge = sep_df[['feature', 'abs_r', 'ks_stat']].rename(
    columns={'abs_r': 'class_sep_r', 'ks_stat': 'class_sep_ks'}
)
composite_df = composite_df.merge(sep_merge, on='feature')

# Normalize each metric to [0, 1] for composite score
from sklearn.preprocessing import MinMaxScaler
_scaler = MinMaxScaler()
score_cols = ['gini_importance', 'perm_importance', 'class_sep_r', 'class_sep_ks']
composite_df[['norm_gini', 'norm_perm', 'norm_sep_r', 'norm_sep_ks']] = _scaler.fit_transform(
    composite_df[score_cols]
)

# Weighted composite: permutation importance gets highest weight (least biased)
composite_df['composite_score'] = (
    0.20 * composite_df['norm_gini'] +
    0.40 * composite_df['norm_perm'] +    # permutation is gold standard
    0.20 * composite_df['norm_sep_r'] +
    0.20 * composite_df['norm_sep_ks']
)

composite_df = composite_df.sort_values('composite_score', ascending=False).reset_index(drop=True)
composite_df['composite_rank'] = range(1, len(composite_df) + 1)

# Assign tiers
def assign_tier(row):
    if row['composite_rank'] <= 10:
        return '🟢 TIER 1 (keep)'
    elif row['composite_rank'] <= 20:
        return '🟡 TIER 2 (likely keep)'
    elif row['composite_rank'] <= 30:
        return '🟠 TIER 3 (test removal)'
    else:
        return '🔴 TIER 4 (safe to cut)'

composite_df['tier'] = composite_df.apply(assign_tier, axis=1)

print(f"\n   Composite score = 0.20×Gini + 0.40×Perm + 0.20×ClassSep_r + 0.20×ClassSep_KS")
print(f"   (Permutation importance weighted highest — least biased for trees)\n")

for _, row in composite_df.iterrows():
    print(f"   {row['composite_rank']:2d}. {row['feature']:40s} "
          f"Score={row['composite_score']:.3f}  "
          f"Gini=#{int(row['gini_rank']):2d}  Perm=#{int(row['perm_rank']):2d}  "
          f"{row['tier']}")

tier_counts = composite_df['tier'].value_counts().sort_index()
print(f"\n   📊 Tier distribution:")
for tier, count in tier_counts.items():
    print(f"      {tier}: {count} features")

# ============================================================
# 7️⃣  RECOMMENDED CUT BATCHES
# ============================================================
print("\n" + "-"*80)
print("7️⃣  RECOMMENDED CUT BATCHES")
print("-"*80)

# Batch 1: Tier 4 — safe to cut, test first
batch1 = composite_df[composite_df['tier'].str.contains('TIER 4')]['feature'].tolist()
# Batch 2: Tier 3 — test removal one-by-one or in small groups
batch2 = composite_df[composite_df['tier'].str.contains('TIER 3')]['feature'].tolist()
# Also flag correlation-based cuts that are in Tier 2/3
corr_in_upper = [f for f in corr_drop_candidates if f in 
                 composite_df[composite_df['composite_rank'] <= 20]['feature'].tolist()]

print(f"\n   BATCH 1 (cut first → run 13A to verify):")
if batch1:
    for f in batch1:
        row = composite_df[composite_df['feature'] == f].iloc[0]
        print(f"      ❌ {f:40s} (Score={row['composite_score']:.3f})")
    print(f"      → Remove all {len(batch1)} features, expect minimal AUC loss")
else:
    print(f"      (No Tier 4 features — feature set may already be lean)")

print(f"\n   BATCH 2 (cut next → run 13A after each sub-batch):")
if batch2:
    for f in batch2:
        row = composite_df[composite_df['feature'] == f].iloc[0]
        in_corr = " ← also correlation-redundant" if f in corr_drop_candidates else ""
        print(f"      ⚠️  {f:40s} (Score={row['composite_score']:.3f}){in_corr}")
    print(f"      → Remove in groups of 3-5, verify AUC after each")
else:
    print(f"      (No Tier 3 features)")

if corr_in_upper:
    print(f"\n   BONUS — Correlation redundancy in top-20 (pick one from each pair):")
    for f in corr_in_upper:
        rows = high_corr_df[(high_corr_df['Feature 1'] == f) | (high_corr_df['Feature 2'] == f)]
        for _, r in rows.iterrows():
            print(f"      🔄 {r['Feature 1']:35s} ↔ {r['Feature 2']:35s} (r={r['Correlation']:+.3f})")

# Target feature count
n_keep_target = min(20, len(composite_df[composite_df['tier'].str.contains('TIER 1|TIER 2')]))
print(f"\n   🎯 Suggested target: ~{n_keep_target} features (Tier 1 + Tier 2)")
print(f"      Current: {len(RF_FEATURES)} → Target: ~{n_keep_target}")
print(f"      To remove: ~{len(RF_FEATURES) - n_keep_target} features across batches")

# ============================================================
# 8️⃣  SUMMARY
# ============================================================
print("\n" + "-"*80)
print("8️⃣  SUMMARY")
print("-"*80)
print(f"\n   Features:              {len(RF_FEATURES)}")
print(f"   Gini dead (<0.01):     {len(gini_dead)}")
print(f"   Perm dead (≤0):        {len(perm_dead)}")
print(f"   Perm marginal (<0.001):{len(perm_marginal)}")
print(f"   High corr pairs:       {len(high_corr_pairs)}")
print(f"   Weak class sep (<0.05):{len(weak_seps)}")
print(f"   Top-10 Gini/Perm agree:{len(top10_overlap)}/10")
print(f"   Spearman ρ:            {rho:.3f}")

print(f"\n   🎯 REDUCTION ROADMAP:")
print(f"      Step 1: Remove Batch 1 ({len(batch1)} features) → run Cell 13A")
print(f"      Step 2: Remove Batch 2 in groups ({len(batch2)} features) → run Cell 13A each time")
print(f"      Step 3: Test correlation pair swaps → run Cell 13A each time")
print(f"      Step 4: Final set → run Cell 13B (Optuna) for final validation")
print(f"      Step 5: Re-run this cell (13C) on final set to confirm clean diagnostics")

# ============================================================
# 📊 VISUALIZATIONS
# ============================================================

# --- Gini vs Permutation scatter ---
print("\n📊 Generating diagnostic plots...")
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Plot 1: Gini vs Permutation importance scatter
ax = axes[0, 0]
colors = ['#2ecc71' if t.startswith('🟢') else '#f1c40f' if t.startswith('🟡') 
          else '#e67e22' if t.startswith('🟠') else '#e74c3c' 
          for t in composite_df['tier']]
ax.scatter(composite_df['gini_importance'], composite_df['perm_importance'], 
           c=colors, s=60, alpha=0.8, edgecolors='black', linewidth=0.5)
for _, row in composite_df.head(10).iterrows():
    ax.annotate(row['feature'], (row['gini_importance'], row['perm_importance']),
                fontsize=7, alpha=0.8, rotation=15)
ax.set_xlabel('Gini Importance')
ax.set_ylabel('Permutation Importance')
ax.set_title(f'Gini vs Permutation (Spearman ρ={rho:.3f})')
ax.axhline(y=0.001, color='red', linestyle='--', alpha=0.5, label='Perm threshold')
ax.axvline(x=0.01, color='blue', linestyle='--', alpha=0.5, label='Gini threshold')
ax.legend(fontsize=8)

# Plot 2: Composite score bar chart with tiers
ax = axes[0, 1]
tier_colors = {'🟢 TIER 1 (keep)': '#2ecc71', '🟡 TIER 2 (likely keep)': '#f1c40f',
               '🟠 TIER 3 (test removal)': '#e67e22', '🔴 TIER 4 (safe to cut)': '#e74c3c'}
bar_colors = [tier_colors.get(t, '#95a5a6') for t in composite_df['tier']]
ax.barh(range(len(composite_df)), composite_df['composite_score'], color=bar_colors, edgecolor='black', linewidth=0.3)
ax.set_yticks(range(len(composite_df)))
ax.set_yticklabels(composite_df['feature'], fontsize=7)
ax.set_xlabel('Composite Score')
ax.set_title('Feature Composite Ranking (by tier)')
ax.invert_yaxis()

# Plot 3: Correlation heatmap (top-20 features only for readability)
ax = axes[1, 0]
top20_feats = composite_df.head(20)['feature'].tolist()
top20_corr = diag_X[top20_feats].corr()
mask = np.triu(np.ones_like(top20_corr, dtype=bool), k=1)
sns.heatmap(top20_corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            mask=mask, square=True, ax=ax, cbar_kws={'shrink': 0.8},
            annot_kws={'size': 6})
ax.set_title(f'Correlation Matrix (Top 20 features)', fontsize=11)
ax.tick_params(axis='both', labelsize=7)

# Plot 4: Rank comparison (Gini rank vs Perm rank)
ax = axes[1, 1]
n_feats = len(composite_df)
for _, row in composite_df.iterrows():
    color = '#2ecc71' if row['tier'].startswith('🟢') else '#f1c40f' if row['tier'].startswith('🟡') else '#e67e22' if row['tier'].startswith('🟠') else '#e74c3c'
    ax.plot([0, 1], [row['gini_rank'], row['perm_rank']], 
            color=color, alpha=0.6, linewidth=1.5)
    ax.scatter([0], [row['gini_rank']], color=color, s=30, zorder=5)
    ax.scatter([1], [row['perm_rank']], color=color, s=30, zorder=5)

# Label top movers (rank_diff lives on agreement_df)
agreement_with_tiers = agreement_df[['feature', 'rank_diff']].merge(
    composite_df[['feature', 'perm_rank']], on='feature'
)
big_movers = agreement_with_tiers[agreement_with_tiers['rank_diff'] > 10]
for _, row in big_movers.iterrows():
    ax.annotate(row['feature'], (1.02, row['perm_rank']), fontsize=6, alpha=0.7)

ax.set_xticks([0, 1])
ax.set_xticklabels(['Gini Rank', 'Permutation Rank'])
ax.set_ylabel('Rank (1 = best)')
ax.set_title(f'Rank Agreement (lines = features, red = big movers)')
ax.set_ylim(n_feats + 1, 0)

plt.tight_layout()
plt.show()

# --- Violin plots for top 6 composite features ---
top_composite = composite_df.head(6)['feature'].tolist()
if len(top_composite) > 0:
    print(f"\n📊 Violin plots for top {len(top_composite)} composite features...")
    n_plots = len(top_composite)
    n_cols = 3
    n_rows = (n_plots + n_cols - 1) // n_cols
    fig, axes_v = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    axes_v = axes_v.flatten() if n_plots > 1 else [axes_v]
    
    for idx, feat in enumerate(top_composite):
        ax = axes_v[idx]
        data_plot = pd.DataFrame({
            'Value': diag_X[feat],
            'Class': diag_y.map({0: 'No Subscribe', 1: 'Subscribe'})
        })
        sns.violinplot(data=data_plot, x='Class', y='Value', ax=ax,
                      palette={'No Subscribe': '#e74c3c', 'Subscribe': '#2ecc71'})
        feat_row = sep_df[sep_df['feature'] == feat].iloc[0]
        ax.set_title(f"{feat}\nd={feat_row['cohens_d']:.3f}, r={feat_row['r_pb']:.3f}", fontsize=10)
        ax.set_xlabel('')
    
    for idx in range(len(top_composite), len(axes_v)):
        axes_v[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()

# ============================================================
# 📋 COPY-PASTE HELPER: Reduced feature set after Batch 1 cut
# ============================================================
if batch1:
    reduced_features = [f for f in RF_FEATURES if f not in batch1]
    print(f"\n" + "="*80)
    print(f"📋 COPY-PASTE: RF_FEATURES after Batch 1 removal ({len(reduced_features)} features)")
    print(f"="*80)
    print(f"\nRF_FEATURES = [")
    for f in reduced_features:
        tier = composite_df[composite_df['feature'] == f]['tier'].values[0]
        rank = int(composite_df[composite_df['feature'] == f]['composite_rank'].values[0])
        print(f"    '{f}',  # #{rank} {tier}")
    print(f"]")

print("\n" + "="*80)
print("✅ RF Diagnostics complete — follow the reduction roadmap above")
print("="*80)


🔬 RF FEATURE DIAGNOSTICS
   Feature set: 8 features
   Model source: 13B (Optuna)
   Model config: max_depth=9, min_samples_leaf=10, n_estimators=800

--------------------------------------------------------------------------------
1️⃣  GINI IMPORTANCE (Mean Decrease in Impurity)
--------------------------------------------------------------------------------
    1. nsd_high                                 0.2428  ████████████████████████  🔥 TOP
    2. jed_elevated                             0.2080  ████████████████████  🔥 TOP
    3. cci_favorable                            0.1789  █████████████████  🔥 TOP
    4. dow_high                                 0.1711  █████████████████  🔥 TOP
    5. behav_high                               0.0948  █████████  🔥 TOP
    6. eci_high                                 0.0648  ██████  🔥 TOP
    7. cpi_high_cellular                        0.0319  ███  🔥 TOP
    8. campaign_fresh                           0.0076    💀 LOW

   📊 Top 6 features capture 

In [ ]:
# ── Merge binary columns back into df_engineered (We will fix this upstream later ) ─────────────
for feat in RF_FEATURES_BINARY:
    df_engineered[feat] = df_binned[feat]

# ── Reassign RF_FEATURES for downstream cells ─────────────────
RF_FEATURES = RF_FEATURES_BINARY

In [88]:
# ============================================================
# CELL 14A: EBM (EXPLAINABLE BOOSTING MACHINE) EXPERIMENT — BALANCED
# ============================================================
# Check if interpret package is available
try:
    from interpret.glassbox import ExplainableBoostingClassifier
    EBM_AVAILABLE = True
except ImportError:
    EBM_AVAILABLE = False
    print("\n" + "="*80)
    print("⚠️  WARNING: interpret package not installed")
    print("="*80)
    print("\nTo install EBM, run:")
    print("   pip install interpret")
    print("\nSkipping EBM experiment...")

if EBM_AVAILABLE:
    from sklearn.utils.class_weight import compute_sample_weight

    print("\n" + "="*80)
    print("🔮 EXPERIMENT 3: EXPLAINABLE BOOSTING MACHINE (EBM) — BALANCED")
    print("="*80)
    
    # Prepare data with EBM-specific features
    X = df_engineered[EBM_FEATURES]
    y = df_engineered[TARGET_COL]
    
    # Compute balanced sample weights
    sample_weights = compute_sample_weight('balanced', y)
    pos_weight = sample_weights[y == 1].mean()
    neg_weight = sample_weights[y == 0].mean()
    
    print(f"\n📊 Dataset: {X.shape[0]:,} samples, {X.shape[1]} features (EBM-optimized)")
    print(f"   Focus on interaction-rich and non-linear features")
    print(f"   Class distribution: {y.value_counts().to_dict()}")
    print(f"   ⚖️  Balanced sample weights: class 0 → {neg_weight:.4f}, class 1 → {pos_weight:.4f}")
    print(f"      (positive class upweighted {pos_weight/neg_weight:.1f}×)")
    
    print("\n🔍 Hyperparameter Tuning...")
    
    # Parameter grid for EBM
    param_grid_ebm = {
        'max_bins': [256, 512],
        'max_interaction_bins': [32, 64],
        'interactions': [1, 2, 3],
        'learning_rate': [0.01, 0.02],
        'min_samples_leaf': [2, 5],
        'max_leaves': [3, 5]
    }
    
    # Setup cross-validation
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    
    # Random search for EBM (full grid would be too slow)
    ebm = ExplainableBoostingClassifier(random_state=42, n_jobs=1)  # n_jobs=1 to avoid pickling errors
    
    random_search_ebm = RandomizedSearchCV(
        ebm,
        param_grid_ebm,
        n_iter=10,  # Try 10 combinations
        cv=5,  # Use 5-fold for tuning
        scoring='roc_auc',
        n_jobs=-1,  # Can use -1 here since base estimator has n_jobs=1
        verbose=0,
        random_state=42
    )
    
    random_search_ebm.fit(X, y, sample_weight=sample_weights)  # ← BALANCED FIT
    
    print(f"\n✅ Best parameters: {random_search_ebm.best_params_}")
    print(f"✅ Best CV AUC: {random_search_ebm.best_score_:.4f}")
    
    # Store best params
    best_params = random_search_ebm.best_params_
    
    # Get best model for later analysis
    best_ebm = ExplainableBoostingClassifier(
        random_state=42,
        n_jobs=1,
        **best_params
    )
    best_ebm.fit(X, y, sample_weight=sample_weights)  # ← BALANCED FIT
    
    # 10-fold evaluation with best params
    print("\n📊 10-Fold Cross-Validation Results (Balanced)...")
    auc_scores = []
    recall_scores = []
    precision_scores = []
    f1_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), 1):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        w_train = sample_weights[train_idx]  # ← FOLD-SPECIFIC WEIGHTS
        
        # Create fresh model each fold with best params
        fold_ebm = ExplainableBoostingClassifier(
            random_state=42,
            n_jobs=1,  # n_jobs=1 to avoid pickling errors
            **best_params
        )
        
        # Train with balanced weights
        fold_ebm.fit(X_train, y_train, sample_weight=w_train)  # ← BALANCED FIT
        
        # Predict
        y_pred_proba = fold_ebm.predict_proba(X_val)[:, 1]
        y_pred = fold_ebm.predict(X_val)
        
        # Metrics
        auc = roc_auc_score(y_val, y_pred_proba)
        recall = recall_score(y_val, y_pred)
        precision = precision_score(y_val, y_pred)
        f1 = f1_score(y_val, y_pred)
        
        auc_scores.append(auc)
        recall_scores.append(recall)
        precision_scores.append(precision)
        f1_scores.append(f1)
        
        print(f"   Fold {fold:2d}: AUC={auc:.4f}, Recall={recall:.4f}, Precision={precision:.4f}, F1={f1:.4f}")
    
    # Summary statistics
    print("\n" + "-"*80)
    print("🔮 EBM SUMMARY (BALANCED)")
    print("-"*80)
    print(f"AUC:       {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
    print(f"Recall:    {np.mean(recall_scores):.4f} ± {np.std(recall_scores):.4f}")
    print(f"Precision: {np.mean(precision_scores):.4f} ± {np.std(precision_scores):.4f}")
    print(f"F1:        {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
    
    # Feature importance (using best_ebm already trained above)
    print("\n🏆 Top 15 Features by Importance:")
    
    # Get global explanation from EBM
    ebm_global = best_ebm.explain_global()
    
    # Extract feature importances (main effects only, not interactions)
    feature_names = []
    feature_scores = []
    
    for i, feat in enumerate(ebm_global.data()['names']):
        if isinstance(feat, str):  # Main effects (not tuple interactions)
            feature_names.append(feat)
            feature_scores.append(ebm_global.data()['scores'][i])
    
    # Create dataframe
    feature_importance = pd.DataFrame({
        'feature': feature_names,
        'importance': feature_scores
    }).sort_values('importance', ascending=False)
    
    for i, row in feature_importance.head(15).iterrows():
        print(f"   {row['feature']:30s} {row['importance']:.4f}")
    
    # Calculate Recall at 10% FPR
    print("\n📊 Recall at 10% FPR:")
    y_pred_proba = best_ebm.predict_proba(X)[:, 1]
    fpr, tpr, thresholds = roc_curve(y, y_pred_proba)
    idx_10fpr = np.argmin(np.abs(fpr - 0.10))
    recall_at_10fpr = tpr[idx_10fpr]
    print(f"   Recall @ 10% FPR: {recall_at_10fpr:.4f}")
    
    print("\n" + "="*80)


🔮 EXPERIMENT 3: EXPLAINABLE BOOSTING MACHINE (EBM) — BALANCED

📊 Dataset: 41,188 samples, 14 features (EBM-optimized)
   Focus on interaction-rich and non-linear features
   Class distribution: {0: 36548, 1: 4640}
   ⚖️  Balanced sample weights: class 0 → 0.5635, class 1 → 4.4384
      (positive class upweighted 7.9×)

🔍 Hyperparameter Tuning...

✅ Best parameters: {'min_samples_leaf': 5, 'max_leaves': 3, 'max_interaction_bins': 64, 'max_bins': 512, 'learning_rate': 0.01, 'interactions': 3}
✅ Best CV AUC: 0.2841

📊 10-Fold Cross-Validation Results (Balanced)...
   Fold  1: AUC=0.8075, Recall=0.6422, Precision=0.3767, F1=0.4749
   Fold  2: AUC=0.7869, Recall=0.6056, Precision=0.3513, F1=0.4446
   Fold  3: AUC=0.8125, Recall=0.6509, Precision=0.3612, F1=0.4646
   Fold  4: AUC=0.7883, Recall=0.6358, Precision=0.3533, F1=0.4542
   Fold  5: AUC=0.8127, Recall=0.6746, Precision=0.3869, F1=0.4918
   Fold  6: AUC=0.8128, Recall=0.6509, Precision=0.3665, F1=0.4689
   Fold  7: AUC=0.7929, Recal

In [89]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 14B: EBM — BAYESIAN OPTIMIZATION (BALANCED)                            ║
# ║  FIX: Added sample_weight='balanced' to all fit() calls                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

print("\n" + "=" * 80)
print("🔮 EXPERIMENT 3: EXPLAINABLE BOOSTING MACHINE (EBM) — BALANCED")
print("=" * 80)

# ── Prepare data ──────────────────────────────────────────
ebm_X = df_engineered[EBM_FEATURES].copy()
ebm_y = df_engineered[TARGET_COL].copy()

print(f"\n📊 Dataset: {ebm_X.shape[0]:,} samples, {ebm_X.shape[1]} features (EBM-optimized)")
print(f"   Focus on interaction-rich and non-linear features")
print(f"   Class distribution: {ebm_y.value_counts().to_dict()}")

# ── Compute balanced sample weights ───────────────────────
ebm_sample_weights = compute_sample_weight('balanced', ebm_y)
pos_weight = ebm_sample_weights[ebm_y == 1].mean()
neg_weight = ebm_sample_weights[ebm_y == 0].mean()
print(f"   ⚖️  Balanced sample weights: class 0 → {neg_weight:.4f}, class 1 → {pos_weight:.4f}")
print(f"      (positive class upweighted {pos_weight/neg_weight:.1f}×)")

# ── Safety check ──────────────────────────────────────────
print("\n🔍 Pre-modeling data quality check...")
ebm_inf_mask = np.isinf(ebm_X).any()
if ebm_inf_mask.any():
    ebm_X = ebm_X.replace([np.inf, -np.inf], np.nan)
if ebm_X.isnull().any().any():
    ebm_nan_cols = ebm_X.columns[ebm_X.isnull().any()].tolist()
    print(f"   ⚠️  NaN values in: {ebm_nan_cols} — imputing with median")
    for col in ebm_nan_cols:
        ebm_X[col] = ebm_X[col].fillna(ebm_X[col].median())
    print(f"   ✅ Data cleaned")
else:
    print(f"   ✅ No inf/NaN values detected")
assert not np.isinf(ebm_X).any().any(), "Infinity values still present!"
assert not ebm_X.isnull().any().any(), "NaN values still present!"

# ── Shared CV strategy ────────────────────────────────────
EBM_N_FOLDS = 5
EBM_RANDOM_STATE = 42
ebm_cv = StratifiedKFold(n_splits=EBM_N_FOLDS, shuffle=True, random_state=EBM_RANDOM_STATE)

# ==============================================================
# BAYESIAN OPTIMIZATION (Optuna) — 150 trials
# ==============================================================
print("\n" + "─" * 80)
print("🧠 Bayesian Optimization (Optuna) — 150 trials, 5-fold Stratified CV")
print("   ⚖️  Using balanced sample weights in all fit() calls")
print("─" * 80)

# ── Storage for interaction analysis ──────────────────────
ebm_interaction_tracker = {'with': [], 'without': []}


def ebm_optuna_objective(trial: optuna.Trial) -> float:
    """Optuna objective for EBM with balanced sample weights."""
    learning_rate = trial.suggest_float('learning_rate', 0.005, 0.05, log=True)
    max_rounds = trial.suggest_int('max_rounds', 500, 5000, step=100)
    max_bins = trial.suggest_int('max_bins', 128, 512, step=32)
    max_interaction_bins = trial.suggest_int('max_interaction_bins', 16, 128, step=16)
    interactions = trial.suggest_int('interactions', 0, 15)

    fold_scores = []
    for fold_idx, (train_idx, val_idx) in enumerate(ebm_cv.split(ebm_X, ebm_y)):
        X_train, X_val = ebm_X.iloc[train_idx], ebm_X.iloc[val_idx]
        y_train, y_val = ebm_y.iloc[train_idx], ebm_y.iloc[val_idx]
        w_train = ebm_sample_weights[train_idx]  # ← BALANCED WEIGHTS

        try:
            model = ExplainableBoostingClassifier(
                learning_rate=learning_rate,
                max_rounds=max_rounds,
                max_bins=max_bins,
                max_interaction_bins=max_interaction_bins,
                interactions=interactions,
                random_state=EBM_RANDOM_STATE,
                n_jobs=1,
            )
            model.fit(X_train, y_train, sample_weight=w_train)  # ← BALANCED FIT
            y_proba = model.predict_proba(X_val)[:, 1]
            auc = roc_auc_score(y_val, y_proba)
        except Exception as e:
            return float('-inf')

        fold_scores.append(auc)
        trial.report(np.mean(fold_scores), fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()

    mean_auc = np.mean(fold_scores)
    if interactions == 0:
        ebm_interaction_tracker['without'].append(mean_auc)
    else:
        ebm_interaction_tracker['with'].append(mean_auc)

    return mean_auc


ebm_study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=EBM_RANDOM_STATE),
    pruner=MedianPruner(n_startup_trials=20, n_warmup_steps=2),
    study_name='ebm_bayesian_opt_balanced',
)

ebm_t0 = time.perf_counter()
ebm_study.optimize(ebm_optuna_objective, n_trials=150, show_progress_bar=True)
ebm_runtime = time.perf_counter() - ebm_t0

# ── Extract results ───────────────────────────────────────
ebm_optuna_params = ebm_study.best_params.copy()
ebm_best_auc_optuna = ebm_study.best_value
ebm_n_complete = len([t for t in ebm_study.trials
                      if t.state == optuna.trial.TrialState.COMPLETE])
ebm_n_pruned = len([t for t in ebm_study.trials
                    if t.state == optuna.trial.TrialState.PRUNED])

print(f"\n   ✅ Completed in {ebm_runtime:.1f}s")
print(f"   Trials completed: {ebm_n_complete} | Pruned: {ebm_n_pruned}")
print(f"   Best CV ROC-AUC:  {ebm_best_auc_optuna:.6f}")

print(f"\n   Best hyperparameters:")
for k, v in sorted(ebm_optuna_params.items()):
    if k == 'learning_rate':
        print(f"      {k:>25s} = {v:.6f}")
    else:
        print(f"      {k:>25s} = {v}")

# ── Interaction analysis ──────────────────────────────────
print("\n" + "─" * 80)
print("🔬 Interaction Analysis")
print("─" * 80)

ebm_mean_with = np.mean(ebm_interaction_tracker['with']) if ebm_interaction_tracker['with'] else 0
ebm_mean_without = np.mean(ebm_interaction_tracker['without']) if ebm_interaction_tracker['without'] else 0
ebm_best_with = max(ebm_interaction_tracker['with']) if ebm_interaction_tracker['with'] else 0
ebm_best_without = max(ebm_interaction_tracker['without']) if ebm_interaction_tracker['without'] else 0
ebm_interaction_delta = ebm_best_with - ebm_best_without
ebm_chosen_interactions = ebm_optuna_params['interactions']

print(f"   Trials with interactions>0:   {len(ebm_interaction_tracker['with']):>4d}  "
      f"(mean AUC: {ebm_mean_with:.6f}, best: {ebm_best_with:.6f})")
print(f"   Trials with interactions=0:   {len(ebm_interaction_tracker['without']):>4d}  "
      f"(mean AUC: {ebm_mean_without:.6f}, best: {ebm_best_without:.6f})")
print(f"   Best interaction Δ (with - without): {ebm_interaction_delta:+.6f}")

if ebm_interaction_delta > 0.005:
    ebm_interaction_verdict = (
        f"Interactions materially improved performance (+{ebm_interaction_delta:.4f} AUC). "
        f"The best config uses {ebm_chosen_interactions} interactions."
    )
elif ebm_interaction_delta > 0.001:
    ebm_interaction_verdict = (
        f"Interactions provided a marginal improvement (+{ebm_interaction_delta:.4f} AUC). "
        f"Useful but not critical — monitor for overfitting on new data."
    )
else:
    ebm_interaction_verdict = (
        f"Interactions did NOT materially improve performance "
        f"(Δ={ebm_interaction_delta:+.4f} AUC). Main effects capture most signal. "
        f"Consider setting interactions=0 for maximum interpretability."
    )
print(f"\n   💡 {ebm_interaction_verdict}")

# ==============================================================
# RECONSTRUCT BEST MODEL & 10-FOLD EVALUATION (BALANCED)
# ==============================================================
print("\n" + "─" * 80)
print("📊 10-Fold Cross-Validation Results (Final Evaluation — Balanced)...")
print("─" * 80)

ebm_best_model = ExplainableBoostingClassifier(
    learning_rate=ebm_optuna_params['learning_rate'],
    max_rounds=ebm_optuna_params['max_rounds'],
    max_bins=ebm_optuna_params['max_bins'],
    max_interaction_bins=ebm_optuna_params['max_interaction_bins'],
    interactions=ebm_optuna_params['interactions'],
    random_state=EBM_RANDOM_STATE,
    n_jobs=1,
)

ebm_cv_eval = StratifiedKFold(n_splits=10, shuffle=True, random_state=EBM_RANDOM_STATE)

ebm_auc_scores = []
ebm_recall_scores = []
ebm_precision_scores = []
ebm_f1_scores = []

for fold, (train_idx, val_idx) in enumerate(ebm_cv_eval.split(ebm_X, ebm_y), 1):
    X_train, X_val = ebm_X.iloc[train_idx], ebm_X.iloc[val_idx]
    y_train, y_val = ebm_y.iloc[train_idx], ebm_y.iloc[val_idx]
    w_train = ebm_sample_weights[train_idx]  # ← BALANCED WEIGHTS

    fold_ebm = ExplainableBoostingClassifier(
        learning_rate=ebm_optuna_params['learning_rate'],
        max_rounds=ebm_optuna_params['max_rounds'],
        max_bins=ebm_optuna_params['max_bins'],
        max_interaction_bins=ebm_optuna_params['max_interaction_bins'],
        interactions=ebm_optuna_params['interactions'],
        random_state=EBM_RANDOM_STATE,
        n_jobs=1,
    )
    fold_ebm.fit(X_train, y_train, sample_weight=w_train)  # ← BALANCED FIT

    y_pred_proba = fold_ebm.predict_proba(X_val)[:, 1]
    y_pred = fold_ebm.predict(X_val)

    auc = roc_auc_score(y_val, y_pred_proba)
    recall = recall_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)

    ebm_auc_scores.append(auc)
    ebm_recall_scores.append(recall)
    ebm_precision_scores.append(precision)
    ebm_f1_scores.append(f1)

    print(f"   Fold {fold:2d}: AUC={auc:.4f}, Recall={recall:.4f}, "
          f"Precision={precision:.4f}, F1={f1:.4f}")

# ── Summary statistics ────────────────────────────────────
print("\n" + "-" * 80)
print("🔮 EBM SUMMARY (BALANCED)")
print("-" * 80)
print(f"AUC:       {np.mean(ebm_auc_scores):.4f} ± {np.std(ebm_auc_scores):.4f}")
print(f"Recall:    {np.mean(ebm_recall_scores):.4f} ± {np.std(ebm_recall_scores):.4f}")
print(f"Precision: {np.mean(ebm_precision_scores):.4f} ± {np.std(ebm_precision_scores):.4f}")
print(f"F1:        {np.mean(ebm_f1_scores):.4f} ± {np.std(ebm_f1_scores):.4f}")

# ── Feature importance ────────────────────────────────────
print("\n🏆 Top 15 Features by Importance:")
ebm_best_model.fit(ebm_X, ebm_y, sample_weight=ebm_sample_weights)  # ← BALANCED FIT
ebm_global = ebm_best_model.explain_global()

ebm_feature_names = []
ebm_feature_scores = []
for i, feat in enumerate(ebm_global.data()['names']):
    if isinstance(feat, str):
        ebm_feature_names.append(feat)
        ebm_feature_scores.append(ebm_global.data()['scores'][i])

ebm_feature_importance = pd.DataFrame({
    'feature': ebm_feature_names,
    'importance': ebm_feature_scores
}).sort_values('importance', ascending=False)

for _, row in ebm_feature_importance.head(15).iterrows():
    print(f"   {row['feature']:30s} {row['importance']:.4f}")

# ── Interaction terms (if present) ────────────────────────
if ebm_optuna_params['interactions'] > 0:
    ebm_interaction_names = []
    ebm_interaction_scores = []
    for i, feat in enumerate(ebm_global.data()['names']):
        if isinstance(feat, tuple):
            ebm_interaction_names.append(' × '.join(feat))
            ebm_interaction_scores.append(ebm_global.data()['scores'][i])

    if ebm_interaction_names:
        ebm_interaction_importance = pd.DataFrame({
            'interaction': ebm_interaction_names,
            'importance': ebm_interaction_scores
        }).sort_values('importance', ascending=False)

        print(f"\n🔗 Top Interaction Terms ({len(ebm_interaction_names)} detected):")
        for _, row in ebm_interaction_importance.head(10).iterrows():
            print(f"   {row['interaction']:40s} {row['importance']:.4f}")

# ── Recall at 10% FPR ────────────────────────────────────
print("\n📊 Recall at 10% FPR:")
ebm_y_pred_proba = ebm_best_model.predict_proba(ebm_X)[:, 1]
ebm_fpr, ebm_tpr, ebm_thresholds = roc_curve(ebm_y, ebm_y_pred_proba)
ebm_idx_10fpr = np.argmin(np.abs(ebm_fpr - 0.10))
ebm_recall_at_10fpr = ebm_tpr[ebm_idx_10fpr]
print(f"   Recall @ 10% FPR: {ebm_recall_at_10fpr:.4f}")

# ── Structured summary ────────────────────────────────────
ebm_cv_std = np.std(ebm_auc_scores)
print("\n" + "=" * 80)
print("📊 EBM — TUNING SUMMARY (BALANCED)")
print("=" * 80)
print(f"""
┌──────────────────────────────────────────────────────────────┐
│  Bayesian Optimization (Optuna) — EBM BALANCED               │
├──────────────────────────────────────────────────────────────┤
│  Trials:              150 ({ebm_n_complete} completed, {ebm_n_pruned} pruned)     │
│  Runtime:             {ebm_runtime:>7.1f}s                                │
│  Best CV ROC-AUC:     {ebm_best_auc_optuna:.6f}                              │
│  10-Fold Mean±Std:    {np.mean(ebm_auc_scores):.4f} ± {ebm_cv_std:.4f}                        │
│  Sample weighting:    balanced (class 1 @ {pos_weight:.2f}×)              │
├──────────────────────────────────────────────────────────────┤
│  learning_rate:       {ebm_optuna_params['learning_rate']:<12.6f}                            │
│  max_rounds:          {ebm_optuna_params['max_rounds']:<12}                            │
│  max_bins:            {ebm_optuna_params['max_bins']:<12}                            │
│  max_interaction_bins:{ebm_optuna_params['max_interaction_bins']:<12}                            │
│  interactions:        {ebm_optuna_params['interactions']:<12}                            │
└──────────────────────────────────────────────────────────────┘
💡 Stability: CV std = {ebm_cv_std:.4f} ({'Low variance — stable model' if ebm_cv_std < 0.015 else 'Moderate variance — acceptable for GAMs' if ebm_cv_std < 0.03 else 'High variance — consider reducing complexity'})
🔬 Interaction verdict: {ebm_interaction_verdict}
⚖️  Balance fix: sample_weight='balanced' applied to all fit() calls
   Previous issue: precision-biased (recall ~0.22, precision ~0.58)
   Expected improvement: recall ↑↑, precision ↓, F1 ↑, AUC stable or slightly improved
📝 Interpretability assessment:
   {'✅ Pure additive model (interactions=0) — fully interpretable shape functions' if ebm_optuna_params['interactions'] == 0 else f'⚠️  {ebm_optuna_params["interactions"]} interaction terms added — still interpretable via 2D heatmaps but adds complexity' if ebm_optuna_params['interactions'] <= 10 else f'⚠️  {ebm_optuna_params["interactions"]} interactions is near the cap — verify each interaction is meaningful'}
""")

print(f"   💾 ebm_optuna_params saved: {ebm_optuna_params}")
print(f"   💾 ebm_best_model saved (fitted with balanced weights on full data)")
print("=" * 80)



🔮 EXPERIMENT 3: EXPLAINABLE BOOSTING MACHINE (EBM) — BALANCED

📊 Dataset: 41,188 samples, 14 features (EBM-optimized)
   Focus on interaction-rich and non-linear features
   Class distribution: {0: 36548, 1: 4640}
   ⚖️  Balanced sample weights: class 0 → 0.5635, class 1 → 4.4384
      (positive class upweighted 7.9×)

🔍 Pre-modeling data quality check...
   ✅ No inf/NaN values detected

────────────────────────────────────────────────────────────────────────────────
🧠 Bayesian Optimization (Optuna) — 150 trials, 5-fold Stratified CV
   ⚖️  Using balanced sample weights in all fit() calls
────────────────────────────────────────────────────────────────────────────────


Best trial: 137. Best value: 0.801405: 100%|██████████| 150/150 [7:25:05<00:00, 178.03s/it]  



   ✅ Completed in 26705.1s
   Trials completed: 64 | Pruned: 86
   Best CV ROC-AUC:  0.801405

   Best hyperparameters:
                   interactions = 15
                  learning_rate = 0.026536
                       max_bins = 288
           max_interaction_bins = 32
                     max_rounds = 4700

────────────────────────────────────────────────────────────────────────────────
🔬 Interaction Analysis
────────────────────────────────────────────────────────────────────────────────
   Trials with interactions>0:     62  (mean AUC: 0.801076, best: 0.801405)
   Trials with interactions=0:      2  (mean AUC: 0.799512, best: 0.799542)
   Best interaction Δ (with - without): +0.001863

   💡 Interactions provided a marginal improvement (+0.0019 AUC). Useful but not critical — monitor for overfitting on new data.

────────────────────────────────────────────────────────────────────────────────
📊 10-Fold Cross-Validation Results (Final Evaluation — Balanced)...
──────────────────

In [90]:
# ============================================================
# CELL 14C: EBM FEATURE DIAGNOSTICS
#   (Shape Function Analysis, Effect Magnitude, Linearity,
#    Cumulative Importance, Interaction Analysis,
#    Composite Ranking & Tier Assignment)
# ============================================================

print("\n" + "="*80)
print("🔬 EBM FEATURE DIAGNOSTICS")
print(f"   Feature set: {len(EBM_FEATURES)} features")
print("="*80)
# --- Setup ---
diag_X = df_engineered[EBM_FEATURES].copy()
diag_y = df_engineered[TARGET_COL]
# Get model from 14B (falls back to 14A if 14B hasn't run)
try:
    _ebm_model = ebm_best_model
    _source = "14B (Optuna, balanced)"
except NameError:
    try:
        _ebm_model = best_ebm
        _source = "14A (RandomSearch)"
    except NameError:
        print("❌ No EBM model found. Run Cell 14A or 14B first.")
        raise
# Ensure model is fitted on full data with balanced weights
from sklearn.utils.class_weight import compute_sample_weight
_ebm_weights = compute_sample_weight('balanced', diag_y)
_ebm_model.fit(diag_X, diag_y, sample_weight=_ebm_weights)
_ebm_global = _ebm_model.explain_global()
print(f"   Model source: {_source}")
print(f"   Model config: interactions={_ebm_model.interactions}, "
      f"learning_rate={_ebm_model.learning_rate}, "
      f"max_bins={_ebm_model.max_bins}, "
      f"max_rounds={_ebm_model.max_rounds}")
# ============================================================
# 1️⃣  GLOBAL FEATURE IMPORTANCE (Main Effects)
# ============================================================
print("\n" + "-"*80)
print("1️⃣  GLOBAL FEATURE IMPORTANCE (Main Effects)")
print("-"*80)
# Separate main effects from interactions
# NOTE: EBM returns interaction terms as STRINGS with ' & ' separator,
# not as tuples. 
main_names = []
main_scores = []
interaction_names_raw = []
interaction_scores_raw = []
for i, feat in enumerate(_ebm_global.data()['names']):
    score = _ebm_global.data()['scores'][i]
    if isinstance(feat, tuple):
        # Tuple format (older EBM versions)
        interaction_names_raw.append(feat)
        interaction_scores_raw.append(score)
    elif isinstance(feat, str) and ' & ' in feat:
        # String format with ' & ' separator (newer EBM versions)
        parts = tuple(feat.split(' & '))
        interaction_names_raw.append(parts)
        interaction_scores_raw.append(score)
    else:
        main_names.append(feat)
        main_scores.append(score)
# Build a combined list for composite ranking (main + interaction display names)
all_feat_names = main_names + [' & '.join(p) if isinstance(p, tuple) else p 
                                for p in interaction_names_raw]
all_feat_scores = main_scores + interaction_scores_raw

importance_df = pd.DataFrame({
    'feature': all_feat_names,
    'importance': all_feat_scores
}).sort_values('importance', ascending=False).reset_index(drop=True)
importance_df['imp_rank'] = range(1, len(importance_df) + 1)
importance_df['imp_cumulative'] = importance_df['importance'].cumsum()
total_importance = importance_df['importance'].sum()
importance_df['imp_cumulative_pct'] = importance_df['imp_cumulative'] / total_importance
for _, row in importance_df.iterrows():
    bar = '█' * int(row['importance'] / importance_df['importance'].max() * 30)
    if row['imp_cumulative_pct'] <= 0.80:
        flag = "🔥 TOP-80%"
    elif row['imp_cumulative_pct'] <= 0.90:
        flag = "⚪ TOP-90%"
    elif row['imp_cumulative_pct'] <= 0.95:
        flag = "⚠️  TOP-95%"
    else:
        flag = "💀 TAIL"
    print(f"   {row['imp_rank']:2d}. {row['feature']:40s} {row['importance']:.4f}  "
          f"cum={row['imp_cumulative_pct']:.1%}  {bar}  {flag}")
# Cumulative importance milestones
n_for_80 = (importance_df['imp_cumulative_pct'] < 0.80).sum() + 1
n_for_90 = (importance_df['imp_cumulative_pct'] < 0.90).sum() + 1
n_for_95 = (importance_df['imp_cumulative_pct'] < 0.95).sum() + 1
tail_features = importance_df[importance_df['imp_cumulative_pct'] > 0.95]['feature'].tolist()
print(f"\n   📊 Cumulative Importance Milestones:")
print(f"      80% importance: top {n_for_80} features")
print(f"      90% importance: top {n_for_90} features")
print(f"      95% importance: top {n_for_95} features")
print(f"      Tail (>95%):   {len(tail_features)} features — candidates for removal")
# ============================================================
# 2️⃣  SHAPE FUNCTION LINEARITY (R² vs linear fit)
# ============================================================
print("\n" + "-"*80)
print("2️⃣  SHAPE FUNCTION LINEARITY (R² of shape vs linear fit)")
print("   → High R² means EBM learned a ~linear shape → LR can handle it")
print("   → Low R² means EBM is exploiting non-linearity → EBM adds value")
print("-"*80)
linearity_data = []
# Analyze ALL features (main effects + interactions) by index
all_ebm_names = _ebm_global.data()['names']
for feat_idx in range(len(all_ebm_names)):
    raw_name = all_ebm_names[feat_idx]
    # Get display name
    if isinstance(raw_name, tuple):
        display_name = ' & '.join(raw_name)
    else:
        display_name = raw_name
    
    try:
        feat_data = _ebm_global.data(feat_idx)
        shape_scores = np.array(feat_data['scores'])
        
        # For 2D interactions, flatten the scores
        if shape_scores.ndim > 1:
            shape_scores = shape_scores.flatten()
        
        # For continuous features, 'names' are bin edges; use midpoints
        # For categorical, use ordinal index
        x_vals = np.arange(len(shape_scores)).reshape(-1, 1)
        
        # Skip features with constant shape (no variation)
        if np.std(shape_scores) < 1e-10:
            linearity_data.append({
                'feature': display_name,
                'linearity_r2': 1.0,  # constant = trivially linear
                'score_range': 0.0,
                'score_std': 0.0,
                'n_bins': len(shape_scores),
                'note': 'constant shape'
            })
            continue
        
        # Fit linear regression to shape function
        lr_fit = LinearRegression()
        lr_fit.fit(x_vals, shape_scores)
        r2 = lr_fit.score(x_vals, shape_scores)
        
        score_range = float(np.max(shape_scores) - np.min(shape_scores))
        score_std = float(np.std(shape_scores))
        
        linearity_data.append({
            'feature': display_name,
            'linearity_r2': max(0, r2),  # clip negative R² to 0
            'score_range': score_range,
            'score_std': score_std,
            'n_bins': len(shape_scores),
            'note': ''
        })
    except Exception as e:
        linearity_data.append({
            'feature': display_name,
            'linearity_r2': np.nan,
            'score_range': np.nan,
            'score_std': np.nan,
            'n_bins': 0,
            'note': f'error: {str(e)[:40]}'
        })
linearity_df = pd.DataFrame(linearity_data)
# Sort by R² descending (most linear first)
linearity_df = linearity_df.sort_values('linearity_r2', ascending=False).reset_index(drop=True)
for _, row in linearity_df.iterrows():
    if pd.isna(row['linearity_r2']):
        flag = "❓ ERROR"
    elif row['linearity_r2'] > 0.90:
        flag = "📏 LINEAR"
    elif row['linearity_r2'] > 0.70:
        flag = "〰️  MODERATE"
    elif row['linearity_r2'] > 0.40:
        flag = "🌊 NON-LINEAR"
    else:
        flag = "🔀 COMPLEX"
    print(f"   {row['feature']:40s} R²={row['linearity_r2']:.3f}  "
          f"range={row['score_range']:.4f}  {flag}  {row['note']}")
highly_linear = linearity_df[linearity_df['linearity_r2'] > 0.90]['feature'].tolist()
complex_shapes = linearity_df[linearity_df['linearity_r2'] < 0.40]['feature'].tolist()
# For upstream check, only look at main effects (interactions can't be in LR/RF)
highly_linear_main = [f for f in highly_linear if ' & ' not in f]
print(f"\n   📏 Highly linear shapes (R² > 0.90): {len(highly_linear)} features")
print(f"      → EBM adds no value over LR for these; check if LR/RF already covers them")
if highly_linear_main:
    for f in highly_linear_main:
        in_lr = "✅ in LR" if f in LR_FEATURES else "❌ not in LR"
        in_rf = "✅ in RF" if f in RF_FEATURES else "❌ not in RF"
        print(f"      {f:40s} {in_lr}  {in_rf}")
print(f"\n   🔀 Complex shapes (R² < 0.40): {len(complex_shapes)} features")
print(f"      → These are where EBM adds unique value")
# ============================================================
# 3️⃣  EFFECT MAGNITUDE (Score Range per Feature)
# ============================================================
print("\n" + "-"*80)
print("3️⃣  EFFECT MAGNITUDE (Score Range per Feature)")
print("   → Features with tiny score range have negligible impact on predictions")
print("-"*80)
magnitude_df = linearity_df[['feature', 'score_range']].copy()
magnitude_df = magnitude_df.sort_values('score_range', ascending=False).reset_index(drop=True)
magnitude_df['mag_rank'] = range(1, len(magnitude_df) + 1)
for _, row in magnitude_df.iterrows():
    bar = '█' * int(row['score_range'] / magnitude_df['score_range'].max() * 30)
    if row['score_range'] >= 0.1:
        flag = "🔥 STRONG"
    elif row['score_range'] >= 0.03:
        flag = "⚪ MODERATE"
    elif row['score_range'] >= 0.01:
        flag = "⚠️  WEAK"
    else:
        flag = "💀 NEGLIGIBLE"
    print(f"   {row['mag_rank']:2d}. {row['feature']:40s} range={row['score_range']:.4f}  {bar}  {flag}")
negligible_effect = magnitude_df[magnitude_df['score_range'] < 0.01]['feature'].tolist()
weak_effect = magnitude_df[(magnitude_df['score_range'] >= 0.01) & 
                           (magnitude_df['score_range'] < 0.03)]['feature'].tolist()
print(f"\n   💀 Negligible effect (range < 0.01): {len(negligible_effect)} features")
if negligible_effect:
    for f in negligible_effect:
        print(f"      → {f}")
print(f"   ⚠️  Weak effect (0.01 ≤ range < 0.03): {len(weak_effect)} features")
# ============================================================
# 4️⃣  INTERACTION TERM ANALYSIS
# ============================================================
print("\n" + "-"*80)
print("4️⃣  INTERACTION TERM ANALYSIS")
print("-"*80)
if interaction_names_raw:
    interaction_df = pd.DataFrame({
        'interaction': [' & '.join(pair) for pair in interaction_names_raw],
        'feat_1': [pair[0] for pair in interaction_names_raw],
        'feat_2': [pair[1] for pair in interaction_names_raw],
        'importance': interaction_scores_raw
    }).sort_values('importance', ascending=False).reset_index(drop=True)
    
    total_main_imp = sum(main_scores)
    total_int_imp = interaction_df['importance'].sum()
    int_share = total_int_imp / (total_main_imp + total_int_imp) * 100
    
    print(f"\n   📊 {len(interaction_df)} interaction terms detected")
    print(f"   Main effects total importance:  {total_main_imp:.4f}")
    print(f"   Interaction total importance:   {total_int_imp:.4f} ({int_share:.1f}% of total)")
    
    for _, row in interaction_df.iterrows():
        bar = '█' * int(row['importance'] / interaction_df['importance'].max() * 20) if interaction_df['importance'].max() > 0 else ''
        print(f"   {row['interaction']:45s} {row['importance']:.4f}  {bar}")
    
    # Which features appear in interactions? (keep these even if main effect is small)
    interacting_features = set()
    for pair in interaction_names_raw:
        interacting_features.update(pair)
    interacting_features = interacting_features & set(main_names)
    print(f"\n   🔗 Features appearing in interactions: {sorted(interacting_features)}")
    print(f"      → These should be retained even if main effect seems small")
else:
    interaction_df = pd.DataFrame()
    interacting_features = set()
    print(f"\n   ℹ️  No interaction terms (interactions=0 or model has no learned interactions)")
# ============================================================
# 5️⃣  REDUNDANCY CHECK: Linear features already in LR/RF
# ============================================================
print("\n" + "-"*80)
print("5️⃣  REDUNDANCY CHECK — Linear features already handled upstream")
print("   → If a feature is linear AND already in LR or RF, EBM is redundant")
print("-"*80)
redundancy_data = []
for _, row in linearity_df.iterrows():
    feat = row['feature']
    is_interaction = ' & ' in feat
    in_lr = feat in LR_FEATURES if not is_interaction else False
    in_rf = feat in RF_FEATURES if not is_interaction else False
    is_linear = row['linearity_r2'] > 0.90
    is_negligible = row['score_range'] < 0.01
    
    # Redundant if: linear AND handled upstream AND not in an interaction
    upstream = in_lr or in_rf
    in_interaction = feat in interacting_features
    
    if is_negligible:
        verdict = "💀 NEGLIGIBLE EFFECT — safe to cut"
    elif is_interaction:
        verdict = "🔗 INTERACTION TERM — auto-learned by EBM"
    elif is_linear and upstream and not in_interaction:
        verdict = "🔄 REDUNDANT — linear + already upstream"
    elif is_linear and upstream and in_interaction:
        verdict = "⚠️  Linear + upstream BUT in interaction — keep"
    elif is_linear and not upstream:
        verdict = "📏 Linear but NOT upstream — EBM is only handler"
    else:
        verdict = "✅ NON-LINEAR — EBM adds unique value"
    
    redundancy_data.append({
        'feature': feat,
        'r2': row['linearity_r2'],
        'score_range': row['score_range'],
        'in_lr': in_lr,
        'in_rf': in_rf,
        'in_interaction': in_interaction,
        'is_interaction_term': is_interaction,
        'verdict': verdict
    })
redundancy_df = pd.DataFrame(redundancy_data)
# Show redundant features first
redundant_mask = redundancy_df['verdict'].str.contains('REDUNDANT|NEGLIGIBLE')
if redundant_mask.any():
    print(f"\n   🎯 Features flagged for removal:")
    for _, row in redundancy_df[redundant_mask].iterrows():
        lr_flag = "LR✓" if row['in_lr'] else "LR✗"
        rf_flag = "RF✓" if row['in_rf'] else "RF✗"
        print(f"      {row['feature']:40s} R²={row['r2']:.3f}  range={row['score_range']:.4f}  "
              f"[{lr_flag} {rf_flag}]  {row['verdict']}")
# Show keepers
keep_mask = ~redundant_mask
print(f"\n   ✅ Features to keep ({keep_mask.sum()}):")
for _, row in redundancy_df[keep_mask].sort_values('score_range', ascending=False).iterrows():
    lr_flag = "LR✓" if row['in_lr'] else "LR✗"
    rf_flag = "RF✓" if row['in_rf'] else "RF✗"
    int_flag = "INT✓" if row['in_interaction'] else ""
    ix_flag = " [IX]" if row['is_interaction_term'] else ""
    print(f"      {row['feature']:40s} R²={row['r2']:.3f}  range={row['score_range']:.4f}  "
          f"[{lr_flag} {rf_flag}]  {int_flag}{ix_flag}  {row['verdict']}")
# ============================================================
# 6️⃣  CORRELATION ANALYSIS 
# ============================================================
print("\n" + "-"*80)
print("6️⃣  CORRELATION ANALYSIS")
print("-"*80)
corr_matrix = diag_X.corr()
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.7:
            feat_i = corr_matrix.columns[i]
            feat_j = corr_matrix.columns[j]
            # Determine which to keep (higher importance)
            imp_i = importance_df[importance_df['feature'] == feat_i]['imp_rank'].values
            imp_j = importance_df[importance_df['feature'] == feat_j]['imp_rank'].values
            rank_i = imp_i[0] if len(imp_i) > 0 else 999
            rank_j = imp_j[0] if len(imp_j) > 0 else 999
            keep = feat_i if rank_i < rank_j else feat_j
            drop = feat_j if rank_i < rank_j else feat_i
            high_corr_pairs.append({
                'Feature 1': feat_i, 'Feature 2': feat_j,
                'Correlation': r,
                'Keep (higher rank)': keep,
                'Drop candidate': drop
            })
if high_corr_pairs:
    high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlation', key=abs, ascending=False)
    print(f"\n   📊 High Correlation Pairs (|r| > 0.7): {len(high_corr_df)} pairs")
    for _, row in high_corr_df.iterrows():
        flag = "🔴" if abs(row['Correlation']) > 0.9 else "🟡"
        print(f"   {flag} {row['Feature 1']:35s} ↔ {row['Feature 2']:35s}  r={row['Correlation']:+.3f}")
        print(f"      → Keep: {row['Keep (higher rank)']},  Drop candidate: {row['Drop candidate']}")
    corr_drop_candidates = list(set(high_corr_df['Drop candidate'].tolist()))
    print(f"\n   🎯 Correlation-based drop candidates: {corr_drop_candidates}")
else:
    high_corr_df = pd.DataFrame()
    corr_drop_candidates = []
    print("\n   ✅ No high correlation pairs (|r| > 0.7)")
# ============================================================
# 7️⃣  COMPOSITE RANKING & TIER ASSIGNMENT
# ============================================================
print("\n" + "-"*80)
print("7️⃣  COMPOSITE RANKING & FEATURE TIERS")
print("-"*80)
# Build composite from all metrics (main effects + interactions)
composite_df = importance_df[['feature', 'importance', 'imp_rank']].copy()
# Merge linearity and magnitude
lin_merge = linearity_df[['feature', 'linearity_r2', 'score_range']].copy()
composite_df = composite_df.merge(lin_merge, on='feature', how='left')
# Merge class separation (only available for main effects)
sep_merge = sep_df[['feature', 'abs_r', 'ks_stat']].rename(
    columns={'abs_r': 'class_sep_r', 'ks_stat': 'class_sep_ks'}
)
composite_df = composite_df.merge(sep_merge, on='feature', how='left')
# Fill NaN for interaction terms (no class sep data)
composite_df['class_sep_r'] = composite_df['class_sep_r'].fillna(0)
composite_df['class_sep_ks'] = composite_df['class_sep_ks'].fillna(0)
# Add interaction membership flag
composite_df['in_interaction'] = composite_df['feature'].isin(interacting_features)
# Flag whether this row IS an interaction term
composite_df['is_interaction_term'] = composite_df['feature'].str.contains(' & ')
# Normalize metrics for composite score
_scaler = MinMaxScaler()
score_cols = ['importance', 'score_range', 'class_sep_r', 'class_sep_ks']
composite_df[['norm_imp', 'norm_range', 'norm_sep_r', 'norm_sep_ks']] = _scaler.fit_transform(
    composite_df[score_cols].fillna(0)
)
# Non-linearity bonus: lower R² = more valuable in EBM
# Invert R² so complex shapes get higher score
composite_df['nonlin_bonus'] = 1 - composite_df['linearity_r2'].fillna(0.5)
composite_df['norm_nonlin'] = MinMaxScaler().fit_transform(
    composite_df[['nonlin_bonus']]
)
# Weighted composite score:
#   Importance (0.30) — how much the EBM relies on it
#   Effect magnitude (0.20) — does it move the score?
#   Non-linearity (0.20) — does EBM add value over linear models?
#   Class separation (0.15 r + 0.15 KS) — signal strength
composite_df['composite_score'] = (
    0.30 * composite_df['norm_imp'] +
    0.20 * composite_df['norm_range'] +
    0.20 * composite_df['norm_nonlin'] +
    0.15 * composite_df['norm_sep_r'] +
    0.15 * composite_df['norm_sep_ks']
)
# Interaction bonus: +0.05 for features participating in interactions
composite_df.loc[composite_df['in_interaction'], 'composite_score'] += 0.05
composite_df = composite_df.sort_values('composite_score', ascending=False).reset_index(drop=True)
composite_df['composite_rank'] = range(1, len(composite_df) + 1)
# Assign tiers (EBM has more features, so wider tiers)
def assign_tier_ebm(row):
    if row['composite_rank'] <= 15:
        return '🟢 TIER 1 (keep)'
    elif row['composite_rank'] <= 25:
        return '🟡 TIER 2 (likely keep)'
    elif row['composite_rank'] <= 40:
        return '🟠 TIER 3 (test removal)'
    else:
        return '🔴 TIER 4 (safe to cut)'
composite_df['tier'] = composite_df.apply(assign_tier_ebm, axis=1)
print(f"\n   Composite = 0.30×Importance + 0.20×EffectRange + 0.20×NonLinearity "
      f"+ 0.15×ClassSep_r + 0.15×ClassSep_KS (+0.05 interaction bonus)")
print(f"   (Non-linearity rewards complex shapes — where EBM adds unique value)\n")
for _, row in composite_df.iterrows():
    int_flag = " 🔗" if row['in_interaction'] else "   "
    ix_flag = " [IX]" if row['is_interaction_term'] else ""
    print(f"   {row['composite_rank']:2d}. {row['feature']:40s} "
          f"Score={row['composite_score']:.3f}  "
          f"Imp=#{int(row['imp_rank']):2d}  R²={row['linearity_r2']:.2f}  "
          f"Range={row['score_range']:.4f}{int_flag}{ix_flag}  {row['tier']}")
tier_counts = composite_df['tier'].value_counts().sort_index()
print(f"\n   📊 Tier distribution:")
for tier, count in tier_counts.items():
    print(f"      {tier}: {count} features")
# ============================================================
# 📊 VISUALIZATIONS
# ============================================================
print("\n📊 Generating diagnostic plots...")
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 3, figsize=(22, 14))
# ── Plot 1: Cumulative importance curve ──
ax = axes[0, 0]
ax.plot(range(1, len(importance_df) + 1), importance_df['imp_cumulative_pct'].values, 
        'b-o', markersize=3, linewidth=1.5)
ax.axhline(y=0.80, color='green', linestyle='--', alpha=0.7, label='80%')
ax.axhline(y=0.90, color='orange', linestyle='--', alpha=0.7, label='90%')
ax.axhline(y=0.95, color='red', linestyle='--', alpha=0.7, label='95%')
ax.axvline(x=n_for_80, color='green', linestyle=':', alpha=0.5)
ax.axvline(x=n_for_90, color='orange', linestyle=':', alpha=0.5)
ax.axvline(x=n_for_95, color='red', linestyle=':', alpha=0.5)
ax.set_xlabel('Number of Features (ranked by importance)')
ax.set_ylabel('Cumulative Importance %')
ax.set_title(f'Cumulative Importance Curve\n80%@{n_for_80}, 90%@{n_for_90}, 95%@{n_for_95} features')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
# ── Plot 2: Importance vs Linearity scatter ──
ax = axes[0, 1]
colors = ['#2ecc71' if t.startswith('🟢') else '#f1c40f' if t.startswith('🟡')
          else '#e67e22' if t.startswith('🟠') else '#e74c3c'
          for t in composite_df['tier']]
scatter_merge = composite_df.merge(linearity_df[['feature', 'linearity_r2', 'score_range']], 
                                    on='feature', suffixes=('', '_dup'))
ax.scatter(scatter_merge['linearity_r2'], scatter_merge['importance'],
           c=colors, s=60, alpha=0.8, edgecolors='black', linewidth=0.5)
# Only annotate main effects in top 10
for _, row in scatter_merge.head(10).iterrows():
    if ' & ' not in row['feature']:
        ax.annotate(row['feature'], (row['linearity_r2'], row['importance']),
                    fontsize=6, alpha=0.8, rotation=10)
ax.axvline(x=0.90, color='red', linestyle='--', alpha=0.5, label='Linear threshold (R²=0.9)')
ax.set_xlabel('Shape Function Linearity (R²)')
ax.set_ylabel('Global Importance')
ax.set_title('Importance vs Linearity\n(top-right = important but linear → may be redundant)')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)
# ── Plot 3: Composite score bar chart with tiers ──
ax = axes[0, 2]
tier_colors = {'🟢 TIER 1 (keep)': '#2ecc71', '🟡 TIER 2 (likely keep)': '#f1c40f',
               '🟠 TIER 3 (test removal)': '#e67e22', '🔴 TIER 4 (safe to cut)': '#e74c3c'}
bar_colors = [tier_colors.get(t, '#95a5a6') for t in composite_df['tier']]
ax.barh(range(len(composite_df)), composite_df['composite_score'], 
        color=bar_colors, edgecolor='black', linewidth=0.3)
ax.set_yticks(range(len(composite_df)))
ax.set_yticklabels(composite_df['feature'], fontsize=6)
ax.set_xlabel('Composite Score')
ax.set_title('Feature Composite Ranking (by tier)')
ax.invert_yaxis()
# ── Plot 4: Score range (effect magnitude) bar chart ──
ax = axes[1, 0]
mag_sorted = magnitude_df.sort_values('score_range', ascending=True)
mag_colors = ['#e74c3c' if sr < 0.01 else '#f1c40f' if sr < 0.03 
              else '#2ecc71' for sr in mag_sorted['score_range']]
ax.barh(range(len(mag_sorted)), mag_sorted['score_range'], 
        color=mag_colors, edgecolor='black', linewidth=0.3)
ax.set_yticks(range(len(mag_sorted)))
ax.set_yticklabels(mag_sorted['feature'], fontsize=6)
ax.axvline(x=0.01, color='red', linestyle='--', alpha=0.7, label='Negligible (<0.01)')
ax.axvline(x=0.03, color='orange', linestyle='--', alpha=0.7, label='Weak (<0.03)')
ax.set_xlabel('Score Range (Effect Magnitude)')
ax.set_title('Effect Magnitude per Feature')
ax.legend(fontsize=7)
# ── Plot 5: Correlation heatmap (top 20 MAIN EFFECTS only) ──
ax = axes[1, 1]
# Filter out interaction terms — they don't exist as columns in diag_X
main_effect_ranked = composite_df[~composite_df['is_interaction_term']]['feature'].tolist()
top20_feats = [f for f in main_effect_ranked if f in diag_X.columns][:20]
top20_corr = diag_X[top20_feats].corr()
mask = np.triu(np.ones_like(top20_corr, dtype=bool), k=1)
sns.heatmap(top20_corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            mask=mask, square=True, ax=ax, cbar_kws={'shrink': 0.8},
            annot_kws={'size': 5})
ax.set_title(f'Correlation Matrix (Top 20 main effects)', fontsize=11)
ax.tick_params(axis='both', labelsize=5)
# ── Plot 6: Linearity R² distribution ──
ax = axes[1, 2]
r2_vals = linearity_df['linearity_r2'].dropna()
ax.hist(r2_vals, bins=20, color='#3498db', edgecolor='black', alpha=0.7)
ax.axvline(x=0.90, color='red', linestyle='--', linewidth=2, label='Linear threshold (0.90)')
ax.axvline(x=0.40, color='orange', linestyle='--', linewidth=2, label='Complex threshold (0.40)')
ax.set_xlabel('Shape Function Linearity (R²)')
ax.set_ylabel('Count')
ax.set_title(f'Linearity Distribution\n{len(highly_linear)} linear (R²>0.9), '
             f'{len(complex_shapes)} complex (R²<0.4)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
# ============================================================
# 📋 COPY-PASTE HELPER: Reduced feature set after Batch 1 cut
# ============================================================
if batch1:
    reduced_features = [f for f in EBM_FEATURES if f not in batch1]
    print(f"\n" + "="*80)
    print(f"📋 COPY-PASTE: EBM_FEATURES after Batch 1 removal ({len(reduced_features)} features)")
    print(f"="*80)
    print(f"\nEBM_FEATURES = [")
    for f in reduced_features:
        tier = composite_df[composite_df['feature'] == f]['tier'].values[0]
        rank = int(composite_df[composite_df['feature'] == f]['composite_rank'].values[0])
        int_flag = "  # 🔗 in interaction" if f in interacting_features else ""
        print(f"    '{f}',  # #{rank} {tier}{int_flag}")
    print(f"]")
print("\n" + "="*80)
print("✅ EBM Diagnostics complete")
print("\n" + "="*80)


🔬 EBM FEATURE DIAGNOSTICS
   Feature set: 14 features
   Model source: 14B (Optuna, balanced)
   Model config: interactions=15, learning_rate=0.026535810872734773, max_bins=288, max_rounds=4700

--------------------------------------------------------------------------------
1️⃣  GLOBAL FEATURE IMPORTANCE (Main Effects)
--------------------------------------------------------------------------------
    1. decay_x_density                          0.3069  cum=12.6%  ██████████████████████████████  🔥 TOP-80%
    2. dow_month_encoded                        0.2855  cum=24.4%  ███████████████████████████  🔥 TOP-80%
    3. cpi_high_cellular                        0.2426  cum=34.4%  ███████████████████████  🔥 TOP-80%
    4. behavioral_favorability                  0.2096  cum=43.0%  ████████████████████  🔥 TOP-80%
    5. overlap_default_clean                    0.1954  cum=51.1%  ███████████████████  🔥 TOP-80%
    6. cons.conf.idx                            0.1769  cum=58.4%  ███████████████

In [92]:
# ================================================================
# CELL 15: META-ARBITER EXPERIMENT 
# ================================================================
print("=" * 80)
print("🧪 META-ARBITER EXPERIMENT")
print("=" * 80)
# ============================================================
# 1. Stages 1 - 3 FEATURE SETS 
# ============================================================
print(f"\n📊 Feature Sets: LR={len(LR_FEATURES)}, "
      f"RF={len(RF_FEATURES)}, EBM={len(EBM_FEATURES)}")
print("\n" + "-" * 80)
print("🔍 CHECKING FOR OPTUNA-TUNED PARAMETERS")
print("-" * 80)
# ── LR params — uses RECALL-BIASED Optuna (lr_recall_optuna_params) ───────────
LR_HARDCODED = dict(C=1.0, class_weight='balanced', solver='lbfgs', max_iter=1000, random_state=42)
LR_OPTUNA = None
try:
    _bp = lr_recall_optuna_params
    _C = _bp.get('C', 1.0)
    _pen = _bp.get('penalty', 'l2')
    _mw = _bp.get('minority_weight', None)
    if _mw is not None:
        _cw = {0: 1, 1: _mw}
    else:
        _cw = _bp.get('class_weight', 'balanced')
        if _cw == 'None': _cw = None
    LR_OPTUNA = dict(C=_C, penalty=_pen, solver='saga', max_iter=5000, random_state=42,
                     class_weight=_cw)
    if _pen == 'elasticnet':
        LR_OPTUNA['l1_ratio'] = _bp.get('l1_ratio', 0.5)
    _mw_str = f"{_mw:.2f}" if _mw is not None else 'N/A'
    print(f"   ✅ LR Recall-Biased Optuna params found: C={_C:.4f}, penalty={_pen}, "
          f"minority_weight={_mw_str}, class_weight={_cw}")
except NameError:
    # Fallback to balanced optuna params if recall-biased not available
    try:
        _bp = lr_optuna_params  # Cell 12B fallback
        _C = _bp.get('C', 1.0)
        _pen = _bp.get('penalty', 'l2')
        _cw = _bp.get('class_weight', 'balanced')
        if _cw == 'None': _cw = None
        LR_OPTUNA = dict(C=_C, penalty=_pen, solver='saga', max_iter=5000, random_state=42,
                         class_weight=_cw)
        if _pen == 'elasticnet':
            LR_OPTUNA['l1_ratio'] = _bp.get('l1_ratio', 0.5)
        print(f"   ⚠️  lr_recall_optuna_params not found — falling back to balanced LR Optuna")
        print(f"      C={_C:.4f}, penalty={_pen}, class_weight={_cw}")
    except NameError:
        print(f"   ℹ️  No LR Optuna params found (run Cell 12B or 12C first) — hardcoded only")

# ── RF params ─────────────────────────────────────────────────
RF_HARDCODED = dict(
    n_estimators=200, max_depth=12, min_samples_leaf=5, min_samples_split=10,
    max_features='sqrt', class_weight='balanced_subsample', random_state=42, n_jobs=-1
)
RF_OPTUNA = None
try:
    _bp = rf_optuna_params  # set at end of Cell 13B
    _md = _bp.get('max_depth', None)
    if 'use_max_depth' in _bp and not _bp.get('use_max_depth', True):
        _md = None
    if _bp.get('max_features_type') == 'fraction':
        _mf = _bp.get('max_features_fraction', 0.5)
    elif _bp.get('max_features_type') in ('sqrt', 'log2'):
        _mf = _bp['max_features_type']
    else:
        _mf = _bp.get('max_features', 'sqrt')
    _cw = _bp.get('class_weight', None)
    if _cw == 'None': _cw = None
    RF_OPTUNA = dict(
        n_estimators=_bp['n_estimators'], max_depth=_md,
        min_samples_leaf=_bp.get('min_samples_leaf', 5),
        max_features=_mf, class_weight=_cw,
        random_state=42, n_jobs=-1
    )
    print(f"   ✅ RF Optuna params found: n_est={_bp['n_estimators']}, "
          f"depth={_md}, leaf={_bp.get('min_samples_leaf', 5)}")
except NameError:
    print(f"   ℹ️  RF Optuna params not found (rf_optuna_params missing) — hardcoded only")

# ── EBM params ────────────────────────────────────────────────
EBM_HARDCODED = dict(
    max_bins=512, max_interaction_bins=64, interactions=15,
    learning_rate=0.02, min_samples_leaf=5, max_leaves=5,
    n_jobs=1, random_state=42
)
EBM_OPTUNA = None
try:
    _bp = ebm_optuna_params  # set at end of Cell 14B
    EBM_OPTUNA = dict(
        learning_rate=_bp['learning_rate'], max_rounds=_bp['max_rounds'],
        max_bins=_bp['max_bins'], max_interaction_bins=_bp['max_interaction_bins'],
        interactions=_bp['interactions'], n_jobs=1, random_state=42
    )
    print(f"   ✅ EBM Optuna params found: lr={_bp['learning_rate']:.4f}, "
          f"rounds={_bp['max_rounds']}, interactions={_bp['interactions']}")
except NameError:
    print(f"   ℹ️  EBM Optuna params not found (ebm_optuna_params missing) — hardcoded only")

# ============================================================
# 3. PREPARE DATA
# ============================================================
print("\n" + "-" * 80)
print("1️⃣  PREPARING DATA")
print("-" * 80)
y_all = df_engineered[TARGET_COL].astype(int)
for name, feats in [("LR", LR_FEATURES), ("RF", RF_FEATURES), ("EBM", EBM_FEATURES)]:
    missing = [f for f in feats if f not in df_engineered.columns]
    print(f"   {'✅' if not missing else '⚠️'}  {name}: {len(feats)} features "
          f"{'— missing: ' + str(missing) if missing else 'OK'}")
train_idx, test_idx = train_test_split(
    np.arange(len(df_engineered)), test_size=0.2, random_state=42, stratify=y_all
)
y_train = y_all.iloc[train_idx].values
y_test  = y_all.iloc[test_idx].values
print(f"   Train: {len(y_train):,} (pos: {y_train.sum():,})  "
      f"Test: {len(y_test):,} (pos: {y_test.sum():,})")

# ============================================================
# 4. THRESHOLD HELPER — recall-targeted for ALL models
# ============================================================
def find_recall_threshold(y_true, y_prob, target_recall=0.70):
    """Find threshold targeting a specific recall on the train set."""
    precs, recs, threshs = precision_recall_curve(y_true, y_prob)
    idx = np.argmin(np.abs(recs[:-1] - target_recall))
    return threshs[idx]

# ============================================================
# 5. TRAIN MODELS (hardcoded + Optuna)
# ============================================================
print("\n" + "-" * 80)
print("2️⃣  TRAINING MODELS")
print("-" * 80)

def _train_lr(params, label):
    print(f"\n   🔵 LR ({label})...")
    scaler_lr = StandardScaler()
    X_tr = scaler_lr.fit_transform(df_engineered.iloc[train_idx][LR_FEATURES].values)
    X_te = scaler_lr.transform(df_engineered.iloc[test_idx][LR_FEATURES].values)
    model = LogisticRegression(**params)
    model.fit(X_tr, y_train)
    p_tr = model.predict_proba(X_tr)[:, 1]
    p_te = model.predict_proba(X_te)[:, 1]
    t = find_recall_threshold(y_train, p_tr, target_recall=0.70)
    pred_te = (p_te >= t).astype(int)
    print(f"      thresh={t:.4f}  Acc={accuracy_score(y_test, pred_te):.4f}  "
          f"Rec={recall_score(y_test, pred_te):.4f}  "
          f"Prec={precision_score(y_test, pred_te):.4f}  "
          f"AUC={roc_auc_score(y_test, p_te):.4f}")
    for feat, coef in sorted(zip(LR_FEATURES, model.coef_[0]), key=lambda x: -abs(x[1])):
        print(f"         {'➕' if coef > 0 else '➖'} {feat:<35} {coef:>+.4f}")
    return model, p_tr, p_te, t, pred_te

def _train_rf(params, label):
    print(f"\n   🌲 RF ({label})...")
    X_tr = df_engineered.iloc[train_idx][RF_FEATURES].values
    X_te = df_engineered.iloc[test_idx][RF_FEATURES].values
    model = RandomForestClassifier(**params)
    model.fit(X_tr, y_train)
    p_tr = model.predict_proba(X_tr)[:, 1]
    p_te = model.predict_proba(X_te)[:, 1]
    t = find_recall_threshold(y_train, p_tr, target_recall=0.70)
    pred_te = (p_te >= t).astype(int)
    print(f"      thresh={t:.4f}  Acc={accuracy_score(y_test, pred_te):.4f}  "
          f"Rec={recall_score(y_test, pred_te):.4f}  "
          f"Prec={precision_score(y_test, pred_te):.4f}  "
          f"AUC={roc_auc_score(y_test, p_te):.4f}")
    return model, p_tr, p_te, t, pred_te

def _train_ebm(params, label):
    print(f"\n   🔮 EBM ({label})...")
    X_tr = df_engineered.iloc[train_idx][EBM_FEATURES].values
    X_te = df_engineered.iloc[test_idx][EBM_FEATURES].values
    try:
        from interpret.glassbox import ExplainableBoostingClassifier
        model = ExplainableBoostingClassifier(**params)
        model.fit(X_tr, y_train)
        p_tr = model.predict_proba(X_tr)[:, 1]
        p_te = model.predict_proba(X_te)[:, 1]
    except ImportError:
        print("      ⚠️ interpret not installed — skipping EBM")
        return None, None, None, 0.50, None, False
    t = find_recall_threshold(y_train, p_tr, target_recall=0.70)
    pred_te = (p_te >= t).astype(int)
    print(f"      thresh={t:.4f}  Acc={accuracy_score(y_test, pred_te):.4f}  "
          f"Rec={recall_score(y_test, pred_te):.4f}  "
          f"Prec={precision_score(y_test, pred_te):.4f}  "
          f"AUC={roc_auc_score(y_test, p_te):.4f}")
    return model, p_tr, p_te, t, pred_te, True

# ── Train hardcoded versions ──────────────────────────────────
lr_model_h,  lr_ptr_h,  lr_pte_h,  lr_t_h,  lr_pred_h  = _train_lr(LR_HARDCODED,  "hardcoded")
rf_model_h,  rf_ptr_h,  rf_pte_h,  rf_t_h,  rf_pred_h  = _train_rf(RF_HARDCODED,  "hardcoded")
ebm_result_h = _train_ebm(EBM_HARDCODED, "hardcoded")
ebm_model_h, ebm_ptr_h, ebm_pte_h, ebm_t_h, ebm_pred_h, HAS_EBM = ebm_result_h

# ── Train Optuna versions (recall-biased LR + tuned RF/EBM) ──
lr_model_o,  lr_ptr_o,  lr_pte_o,  lr_t_o,  lr_pred_o  = (None,)*5
rf_model_o,  rf_ptr_o,  rf_pte_o,  rf_t_o,  rf_pred_o  = (None,)*5
ebm_model_o, ebm_ptr_o, ebm_pte_o, ebm_t_o, ebm_pred_o = (None,)*5

if LR_OPTUNA:
    lr_model_o, lr_ptr_o, lr_pte_o, lr_t_o, lr_pred_o = _train_lr(LR_OPTUNA, "Optuna (recall-biased)")
if RF_OPTUNA:
    rf_model_o, rf_ptr_o, rf_pte_o, rf_t_o, rf_pred_o = _train_rf(RF_OPTUNA, "Optuna")
if EBM_OPTUNA:
    ebm_result_o = _train_ebm(EBM_OPTUNA, "Optuna")
    ebm_model_o, ebm_ptr_o, ebm_pte_o, ebm_t_o, ebm_pred_o = ebm_result_o[:5]

# ============================================================
# 6. SHARED HELPER FUNCTIONS
# ============================================================
def compute_cal(y, p):
    brier = brier_score_loss(y, p)
    bins = np.linspace(0, 1, 11)
    ece = sum(
        abs(y[(p > bins[i]) & (p <= bins[i+1])].mean() -
            p[(p > bins[i]) & (p <= bins[i+1])].mean())
        * ((p > bins[i]) & (p <= bins[i+1])).mean()
        for i in range(10) if ((p > bins[i]) & (p <= bins[i+1])).sum() > 0
    )
    return brier, ece

def arbiter(lr_p, rf_p, ebm_p, lr_t, rf_t, ebm_t, w, abstain=True, min_conf=0.15):
    n = len(lr_p)
    pred, prob = np.zeros(n, dtype=int), np.zeros(n)
    wa = np.array([w['lr'], w['rf'], w['ebm']])
    wa /= wa.sum()
    threshs = np.array([lr_t, rf_t, ebm_t])
    for i in range(n):
        ps    = np.array([lr_p[i], rf_p[i], ebm_p[i]])
        votes = (ps >= threshs).astype(int)
        confs = np.abs(ps - threshs)
        prob[i] = np.sum(ps * wa)
        c1 = np.sum(confs * wa * (votes == 1))
        c0 = np.sum(confs * wa * (votes == 0))
        if abstain and max(c1, c0) < min_conf:
            pred[i] = -1
        elif c1 > c0:
            pred[i] = 1
        else:
            pred[i] = 0
    return pred, prob

def eval_arb(y, pred, prob):
    cov = (pred != -1)
    if cov.sum() == 0:
        return {k: 0 for k in ['coverage','accuracy','recall','precision','f1','f2','auc']}
    yc, pc, prc = y[cov], pred[cov], prob[cov]
    return {
        'coverage':  cov.mean(),
        'accuracy':  accuracy_score(yc, pc),
        'recall':    recall_score(yc, pc, zero_division=0),
        'precision': precision_score(yc, pc, zero_division=0),
        'f1':        f1_score(yc, pc, zero_division=0),
        'f2':        fbeta_score(yc, pc, beta=2, zero_division=0),
        'auc':       roc_auc_score(yc, prc) if len(np.unique(yc)) > 1 else 0
    }

def compute_weights(y, probs_list, preds_list):
    briers  = np.array([brier_score_loss(y, p) for p in probs_list])
    inv_b   = 1.0 / (briers + 1e-6)
    accs    = np.array([accuracy_score(y, p) for p in preds_list])
    hybrid  = 0.5 * (inv_b / inv_b.sum()) + 0.5 * (accs / accs.sum())
    hybrid /= hybrid.sum()
    return {'lr': hybrid[0], 'rf': hybrid[1], 'ebm': hybrid[2]}

def run_arbiter_sweep(lr_p, rf_p, ebm_p, lr_t, rf_t, ebm_t, w, y,
                      mc_range=np.arange(0.03, 0.55, 0.02), min_cov=0.50):
    print(f"\n   {'mc':>6} {'cover':>8} {'acc':>8} {'recall':>8} {'prec':>8} {'f1':>8} {'f2':>8}")
    print(f"   {'-'*58}")
    best_m, best_s, best_c = None, 0, 0
    for mc in mc_range:
        p, pr = arbiter(lr_p, rf_p, ebm_p, lr_t, rf_t, ebm_t, w, True, mc)
        m = eval_arb(y, p, pr)
        s = m['recall'] * 0.4 + m['accuracy'] * 0.3 + m['coverage'] * 0.3
        print(f"   {mc:>6.2f} {m['coverage']:>8.1%} {m['accuracy']:>8.4f} {m['recall']:>8.4f} "
              f"{m['precision']:>8.4f} {m['f1']:>8.4f} {m['f2']:>8.4f}")
        if s > best_s and m['coverage'] >= min_cov:
            best_s, best_c, best_m = s, mc, m.copy()
    return best_m, best_c

def find_youden_threshold(y_true, y_prob):
    fpr, tpr, threshs = roc_curve(y_true, y_prob)
    opt_idx = np.argmax(tpr - fpr)
    return threshs[opt_idx], tpr[opt_idx], fpr[opt_idx]

def find_f2_threshold(y_true, y_prob):
    precs, recs, threshs = precision_recall_curve(y_true, y_prob)
    f2s = (5 * precs[:-1] * recs[:-1]) / (4 * precs[:-1] + recs[:-1] + 1e-10)
    best_idx = np.argmax(f2s)
    return threshs[best_idx], recs[best_idx], precs[best_idx], f2s[best_idx]

def run_full_experiment(lr_ptr, rf_ptr, ebm_ptr, lr_pte, rf_pte, ebm_pte,
                        lr_t, rf_t, ebm_t, y_train, y_test, label=""):
    print(f"\n{'─'*80}")
    print(f"   ⚙️  CONFIG: {label}")
    print(f"   📐 Thresholds: LR={lr_t:.4f}, RF={rf_t:.4f}, EBM={ebm_t:.4f}")
    print(f"{'─'*80}")
    # Disagreement on train set
    lr_pt  = (lr_ptr  >= lr_t).astype(int)
    rf_pt  = (rf_ptr  >= rf_t).astype(int)
    ebm_pt = (ebm_ptr >= ebm_t).astype(int)
    for la, lb, pa, pb in [("LR","RF",lr_pt,rf_pt),
                            ("LR","EBM",lr_pt,ebm_pt),
                            ("RF","EBM",rf_pt,ebm_pt)]:
        dis = (pa != pb)
        if dis.sum():
            a_right = (pa[dis] == y_train[dis]).mean()
            b_right = (pb[dis] == y_train[dis]).mean()
            print(f"   {la} vs {lb}: disagree {dis.mean():.1%} → "
                  f"{la} correct {a_right:.1%}, {lb} correct {b_right:.1%}")
    all_agree = (lr_pt == rf_pt) & (rf_pt == ebm_pt)
    print(f"   All agree: {all_agree.mean():.1%}, "
          f"accuracy: {accuracy_score(y_train[all_agree], lr_pt[all_agree]):.4f}")
    # Calibration + weights
    for name, prob in [("LR", lr_ptr), ("RF", rf_ptr), ("EBM", ebm_ptr)]:
        b, e = compute_cal(y_train, prob)
        print(f"   {name:<5} Brier={b:.4f}  ECE={e:.4f}")
    W = compute_weights(y_train, [lr_ptr, rf_ptr, ebm_ptr], [lr_pt, rf_pt, ebm_pt])
    print(f"   Weights: LR={W['lr']:.3f}, RF={W['rf']:.3f}, EBM={W['ebm']:.3f}")
    # No-abstention
    pred_na, prob_na = arbiter(lr_pte, rf_pte, ebm_pte, lr_t, rf_t, ebm_t, W, abstain=False)
    m_na = eval_arb(y_test, pred_na, prob_na)
    print(f"\n   No-Abstention:")
    for k, v in m_na.items():
        print(f"      {k:<12} {v:.4f}")
    # Abstention sweep
    print(f"\n   Abstention sweep:")
    best_m, best_c = run_arbiter_sweep(lr_pte, rf_pte, ebm_pte, lr_t, rf_t, ebm_t, W, y_test)
    if best_m:
        print(f"\n   ✨ Best: min_conf={best_c:.2f}")
        for k, v in best_m.items():
            print(f"      {k:<12} {v:.4f}")
    return m_na, best_m, best_c, W, prob_na

# ============================================================
# 7. EXPERIMENT 1: RECALL-TUNED THRESHOLDS
# ============================================================
print("\n" + "="*80)
print("📊 EXPERIMENT 1: RECALL-TUNED THRESHOLDS")
print("="*80)
m_na_h, best_m_h, best_c_h, W_h, prob_na_h = run_full_experiment(
    lr_ptr_h, rf_ptr_h, ebm_ptr_h, lr_pte_h, rf_pte_h, ebm_pte_h,
    lr_t_h, rf_t_h, ebm_t_h, y_train, y_test,
    label="Hardcoded params, recall-tuned thresholds"
)
m_na_o = best_m_o = best_c_o = W_o = prob_na_o = None
_optuna_available = {
    'LR':  lr_pte_o  is not None,
    'RF':  rf_pte_o  is not None,
    'EBM': ebm_pte_o is not None,
}
_have_all_optuna  = all(_optuna_available.values())
_have_names   = [n for n, ok in _optuna_available.items() if ok]
_missing_names = [n for n, ok in _optuna_available.items() if not ok]
if _have_all_optuna:
    m_na_o, best_m_o, best_c_o, W_o, prob_na_o = run_full_experiment(
        lr_ptr_o, rf_ptr_o, ebm_ptr_o, lr_pte_o, rf_pte_o, ebm_pte_o,
        lr_t_o, rf_t_o, ebm_t_o, y_train, y_test,
        label="Optuna params (LR recall-biased + RF/EBM tuned), recall-tuned thresholds"
    )
else:
    print(f"\n   ⚠️  Optuna comparison requires all 3 models.")
    print(f"      Have: {_have_names or 'none'}")
    print(f"      Missing: {_missing_names}")

# ============================================================
# 8. EXPERIMENT 2: YOUDEN + F2 THRESHOLDS
# ============================================================
print("\n" + "="*80)
print("📊 EXPERIMENT 2: YOUDEN & F2 THRESHOLDS")
print("="*80)
print("\n" + "-"*80)
print("OPTIMAL THRESHOLDS (from train set)")
print("-"*80)
lr_t_y,  lr_tpr_y,  lr_fpr_y  = find_youden_threshold(y_train, lr_ptr_h)
rf_t_y,  rf_tpr_y,  rf_fpr_y  = find_youden_threshold(y_train, rf_ptr_h)
ebm_t_y, ebm_tpr_y, ebm_fpr_y = find_youden_threshold(y_train, ebm_ptr_h)
lr_t_f2,  lr_rec_f2,  lr_prec_f2,  lr_f2_f2  = find_f2_threshold(y_train, lr_ptr_h)
rf_t_f2,  rf_rec_f2,  rf_prec_f2,  rf_f2_f2  = find_f2_threshold(y_train, rf_ptr_h)
ebm_t_f2, ebm_rec_f2, ebm_prec_f2, ebm_f2_f2 = find_f2_threshold(y_train, ebm_ptr_h)
print(f"\n   {'Model':<8} {'Youden':>8} {'TPR':>8} {'FPR':>8}  |  {'F2':>8} {'Recall':>8} {'Prec':>8}")
print(f"   {'-'*65}")
print(f"   {'LR':<8} {lr_t_y:>8.4f} {lr_tpr_y:>8.4f} {lr_fpr_y:>8.4f}  |  "
      f"{lr_t_f2:>8.4f} {lr_rec_f2:>8.4f} {lr_prec_f2:>8.4f}")
print(f"   {'RF':<8} {rf_t_y:>8.4f} {rf_tpr_y:>8.4f} {rf_fpr_y:>8.4f}  |  "
      f"{rf_t_f2:>8.4f} {rf_rec_f2:>8.4f} {rf_prec_f2:>8.4f}")
print(f"   {'EBM':<8} {ebm_t_y:>8.4f} {ebm_tpr_y:>8.4f} {ebm_fpr_y:>8.4f}  |  "
      f"{ebm_t_f2:>8.4f} {ebm_rec_f2:>8.4f} {ebm_prec_f2:>8.4f}")
m_na_y, best_m_y, best_c_y, W_y, prob_na_y = run_full_experiment(
    lr_ptr_h, rf_ptr_h, ebm_ptr_h, lr_pte_h, rf_pte_h, ebm_pte_h,
    lr_t_y, rf_t_y, ebm_t_y, y_train, y_test,
    label="Hardcoded params, Youden thresholds"
)
m_na_f2, best_m_f2, best_c_f2, W_f2, prob_na_f2 = run_full_experiment(
    lr_ptr_h, rf_ptr_h, ebm_ptr_h, lr_pte_h, rf_pte_h, ebm_pte_h,
    lr_t_f2, rf_t_f2, ebm_t_f2, y_train, y_test,
    label="Hardcoded params, F2 thresholds"
)

# ============================================================
# 9. GRAND COMPARISON TABLE
# ============================================================
print("\n" + "="*80)
print("🏆 GRAND COMPARISON")
print("="*80)
print(f"\n   {'Config':<40} {'Acc':>7} {'Recall':>7} {'Prec':>7} {'F1':>7} {'F2':>7} {'AUC':>7} {'Cover':>7}")
print(f"   {'-'*89}")
rows = [
    ("Hardcoded + Recall (full)",     m_na_h),
    ("Hardcoded + Recall (abstain)",  best_m_h),
]
if m_na_o:
    rows.append(("Optuna (LR recall-biased) full",   m_na_o))
if best_m_o:
    rows.append(("Optuna (LR recall-biased) abstain", best_m_o))
rows += [
    ("Hardcoded + Youden (full)",     m_na_y),
    ("Hardcoded + Youden (abstain)",  best_m_y),
    ("Hardcoded + F2 (full)",         m_na_f2),
    ("Hardcoded + F2 (abstain)",      best_m_f2),
]
for label, m in rows:
    if m is None: continue
    cov = m.get('coverage', 1.0)
    cov_str = f"{cov:>6.1%}" if cov < 1.0 else " 100.0%"
    print(f"   {label:<40} {m['accuracy']:>7.4f} {m['recall']:>7.4f} "
          f"{m['precision']:>7.4f} {m['f1']:>7.4f} {m['f2']:>7.4f} "
          f"{m['auc']:>7.4f} {cov_str}")

print(f"\n   {'-'*89}")
print(f"   {'INDIVIDUAL MODELS':<40}")
print(f"   {'-'*89}")
for name, pred, prob, thresh in [
    ("LR (hardcoded)",  lr_pred_h,  lr_pte_h,  lr_t_h),
    ("RF (hardcoded)",  rf_pred_h,  rf_pte_h,  rf_t_h),
    ("EBM (hardcoded)", ebm_pred_h, ebm_pte_h, ebm_t_h),
]:
    print(f"   {name+f' t={thresh:.3f}':<40} {accuracy_score(y_test, pred):>7.4f} "
          f"{recall_score(y_test, pred):>7.4f} "
          f"{precision_score(y_test, pred):>7.4f} {f1_score(y_test, pred):>7.4f} "
          f"{fbeta_score(y_test, pred, beta=2):>7.4f} {roc_auc_score(y_test, prob):>7.4f}  100.0%")

for name, pred, prob, thresh in [
    x for x in [
        ("LR (Optuna recall-biased)", lr_pred_o,  lr_pte_o,  lr_t_o)  if lr_pred_o  is not None else None,
        ("RF (Optuna)",               rf_pred_o,  rf_pte_o,  rf_t_o)  if rf_pred_o  is not None else None,
        ("EBM (Optuna)",              ebm_pred_o, ebm_pte_o, ebm_t_o) if ebm_pred_o is not None else None,
    ] if x is not None
]:
    print(f"   {name+f' t={thresh:.3f}':<40} {accuracy_score(y_test, pred):>7.4f} "
          f"{recall_score(y_test, pred):>7.4f} "
          f"{precision_score(y_test, pred):>7.4f} {f1_score(y_test, pred):>7.4f} "
          f"{fbeta_score(y_test, pred, beta=2):>7.4f} {roc_auc_score(y_test, prob):>7.4f}  100.0%")

print(f"\n   Recall @ 10% FPR:")
all_probs = [("LR", lr_pte_h), ("RF", rf_pte_h), ("EBM", ebm_pte_h), ("Arbiter (HC)", prob_na_h)]
if prob_na_o is not None:
    all_probs.append(("Arbiter (Optuna recall-LR)", prob_na_o))
for name, prob in all_probs:
    fpr, tpr, _ = roc_curve(y_test, prob)
    print(f"      {name:<30} {tpr[np.argmin(np.abs(fpr - 0.10))]:.4f}")

print(f"\n   📌 BASELINE (from cascade with GLASS-BRW):")
print(f"   {'Baseline no-abstain':<40} {'0.8687':>7} {'0.6002':>7} {'0.4393':>7} "
      f"{'0.5073':>7} {'---':>7} {'0.8016':>7} {'100%':>7}")
print(f"   {'Baseline abstain (75.8%)':<40} {'0.9077':>7} {'0.6000':>7} {'0.5165':>7} "
      f"{'0.5551':>7} {'---':>7} {'0.7961':>7} {'75.8%':>7}")

print("\n" + "="*80)
print("🎉 META-ARBITER EXPERIMENT COMPLETE")
print("="*80)

🧪 META-ARBITER EXPERIMENT

📊 Feature Sets: LR=3, RF=29, EBM=14

--------------------------------------------------------------------------------
🔍 CHECKING FOR OPTUNA-TUNED PARAMETERS
--------------------------------------------------------------------------------
   ✅ LR Recall-Biased Optuna params found: C=0.1998, penalty=elasticnet, minority_weight=9.87, class_weight={0: 1, 1: 9.865540928555221}
   ✅ RF Optuna params found: n_est=800, depth=9, leaf=10
   ✅ EBM Optuna params found: lr=0.0265, rounds=4700, interactions=15

--------------------------------------------------------------------------------
1️⃣  PREPARING DATA
--------------------------------------------------------------------------------
   ✅  LR: 3 features OK
   ✅  RF: 29 features OK
   ✅  EBM: 14 features OK
   Train: 32,950 (pos: 3,712)  Test: 8,238 (pos: 928)

--------------------------------------------------------------------------------
2️⃣  TRAINING MODELS
--------------------------------------------------------

In [99]:
# ================================================================
# CELL 16: VENN DIAGRAM TRACE — BITMASK ARBITER FOUNDATION
# ================================================================
# Priority order for each model:
#   1. Recall-biased Optuna (if available)
#   2. Balanced Optuna (if available)
#   3. Hardcoded fallback
#
# RF special handling: recall-biased RF uses df_binned + 
# RF_FEATURES_BINARY, not df_engineered + RF_FEATURES.
# ================================================================
print("=" * 80)
print("🔬 VENN DIAGRAM TRACE — MODEL DISAGREEMENT ANALYSIS")
print("=" * 80)
print("   Checking if disagreement is STRUCTURED (useful) or NOISE (useless)")
print("   Each sample gets a Membership Bitmask: [LR, RF, EBM] → Correct/Wrong")
print("=" * 80)

import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib_venn import venn3
from collections import Counter

FIGURES_DIR = r"C:\Users\phill\projects\ai-business-coach\prototype\glass_pipeline\research_logs\figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

# ============================================================
# 0. RESOLVE WHICH MODEL VARIANTS TO USE
# ============================================================
# --- LR ---
# Priority: recall-biased Optuna > balanced Optuna > hardcoded
try:
    _lr_source = "Optuna recall-biased"
    _lr_pred_use = lr_pred_o
    _lr_pte_use  = lr_pte_o
    _lr_t_use    = lr_t_o
    assert _lr_pred_use is not None
except (NameError, AssertionError):
    _lr_source = "hardcoded"
    _lr_pred_use = lr_pred_h
    _lr_pte_use  = lr_pte_h
    _lr_t_use    = lr_t_h

# --- RF ---
# Priority: recall-biased binary > balanced binary > balanced Optuna > hardcoded
# The recall-biased RF uses df_binned[RF_FEATURES_BINARY], so we need to
# regenerate predictions on test set from that model.
_rf_source = None

# Try recall-biased binary RF first
try:
    assert rf_bin_recall_clf is not None
    assert RF_FEATURES_BINARY is not None
    _rf_test_X = df_binned.iloc[test_idx][RF_FEATURES_BINARY].copy()
    _rf_pte_use = rf_bin_recall_clf.predict_proba(_rf_test_X)[:, 1]
    _rf_t_use = np.mean(rf_binr_thresh_scores)
    _rf_pred_use = (_rf_pte_use >= _rf_t_use).astype(int)
    _rf_source = "Recall-biased binary (29-bin)"
except (NameError, AssertionError, Exception) as e:
    pass

# Try balanced binary RF
if _rf_source is None:
    try:
        assert rf_bin_best_pipe is not None
        assert RF_FEATURES_BINARY is not None
        _rf_test_X = df_binned.iloc[test_idx][RF_FEATURES_BINARY].copy()
        _rf_pte_use = rf_bin_best_pipe.predict_proba(_rf_test_X)[:, 1]
        _rf_t_use = rf_t_o if rf_t_o is not None else 0.5
        _rf_pred_use = (_rf_pte_use >= _rf_t_use).astype(int)
        _rf_source = "Balanced binary (29-bin)"
    except (NameError, AssertionError, Exception):
        pass

# Try continuous Optuna RF
if _rf_source is None:
    try:
        assert rf_pred_o is not None
        _rf_pred_use = rf_pred_o
        _rf_pte_use  = rf_pte_o
        _rf_t_use    = rf_t_o
        _rf_source = "Optuna continuous"
    except (NameError, AssertionError):
        pass

# Hardcoded fallback
if _rf_source is None:
    _rf_pred_use = rf_pred_h
    _rf_pte_use  = rf_pte_h
    _rf_t_use    = rf_t_h
    _rf_source = "hardcoded"

# --- EBM ---
# Priority: Optuna > hardcoded
try:
    _ebm_source = "Optuna"
    _ebm_pred_use = ebm_pred_o
    _ebm_pte_use  = ebm_pte_o
    _ebm_t_use    = ebm_t_o
    assert _ebm_pred_use is not None
except (NameError, AssertionError):
    _ebm_source = "hardcoded"
    _ebm_pred_use = ebm_pred_h
    _ebm_pte_use  = ebm_pte_h
    _ebm_t_use    = ebm_t_h

print(f"\n   Using:")
print(f"      LR  = {_lr_source}")
print(f"      RF  = {_rf_source}")
print(f"      EBM = {_ebm_source}")
print(f"   Thresholds: LR={_lr_t_use:.4f}, RF={_rf_t_use:.4f}, EBM={_ebm_t_use:.4f}")

# ============================================================
# 1. BUILD BITMASK TABLE ON TEST SET
# ============================================================
_lr_correct  = (_lr_pred_use  == y_test).astype(int)
_rf_correct  = (_rf_pred_use  == y_test).astype(int)
_ebm_correct = (_ebm_pred_use == y_test).astype(int)

def bitmask_label(lr, rf, ebm):
    return f"LR={'✓' if lr else '✗'} RF={'✓' if rf else '✗'} EBM={'✓' if ebm else '✗'}"

bitmask_counts = Counter(
    bitmask_label(l, r, e)
    for l, r, e in zip(_lr_correct, _rf_correct, _ebm_correct)
)

print(f"\n{'─'*80}")
print("📊 BITMASK DISTRIBUTION (all samples)")
print(f"{'─'*80}")
print(f"   {'Pattern':<30} {'Count':>8} {'%':>8}  {'Class Balance'}")
print(f"   {'-'*65}")
total = len(y_test)
for pattern, count in sorted(bitmask_counts.items(), key=lambda x: -x[1]):
    pct = count / total * 100
    mask = np.array([
        bitmask_label(l, r, e) == pattern
        for l, r, e in zip(_lr_correct, _rf_correct, _ebm_correct)
    ])
    pos_rate = y_test[mask].mean()
    bar = "█" * int(pct / 2)
    print(f"   {pattern:<30} {count:>8,} {pct:>7.1f}%  pos={pos_rate:.1%}  {bar}")

# ============================================================
# 2. AGREEMENT RATE SUMMARY
# ============================================================
all_agree_mask = (_lr_correct == _rf_correct) & (_rf_correct == _ebm_correct)
all_correct    = (_lr_correct == 1) & (_rf_correct == 1) & (_ebm_correct == 1)
all_wrong      = (_lr_correct == 0) & (_rf_correct == 0) & (_ebm_correct == 0)
any_disagree   = ~all_agree_mask

print(f"\n{'─'*80}")
print("📈 AGREEMENT SUMMARY")
print(f"{'─'*80}")
print(f"   All 3 agree      : {all_agree_mask.mean():.1%}  ({all_agree_mask.sum():,} samples)")
print(f"     → All correct  : {all_correct.mean():.1%}  ({all_correct.sum():,} samples)")
print(f"     → All wrong    : {all_wrong.mean():.1%}  ({all_wrong.sum():,} samples)")
print(f"   Any disagreement : {any_disagree.mean():.1%}  ({any_disagree.sum():,} samples)")

# ============================================================
# 3. GOLD SAMPLES — unique winners
# ============================================================
print(f"\n{'─'*80}")
print("🥇 GOLD SAMPLES — Unique Winners (one model correct, others wrong)")
print(f"{'─'*80}")
print("   These are your most valuable disagreement cases\n")

for name, correct, others in [
    ("LR only correct",  _lr_correct,  [_rf_correct,  _ebm_correct]),
    ("RF only correct",  _rf_correct,  [_lr_correct,  _ebm_correct]),
    ("EBM only correct", _ebm_correct, [_lr_correct,  _rf_correct]),
]:
    mask = (correct == 1) & np.array([o == 0 for o in others]).all(axis=0)
    if mask.sum() == 0:
        print(f"   {name:<20}: 0 samples")
        continue
    pos_rate = y_test[mask].mean()
    pct = mask.mean() * 100
    print(f"   {name:<20}: {mask.sum():,} samples ({pct:.1f}%)  "
          f"| target rate={pos_rate:.1%}  ← {'🔥 HIGH SIGNAL' if pos_rate > 0.3 else '⚪ low signal'}")

# ============================================================
# 4. TARGET-SPECIFIC ANALYSIS (minority class only)
# ============================================================
print(f"\n{'─'*80}")
print("🎯 TARGET CLASS ANALYSIS (positive samples only)")
print(f"{'─'*80}")

pos_mask = (y_test == 1)
print(f"   Total positive samples in test: {pos_mask.sum():,}\n")

_lr_catches  = (_lr_correct  & pos_mask)
_rf_catches  = (_rf_correct  & pos_mask)
_ebm_catches = (_ebm_correct & pos_mask)

print(f"   LR  catches  : {_lr_catches.sum():,} / {pos_mask.sum():,} targets ({_lr_catches.sum()/pos_mask.sum():.1%})")
print(f"   RF  catches  : {_rf_catches.sum():,} / {pos_mask.sum():,} targets ({_rf_catches.sum()/pos_mask.sum():.1%})")
print(f"   EBM catches  : {_ebm_catches.sum():,} / {pos_mask.sum():,} targets ({_ebm_catches.sum()/pos_mask.sum():.1%})")

for name, catches, others in [
    ("LR unique targets",  _lr_catches,  [_rf_catches,  _ebm_catches]),
    ("RF unique targets",  _rf_catches,  [_lr_catches,  _ebm_catches]),
    ("EBM unique targets", _ebm_catches, [_lr_catches,  _rf_catches]),
]:
    unique = catches & ~others[0] & ~others[1]
    pct_of_pos = unique.sum() / pos_mask.sum() * 100
    print(f"   {name:<22}: {unique.sum():,} ({pct_of_pos:.1f}% of all targets) "
          f"← {'✅ contributing' if unique.sum() > 0 else '❌ redundant'}")

all_catch  = _lr_catches & _rf_catches & _ebm_catches
none_catch = pos_mask & ~_lr_catches & ~_rf_catches & ~_ebm_catches

print(f"\n   All 3 catch (redundant)  : {all_catch.sum():,} ({all_catch.sum()/pos_mask.sum():.1%} of targets)")
print(f"   None catch (hard floor)  : {none_catch.sum():,} ({none_catch.sum()/pos_mask.sum():.1%} of targets) ← ceiling")

# ============================================================
# 5. BITMASK STATE ENCODING (2-bit per model)
# ============================================================
print(f"\n{'─'*80}")
print("🔢 BITMASK ARBITER STATE ENCODING")
print(f"{'─'*80}")
print("   Encoding: Bit1=triggered@low_thresh, Bit0=triggered@high_thresh")
print("   States: 00=Silent, 10=Candidate, 11=Confirmed\n")

_thresholds = {
    'LR':  (_lr_t_use,  max(0.01, _lr_t_use  - 0.07)),
    'RF':  (_rf_t_use,  max(0.01, _rf_t_use  - 0.07)),
    'EBM': (_ebm_t_use, max(0.01, _ebm_t_use - 0.07)),
}

def get_bitmask_state(prob, t_high, t_low):
    b0 = int(prob >= t_high)
    b1 = int(prob >= t_low)
    return (b1, b0)

lr_states  = [get_bitmask_state(p, *_thresholds['LR'])  for p in _lr_pte_use]
rf_states  = [get_bitmask_state(p, *_thresholds['RF'])  for p in _rf_pte_use]
ebm_states = [get_bitmask_state(p, *_thresholds['EBM']) for p in _ebm_pte_use]

state_labels = {(0,0): "00 Silent", (1,0): "10 Candidate", (1,1): "11 Confirmed"}

state_combos = Counter(
    (state_labels[tuple(l)], state_labels[tuple(r)], state_labels[tuple(e)])
    for l, r, e in zip(lr_states, rf_states, ebm_states)
)

print(f"   {'LR State':<15} {'RF State':<15} {'EBM State':<15} {'Count':>8} {'%':>7}  {'Target Rate'}")
print(f"   {'-'*70}")
for (ls, rs, es), cnt in sorted(state_combos.items(), key=lambda x: -x[1])[:15]:
    pct = cnt / total * 100
    smask = np.array([
        state_labels[tuple(l)] == ls and
        state_labels[tuple(r)] == rs and
        state_labels[tuple(e)] == es
        for l, r, e in zip(lr_states, rf_states, ebm_states)
    ])
    tr = y_test[smask].mean() if smask.sum() > 0 else 0
    flag = "🎯" if (ls == "11 Confirmed" and rs == "11 Confirmed" and es == "11 Confirmed") else \
           "⚡" if tr > 0.5 else \
           "🔥" if tr > 0.3 else ""
    print(f"   {ls:<15} {rs:<15} {es:<15} {cnt:>8,} {pct:>6.1f}%  {tr:.1%} {flag}")

# ============================================================
# 6. VENN DIAGRAM — saved to figures dir
# ============================================================
print(f"\n{'─'*80}")
print("🖼️  Generating Venn Diagram...")
print(f"{'─'*80}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f"Venn Diagram Trace — Model Agreement Analysis\n"
             f"LR={_lr_source} | RF={_rf_source} | EBM={_ebm_source}",
             fontsize=12, fontweight='bold')

lr_set  = set(np.where(_lr_correct)[0])
rf_set  = set(np.where(_rf_correct)[0])
ebm_set = set(np.where(_ebm_correct)[0])

plt.sca(axes[0])
venn3([lr_set, rf_set, ebm_set], set_labels=('LR', 'RF', 'EBM'), ax=axes[0])
axes[0].set_title("All Samples\n(correct predictions overlap)", fontsize=11)

lr_pos_set  = set(np.where(_lr_catches)[0])
rf_pos_set  = set(np.where(_rf_catches)[0])
ebm_pos_set = set(np.where(_ebm_catches)[0])

plt.sca(axes[1])
venn3([lr_pos_set, rf_pos_set, ebm_pos_set], set_labels=('LR', 'RF', 'EBM'), ax=axes[1])
axes[1].set_title("Target Class Only\n(correctly caught positives overlap)", fontsize=11)

plt.tight_layout()
_save_path = os.path.join(FIGURES_DIR, "venn_diagram_trace.png")
plt.savefig(_save_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"\n   ✅ Venn diagram saved → {_save_path}")

# ============================================================
# 7. COMPARISON TABLE (if multiple RF variants available)
# ============================================================
print(f"\n{'─'*80}")
print("📊 RF VARIANT COMPARISON (if available)")
print(f"{'─'*80}")

_rf_variants = {}

# Hardcoded RF
try:
    _rf_variants['Hardcoded'] = {
        'catches': (rf_pred_h == 1) & (y_test == 1),
        'total_correct': (rf_pred_h == y_test).sum(),
    }
except NameError:
    pass

# Balanced Optuna (continuous)
try:
    if rf_pred_o is not None:
        _rf_variants['Optuna continuous'] = {
            'catches': (rf_pred_o == 1) & (y_test == 1),
            'total_correct': (rf_pred_o == y_test).sum(),
        }
except NameError:
    pass

# Balanced binary
try:
    if rf_bin_best_pipe is not None:
        _bin_test = df_binned.iloc[test_idx][RF_FEATURES_BINARY]
        _bin_pte = rf_bin_best_pipe.predict_proba(_bin_test)[:, 1]
        _bin_pred = (_bin_pte >= rf_t_o).astype(int) if rf_t_o is not None else rf_bin_best_pipe.predict(_bin_test)
        _rf_variants['Balanced binary (29-bin)'] = {
            'catches': (_bin_pred == 1) & (y_test == 1),
            'total_correct': (_bin_pred == y_test).sum(),
        }
except (NameError, Exception):
    pass

# Recall-biased binary
try:
    if rf_bin_recall_clf is not None:
        _rbin_test = df_binned.iloc[test_idx][RF_FEATURES_BINARY]
        _rbin_pte = rf_bin_recall_clf.predict_proba(_rbin_test)[:, 1]
        _rbin_t = np.mean(rf_binr_thresh_scores)
        _rbin_pred = (_rbin_pte >= _rbin_t).astype(int)
        _rf_variants['Recall-biased binary (29-bin)'] = {
            'catches': (_rbin_pred == 1) & (y_test == 1),
            'total_correct': (_rbin_pred == y_test).sum(),
        }
except (NameError, Exception):
    pass

if len(_rf_variants) > 1:
    n_pos = pos_mask.sum()
    print(f"\n   {'RF Variant':<35} {'Catches':>8} {'Recall':>8} {'Accuracy':>10} {'Unique Targets':>15}")
    print(f"   {'-'*80}")
    for vname, vdata in _rf_variants.items():
        catches = vdata['catches'].sum()
        recall = catches / n_pos
        acc = vdata['total_correct'] / total
        # Unique targets: this RF catches but neither LR nor EBM catch
        unique_t = (vdata['catches'] & ~_lr_catches & ~_ebm_catches).sum()
        marker = " ← ACTIVE" if vname == _rf_source else ""
        print(f"   {vname:<35} {catches:>8,} {recall:>7.1%} {acc:>9.1%} {unique_t:>15,}{marker}")
else:
    print(f"   Only one RF variant available ({_rf_source})")

# ============================================================
# 8. VERDICT
# ============================================================
print(f"\n{'='*80}")
print("📋 BITMASK ARBITER FEASIBILITY VERDICT")
print(f"{'='*80}")

_unique_lr  = sum(1 for l,r,e in zip(_lr_correct, _rf_correct, _ebm_correct) if l==1 and r==0 and e==0)
_unique_rf  = sum(1 for l,r,e in zip(_lr_correct, _rf_correct, _ebm_correct) if l==0 and r==1 and e==0)
_unique_ebm = sum(1 for l,r,e in zip(_lr_correct, _rf_correct, _ebm_correct) if l==0 and r==0 and e==1)

_agreement_pct = all_agree_mask.mean() * 100

print(f"\n   Agreement rate   : {_agreement_pct:.1f}%  (target: 65–75% for healthy arbiter)")
print(f"   Redundant (all ✓): {all_correct.sum()/total:.1%}  (ideally <70%)")
print(f"   Unique LR wins   : {_unique_lr:,}")
print(f"   Unique RF wins   : {_unique_rf:,}")
print(f"   Unique EBM wins  : {_unique_ebm:,}")
print(f"   Hard floor (none): {none_catch.sum():,} targets no model catches\n")

if _agreement_pct > 85:
    print("   ⚠️  HIGH AGREEMENT — models too correlated.")
    print("      → Proceed with LR recall-bias retuning + feature subsetting")
    print("      → Re-run this trace after retuning to check improvement")
elif _agreement_pct > 75:
    print("   🟡 MODERATE AGREEMENT — some diversity present.")
    print("      → Bitmask Arbiter viable but diversity tuning will help further")
else:
    print("   ✅ HEALTHY DISAGREEMENT — Bitmask Arbiter is well motivated")
    print("      → Proceed to Stage 4 with agreement/disagreement features")

print(f"\n{'='*80}")
print("🎉 VENN DIAGRAM TRACE COMPLETE")
print(f"{'='*80}")

🔬 VENN DIAGRAM TRACE — MODEL DISAGREEMENT ANALYSIS
   Checking if disagreement is STRUCTURED (useful) or NOISE (useless)
   Each sample gets a Membership Bitmask: [LR, RF, EBM] → Correct/Wrong

   Using:
      LR  = Optuna recall-biased
      RF  = Recall-biased binary (29-bin)
      EBM = Optuna
   Thresholds: LR=0.4699, RF=0.9314, EBM=0.0835

────────────────────────────────────────────────────────────────────────────────
📊 BITMASK DISTRIBUTION (all samples)
────────────────────────────────────────────────────────────────────────────────
   Pattern                           Count        %  Class Balance
   -----------------------------------------------------------------
   LR=✓ RF=✓ EBM=✓                   5,815    70.6%  pos=11.1%  ███████████████████████████████████
   LR=✗ RF=✗ EBM=✗                   1,497    18.2%  pos=15.0%  █████████
   LR=✗ RF=✗ EBM=✓                     480     5.8%  pos=3.3%  ██
   LR=✓ RF=✗ EBM=✓                     190     2.3%  pos=0.0%  █
   LR=✓ RF=✓ 

In [104]:
# ================================================================
# CELL 17: MISSED SAMPLE ANALYSIS — EXPORT TO EXCEL
# ================================================================
# Exports samples the models miss (especially the hard floor) with
# full feature profiles + statistical summaries so we can understand
# WHY they're missed and whether there's a fixable pattern.
#
# Sheets:
#   1. hard_floor       — targets NO model catches (the ceiling)
#   2. lr_only_misses   — targets RF+EBM catch but LR misses
#   3. rf_only_misses   — targets LR+EBM catch but RF misses
#   4. ebm_only_misses  — targets LR+RF catch but EBM misses
#   5. statistical_profile — feature distributions comparing
#                            caught vs missed across all models
#   6. summary           — high-level counts and key findings
# ================================================================
print("=" * 80)
print("📊 MISSED SAMPLE ANALYSIS — EXPORT TO EXCEL")
print("=" * 80)

import os
import numpy as np
import pandas as pd
from scipy import stats

RESEARCH_DIR = r"C:\Users\phill\projects\ai-business-coach\prototype\glass_pipeline\research_logs"
OUTPUT_PATH = os.path.join(RESEARCH_DIR, "missed_sample_analysis.xlsx")

# ============================================================
# 1. BUILD SAMPLE MASKS
# ============================================================
print("\n1️⃣  Building sample masks...")

pos_mask = (y_test == 1)
test_df = df_engineered.iloc[test_idx].copy().reset_index(drop=True)
test_df['y_true'] = y_test

# Model predictions and probabilities (use best available)
_lr_pred = lr_pred_o  if lr_pred_o  is not None else lr_pred_h
_rf_pred = rf_pred_o  if rf_pred_o  is not None else rf_pred_h
_ebm_pred = ebm_pred_o if ebm_pred_o is not None else ebm_pred_h

_lr_prob = lr_pte_o  if lr_pte_o  is not None else lr_pte_h
_rf_prob = rf_pte_o  if rf_pte_o  is not None else rf_pte_h
_ebm_prob = ebm_pte_o if ebm_pte_o is not None else ebm_pte_h

test_df['lr_pred']  = _lr_pred
test_df['rf_pred']  = _rf_pred
test_df['ebm_pred'] = _ebm_pred
test_df['lr_prob']  = _lr_prob
test_df['rf_prob']  = _rf_prob
test_df['ebm_prob'] = _ebm_prob

# Correctness masks (on positive class only)
lr_catches  = (_lr_pred  == 1) & pos_mask
rf_catches  = (_rf_pred  == 1) & pos_mask
ebm_catches = (_ebm_pred == 1) & pos_mask

# Key sample groups
hard_floor    = pos_mask & ~lr_catches & ~rf_catches & ~ebm_catches  # none catch
all_catch     = pos_mask & lr_catches & rf_catches & ebm_catches      # all catch (redundant)
lr_only_miss  = pos_mask & ~lr_catches &  rf_catches &  ebm_catches   # LR fails, others succeed
rf_only_miss  = pos_mask &  lr_catches & ~rf_catches &  ebm_catches   # RF fails, others succeed
ebm_only_miss = pos_mask &  lr_catches &  rf_catches & ~ebm_catches   # EBM fails, others succeed

print(f"   Total positive samples in test: {pos_mask.sum():,}")
print(f"   Hard floor (none catch):  {hard_floor.sum():,} ({hard_floor.sum()/pos_mask.sum():.1%})")
print(f"   All 3 catch (redundant):  {all_catch.sum():,} ({all_catch.sum()/pos_mask.sum():.1%})")
print(f"   LR only misses:          {lr_only_miss.sum():,}")
print(f"   RF only misses:          {rf_only_miss.sum():,}")
print(f"   EBM only misses:         {ebm_only_miss.sum():,}")

# ============================================================
# 2. STATISTICAL PROFILE — caught vs hard floor
# ============================================================
print("\n2️⃣  Computing statistical profiles...")

# Features to analyze — use all available feature sets
all_features = sorted(set(LR_FEATURES + RF_FEATURES + EBM_FEATURES))
# Filter to features that exist in test_df
all_features = [f for f in all_features if f in test_df.columns]

caught_df = test_df[all_catch].copy()
floor_df  = test_df[hard_floor].copy()
all_pos_df = test_df[pos_mask].copy()

profile_rows = []
for feat in all_features:
    caught_vals = caught_df[feat].values
    floor_vals  = floor_df[feat].values
    all_pos_vals = all_pos_df[feat].values

    row = {
        'feature': feat,
        'caught_mean': np.mean(caught_vals) if len(caught_vals) > 0 else np.nan,
        'caught_std':  np.std(caught_vals)  if len(caught_vals) > 0 else np.nan,
        'caught_median': np.median(caught_vals) if len(caught_vals) > 0 else np.nan,
        'floor_mean': np.mean(floor_vals) if len(floor_vals) > 0 else np.nan,
        'floor_std':  np.std(floor_vals)  if len(floor_vals) > 0 else np.nan,
        'floor_median': np.median(floor_vals) if len(floor_vals) > 0 else np.nan,
        'all_pos_mean': np.mean(all_pos_vals),
        'delta_mean': (np.mean(floor_vals) - np.mean(caught_vals))
                      if len(floor_vals) > 0 and len(caught_vals) > 0 else np.nan,
    }

    # Statistical test: are the distributions different?
    if len(caught_vals) > 5 and len(floor_vals) > 5:
        try:
            t_stat, p_val = stats.mannwhitneyu(caught_vals, floor_vals, alternative='two-sided')
            row['mannwhitney_p'] = p_val
            row['significant'] = 'YES' if p_val < 0.05 else 'no'
        except Exception:
            row['mannwhitney_p'] = np.nan
            row['significant'] = 'N/A'

        # Effect size (Cohen's d)
        pooled_std = np.sqrt((np.var(caught_vals) + np.var(floor_vals)) / 2)
        if pooled_std > 0:
            row['cohens_d'] = (np.mean(floor_vals) - np.mean(caught_vals)) / pooled_std
        else:
            row['cohens_d'] = 0.0
    else:
        row['mannwhitney_p'] = np.nan
        row['significant'] = 'N/A'
        row['cohens_d'] = np.nan

    # Which model stage uses this feature
    in_lr  = feat in LR_FEATURES
    in_rf  = feat in RF_FEATURES
    in_ebm = feat in EBM_FEATURES
    row['in_LR']  = in_lr
    row['in_RF']  = in_rf
    row['in_EBM'] = in_ebm

    profile_rows.append(row)

profile_df = pd.DataFrame(profile_rows)
profile_df = profile_df.sort_values('mannwhitney_p', ascending=True)

# Print top discriminating features
print(f"\n   Top features discriminating CAUGHT vs HARD FLOOR:")
print(f"   {'Feature':<40} {'Caught μ':>10} {'Floor μ':>10} {'Δ':>10} {'Cohen d':>10} {'p-val':>10}")
print(f"   {'-'*90}")
for _, row in profile_df.head(15).iterrows():
    sig = "***" if row.get('mannwhitney_p', 1) < 0.001 else "**" if row.get('mannwhitney_p', 1) < 0.01 else "*" if row.get('mannwhitney_p', 1) < 0.05 else ""
    print(f"   {row['feature']:<40} {row['caught_mean']:>10.4f} {row['floor_mean']:>10.4f} "
          f"{row['delta_mean']:>+10.4f} {row.get('cohens_d', 0):>10.3f} "
          f"{row.get('mannwhitney_p', 1):>9.2e} {sig}")

# ============================================================
# 3. PROBABILITY ANALYSIS — how close were misses?
# ============================================================
print(f"\n3️⃣  Probability analysis on hard floor samples...")

floor_probs = pd.DataFrame({
    'lr_prob':  _lr_prob[hard_floor],
    'rf_prob':  _rf_prob[hard_floor],
    'ebm_prob': _ebm_prob[hard_floor],
})

# Use thresholds
_lr_t  = lr_t_o  if lr_t_o  is not None else lr_t_h
_rf_t  = rf_t_o  if rf_t_o  is not None else rf_t_h
_ebm_t = ebm_t_o if ebm_t_o is not None else ebm_t_h

print(f"   Thresholds: LR={_lr_t:.4f}, RF={_rf_t:.4f}, EBM={_ebm_t:.4f}")
print(f"\n   Hard floor probability distributions:")
print(f"   {'Model':<8} {'Mean':>8} {'Median':>8} {'Max':>8} {'Near-miss (>80% of t)':>22}")
for name, probs, thresh in [('LR', floor_probs['lr_prob'], _lr_t),
                              ('RF', floor_probs['rf_prob'], _rf_t),
                              ('EBM', floor_probs['ebm_prob'], _ebm_t)]:
    near_miss = (probs >= thresh * 0.80).sum()
    print(f"   {name:<8} {probs.mean():>8.4f} {probs.median():>8.4f} {probs.max():>8.4f} "
          f"{near_miss:>8,} ({near_miss/len(probs):.1%})")

# ============================================================
# 4. EXPORT TO EXCEL
# ============================================================
print(f"\n4️⃣  Exporting to Excel...")

# Columns to include in sample sheets
pred_cols = ['y_true', 'lr_pred', 'rf_pred', 'ebm_pred', 'lr_prob', 'rf_prob', 'ebm_prob']
feature_cols = [c for c in all_features if c in test_df.columns]
export_cols = pred_cols + feature_cols

with pd.ExcelWriter(OUTPUT_PATH, engine='openpyxl') as writer:

    # Sheet 1: Summary
    summary_data = {
        'Metric': [
            'Total test samples',
            'Total positive (targets)',
            'Hard floor (none catch)',
            'Hard floor % of targets',
            'All 3 catch (redundant)',
            'LR catches',
            'RF catches',
            'EBM catches',
            'LR unique catches',
            'RF unique catches',
            'EBM unique catches',
            'LR only misses',
            'RF only misses',
            'EBM only misses',
            'LR threshold',
            'RF threshold',
            'EBM threshold',
        ],
        'Value': [
            len(y_test),
            int(pos_mask.sum()),
            int(hard_floor.sum()),
            f"{hard_floor.sum()/pos_mask.sum():.1%}",
            int(all_catch.sum()),
            int(lr_catches.sum()),
            int(rf_catches.sum()),
            int(ebm_catches.sum()),
            int((lr_catches & ~rf_catches & ~ebm_catches).sum()),
            int((rf_catches & ~lr_catches & ~ebm_catches).sum()),
            int((ebm_catches & ~lr_catches & ~rf_catches).sum()),
            int(lr_only_miss.sum()),
            int(rf_only_miss.sum()),
            int(ebm_only_miss.sum()),
            f"{_lr_t:.4f}",
            f"{_rf_t:.4f}",
            f"{_ebm_t:.4f}",
        ]
    }
    pd.DataFrame(summary_data).to_excel(writer, sheet_name='summary', index=False)

    # Sheet 2: Hard floor samples (the most important sheet)
    floor_export = test_df[hard_floor][export_cols].copy()
    floor_export['max_prob'] = floor_export[['lr_prob', 'rf_prob', 'ebm_prob']].max(axis=1)
    floor_export['closest_model'] = floor_export[['lr_prob', 'rf_prob', 'ebm_prob']].idxmax(axis=1)
    floor_export = floor_export.sort_values('max_prob', ascending=False)
    floor_export.to_excel(writer, sheet_name='hard_floor', index=False)

    # Sheet 3: LR only misses
    if lr_only_miss.sum() > 0:
        test_df[lr_only_miss][export_cols].to_excel(
            writer, sheet_name='lr_only_misses', index=False)

    # Sheet 4: RF only misses
    if rf_only_miss.sum() > 0:
        test_df[rf_only_miss][export_cols].to_excel(
            writer, sheet_name='rf_only_misses', index=False)

    # Sheet 5: EBM only misses
    if ebm_only_miss.sum() > 0:
        test_df[ebm_only_miss][export_cols].to_excel(
            writer, sheet_name='ebm_only_misses', index=False)

    # Sheet 6: Statistical profile (caught vs floor)
    profile_df.to_excel(writer, sheet_name='statistical_profile', index=False)

    # Sheet 7: Probability heatmap — all positives with model probs and catch status
    prob_sheet = test_df[pos_mask][['y_true', 'lr_prob', 'rf_prob', 'ebm_prob']].copy()
    prob_sheet['lr_catches']  = lr_catches[pos_mask].astype(int)
    prob_sheet['rf_catches']  = rf_catches[pos_mask].astype(int)
    prob_sheet['ebm_catches'] = ebm_catches[pos_mask].astype(int)
    prob_sheet['n_models_catch'] = prob_sheet[['lr_catches', 'rf_catches', 'ebm_catches']].sum(axis=1)
    prob_sheet['category'] = prob_sheet['n_models_catch'].map({
        0: 'hard_floor', 1: 'one_catches', 2: 'two_catch', 3: 'all_catch'
    })
    prob_sheet = prob_sheet.sort_values('n_models_catch', ascending=True)
    prob_sheet.to_excel(writer, sheet_name='probability_heatmap', index=False)

print(f"\n   ✅ Exported to: {OUTPUT_PATH}")
print(f"   Sheets: summary, hard_floor, lr_only_misses, rf_only_misses, "
      f"ebm_only_misses, statistical_profile, probability_heatmap")

# ============================================================
# 5. KEY FINDINGS SUMMARY
# ============================================================
print(f"\n{'='*80}")
print("📋 KEY FINDINGS")
print(f"{'='*80}")

# What makes hard floor samples different?
sig_features = profile_df[profile_df['significant'] == 'YES'].head(10)
if len(sig_features) > 0:
    print(f"\n   Features that SIGNIFICANTLY differ between caught and missed targets:")
    for _, row in sig_features.iterrows():
        direction = "HIGHER" if row['delta_mean'] > 0 else "LOWER"
        print(f"      {row['feature']:<35} → floor is {direction} (d={row.get('cohens_d', 0):.2f})")

# Near-misses — how many hard floor samples were close?
for name, probs, thresh in [('LR', _lr_prob[hard_floor], _lr_t),
                              ('RF', _rf_prob[hard_floor], _rf_t),
                              ('EBM', _ebm_prob[hard_floor], _ebm_t)]:
    near = (probs >= thresh * 0.90).sum()
    if near > 0:
        print(f"\n   ⚡ {near} hard floor samples within 10% of {name} threshold "
              f"— threshold tuning could recover these")

any_near = (((_lr_prob[hard_floor] >= _lr_t * 0.85) |
             (_rf_prob[hard_floor] >= _rf_t * 0.85) |
             (_ebm_prob[hard_floor] >= _ebm_t * 0.85))).sum()
truly_invisible = hard_floor.sum() - any_near
print(f"\n   📊 Hard floor breakdown:")
print(f"      Near-miss (any model >85% of thresh): {any_near} — recoverable with tuning")
print(f"      Truly invisible (all models <85%):     {truly_invisible} — need new features or models")

print(f"\n{'='*80}")
print("🎉 MISSED SAMPLE ANALYSIS COMPLETE")
print(f"{'='*80}")

📊 MISSED SAMPLE ANALYSIS — EXPORT TO EXCEL

1️⃣  Building sample masks...
   Total positive samples in test: 928
   Hard floor (none catch):  230 (24.8%)
   All 3 catch (redundant):  638 (68.8%)
   LR only misses:          8
   RF only misses:          7
   EBM only misses:         15

2️⃣  Computing statistical profiles...

   Top features discriminating CAUGHT vs HARD FLOOR:
   Feature                                    Caught μ    Floor μ          Δ    Cohen d      p-val
   ------------------------------------------------------------------------------------------
   nsd_cold                                     0.0125     0.8522    +0.8396      3.192 5.14e-146 ***
   jed_cold                                     0.0125     0.8522    +0.8396      3.192 5.14e-146 ***
   overlap_behavioral_score                     0.0050     0.2587    +0.2537      2.017 1.82e-145 ***
   cellular_crisis                              0.9310     0.0261    -0.9049     -4.275 6.33e-143 ***
   overlap_default_